# Project ETH news


Overview

# Setup

In [30]:
from getpass import getpass

# Enter your token securely
token = getpass('Enter your GitHub PAT (Personal Access Token): ')

Enter your GitHub PAT (Personal Access Token): ··········


In [31]:
!git clone https://{token}@github.com/ddannia/news-qa-ethz1.git

Cloning into 'news-qa-ethz1'...
remote: Enumerating objects: 37360, done.
remote: Counting objects: 100% (13662/13662), done.
remote: Compressing objects: 100% (7105/7105), done.
remote: Total 37360 (delta 8694), reused 11256 (delta 6554), pack-reused 23698 (from 1)
Receiving objects: 100% (37360/37360), 31.49 MiB | 15.17 MiB/s, done.
Resolving deltas: 100% (23177/23177), done.
Updating files: 100% (13193/13193), done.


In [27]:
import os

# Replace with your repo name
repo_name = "news-qa-ethz1"

if os.path.exists(repo_name):
    print(f"✅ Repository '{repo_name}' is already cloned.")
else:
    print(f"❌ Repository '{repo_name}' is NOT cloned yet.")

❌ Repository 'news-qa-ethz1' is NOT cloned yet.


In [56]:
# 1. Set your Git identity (once per session)
!git config --global user.name "Ddannia"
!git config --global user.email "danniayw@googlemail.com"

In [ ]:
# 2. Pull latest changes (sync local with remote)
%cd news-qa-ethz1
!git pull origin main  # or 'master' depending on the default branch

[Errno 2] No such file or directory: 'content'
/content/news-qa-ethz1/news-qa-ethz1
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory


In [43]:
# 3. Make changes and stage them
!git add .

# or add a specific file:
#!git add myfile.py

#!git add processed_data/


In [42]:
# 4. commit
!git commit -m "update step 1.2"

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


news-qa-ethz1






# Load

### load a subset (4)

In [ ]:
#!rm -rf /content/news-qa-ethz1/news-qa-ethz1

In [5]:
import os

# Define the path to your parsed articles folder
folder_path = 'news-qa-ethz1/notebooks/parsed_markdown'

# Ensure the folder exists
if not os.path.exists(folder_path):
    raise FileNotFoundError(f"Folder not found: {folder_path}")

# List all files in the folder
files = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
print(f"Found {len(files)} files.")

# Read all files into a dictionary
parsed_articles = {}

for file in files:
    full_path = os.path.join(folder_path, file)
    with open(full_path, 'r', encoding='utf-8') as f:
        parsed_articles[file] = f.read()

# Display the first few loaded file names
list(parsed_articles.keys())[:5]

Found 4 files.


['die-eth-karte-erhaelt-ein-neues-design.md',
 'erc-advanced-grants.md',
 'in-memory-of-konrad-steffen.md',
 'detecting-storms-thanks-to-gps.md']

In [ ]:
# Pick one example to display (e.g., the first one)
example_filename = list(parsed_articles.keys())[3]
example_content = parsed_articles[example_filename]

print(f"📄 Filename: {example_filename}\n")
print(example_content)

📄 Filename: erc-advanced-grants.md

# erc-advanced-grants.html

## Die ETH muss attraktiv bleiben

Die ERC Advanced Grants gehören zu den begehrtesten Auszeichnungen im europäischen Forschungsraum. Mit ihnen fördert der Europäische Forschungsrat (ERC) ausschliesslich Projekte von etablierten Spitzenforschenden. Wer sich erfolgreich um diese Fördermittel bewirbt, erhält neben viel Renommee auch namhafte finanzielle Unterstützung. Die angenommenen Projekte werden während fünf Jahren mit rund 2,2 bis 3,8 Millionen Franken unterstützt.

17 Forschende der ETH Zürich haben sich für die ERC Advanced Grants beworben. Von ihnen schafften es 88 Prozent in die zweite Ausschreibungsrunde, und mehr als die Hälfte wurden mit «ausgezeichnet» (Kategorie A) bewertet und erfüllen somit die Kriterien für einen Grant. Wer am Schluss tatsächlich einen Grant erhält, hängt von vielen Faktoren ab, zum Beispiel davon, wie viel Geld insgesamt dem ERC zur Verfügung steht und wie viel davon an jeden einzelnen For

# Processing on all articles

### version 8

#### Basic preprocessing

In [3]:
#!pip install langdetect

In [4]:
import os
import json
import re
import string
import unicodedata
import logging
from tqdm import tqdm
from langdetect import detect
from pathlib import Path


logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def basic_text_cleaning(text):
    text = unicodedata.normalize('NFKC', text)
    text = text.strip()
    return text

def detect_language(text):
    if not text:
        return "unknown"

    try:
        sample = text[:1000]
        language = detect(sample)
        return language
    except:
        return "unknown"

def extract_article_structure(markdown_text):
    article = {
        'original_filename': '',
        'sections': {},
    }

    header_match = re.search(r'^# (.+?)$', markdown_text, re.MULTILINE)
    if header_match:
        article['original_filename'] = header_match.group(1).strip()

    lines = markdown_text.split('\n')
    current_section = None
    section_content = []

    for line in lines:
        if line.startswith('## '):
            if current_section is not None:
                article['sections'][current_section] = '\n'.join(section_content).strip()

            current_section = line[3:].strip()
            section_content = []
        elif current_section is not None:
            section_content.append(line)

    if current_section is not None and section_content:
        article['sections'][current_section] = '\n'.join(section_content).strip()

    return article

def extract_title(article_dict, filename):
    title_sections = ["In brief", "Main article", "Title", "Headline", "Titel", "Überschrift", "Kurz gefasst"]

    for section_name in title_sections:
        if section_name in article_dict['sections']:
            content = article_dict['sections'][section_name]
            lines = content.split('\n')
            if lines:
                for line in lines:
                    if line.strip():
                        return line.strip()

    if article_dict['original_filename']:
        clean_filename = article_dict['original_filename'].replace('.html', '')
        words = clean_filename.split('-')
        title = ' '.join(word.capitalize() for word in words)
        return title

    clean_filename = filename.replace('.md', '')
    words = clean_filename.split('-')
    title = ' '.join(word.capitalize() for word in words)
    return title

#### Extract date

*   Priority-based date matching: The function now scores dates based on how well they match the year and month from the filepath, giving highest priority to matches with both.
*   Simplified pattern handling: I consolidated the patterns and their handling logic using a list of tuples with pattern and formatter functions.
*   Better filepath parsing: The function now looks for the last occurrence of a year pattern in the path, which is typically the most specific, and checks if the next component is a valid month.
*   More robust date validation: It now validates all extracted date components before considering them as candidates.
*   Position-based prioritization: When multiple dates have the same score, it prioritizes dates that appear earlier in the text.
*   Cleaner fallback mechanism: If no dates are found in the text, it uses path information in a cleaner way.

In [5]:
import re
from pathlib import Path
from datetime import datetime

def extract_date(text, filepath=None):
    """
    Extract date from text, prioritizing dates that match the filepath's year and month.

    Args:
        text (str): Text to extract date from
        filepath (str, optional): File path that may contain year/month in format /YYYY/MM/

    Returns:
        str: Date in YYYY-MM-DD format or empty string if no date found
    """
    # Parse year and month from filepath if provided
    path_year = None
    path_month = None

    if filepath:
        path = Path(filepath)
        parts = list(path.parts)

        # Look for year pattern in path
        year_indices = [i for i, part in enumerate(parts) if re.fullmatch(r'20\d{2}', part)]
        if year_indices:
            # Get the last year occurrence (most specific)
            year_idx = year_indices[-1]
            path_year = parts[year_idx]

            # Check if the next part is a month (1-12)
            if year_idx + 1 < len(parts) and re.fullmatch(r'(?:0?[1-9]|1[0-2])', parts[year_idx + 1]):
                path_month = parts[year_idx + 1].zfill(2)

    # Define month name mappings
    months = {
        'january': '01', 'february': '02', 'march': '03', 'april': '04',
        'may': '05', 'june': '06', 'july': '07', 'august': '08',
        'september': '09', 'october': '10', 'november': '11', 'december': '12',
        'januar': '01', 'februar': '02', 'märz': '03', 'april': '04',
        'mai': '05', 'juni': '06', 'juli': '07', 'august': '08',
        'september': '09', 'oktober': '10', 'november': '11', 'dezember': '12',
        'jan': '01', 'feb': '02', 'mar': '03', 'apr': '04',
        'may': '05', 'jun': '06', 'jul': '07', 'aug': '08',
        'sep': '09', 'oct': '10', 'nov': '11', 'dec': '12'
    }

    # Consolidated date patterns
    date_patterns = [
        # Full date patterns DD.MM.YYYY or similar
        (r'(\d{1,2})\.(\d{1,2})\.(\d{4})',
         lambda m: (m.group(3), m.group(2).zfill(2), m.group(1).zfill(2))),

        (r'(\d{1,2})[/](\d{1,2})[/](\d{4})',
         lambda m: (m.group(3), m.group(2).zfill(2), m.group(1).zfill(2))),

        # ISO format YYYY-MM-DD
        (r'(\d{4})-(\d{1,2})-(\d{1,2})',
         lambda m: (m.group(1), m.group(2).zfill(2), m.group(3).zfill(2))),

        # Textual format like 13th July 2021
        (r'(\d{1,2})(?:st|nd|rd|th)? (?:of )?([A-Za-z]+)[,]? (\d{4})',
         lambda m: (m.group(3), months.get(m.group(2).lower(), '01'), m.group(1).zfill(2))),

        # "Published on" patterns
        (r'published (?:online )?(?:on )?(?:the )?(\d{1,2})(?:st|nd|rd|th)? ([A-Za-z]+)(?:[,]? (\d{4}))?',
         lambda m: (m.group(3) or path_year or str(datetime.now().year),
                    months.get(m.group(2).lower(), '01'), m.group(1).zfill(2))),

        (r'published (?:online )?(?:on )?(?:the )?([A-Za-z]+) (\d{1,2})(?:st|nd|rd|th)?(?:[,]? (\d{4}))?',
         lambda m: (m.group(3) or path_year or str(datetime.now().year),
                    months.get(m.group(1).lower(), '01'), m.group(2).zfill(2))),

        (r'published:? (\d{1,2})\.(\d{1,2})\.(\d{4})',
         lambda m: (m.group(3), m.group(2).zfill(2), m.group(1).zfill(2))),

        (r'published:? (\d{1,2})[/\-](\d{1,2})[/\-](\d{4})',
         lambda m: (m.group(3), m.group(2).zfill(2), m.group(1).zfill(2))),

        (r'published in ([A-Za-z]+) (\d{4})',
         lambda m: (m.group(2), months.get(m.group(1).lower(), '01'), '01')),

        # Publication date patterns
        (r'(?:date|pub\. date|publication date)[:\s]+(\d{1,2})\.(\d{1,2})\.(\d{4})',
         lambda m: (m.group(3), m.group(2).zfill(2), m.group(1).zfill(2))),

        (r'(?:date|pub\. date|publication date)[:\s]+(\d{1,2})[/\-](\d{1,2})[/\-](\d{4})',
         lambda m: (m.group(3), m.group(2).zfill(2), m.group(1).zfill(2))),

        # Partial date patterns
        (r'(\d{2})\.(\d{2})',  # MM.YY
         lambda m: (f"20{m.group(2)}" if int(m.group(2)) < 50 else f"19{m.group(2)}",
                   m.group(1).zfill(2), '01')),

        (r'(\d{2})[/\-](\d{4})',  # MM/YYYY
         lambda m: (m.group(2), m.group(1).zfill(2), '01')),

        (r'(\d{2})[/\-](\d{2})',  # MM/YY
         lambda m: (f"20{m.group(2)}" if int(m.group(2)) < 50 else f"19{m.group(2)}",
                   m.group(1).zfill(2), '01'))
    ]

    # Find all date matches in the text
    date_candidates = []

    for pattern, formatter in date_patterns:
        for match in re.finditer(pattern, text, re.IGNORECASE):
            try:
                year, month, day = formatter(match)

                # Validate date components
                if (1 <= int(month) <= 12 and 1 <= int(day) <= 31 and len(year) == 4):
                    date_tuple = (year, month, day)

                    # Calculate score based on matching path_year and path_month
                    score = 0
                    if path_year and year == path_year:
                        score += 100  # High priority for matching year
                    if path_month and month == path_month:
                        score += 50   # Medium priority for matching month

                    # Add position score (earlier in text = higher priority)
                    position_score = 1000 - match.start()  # Lower position = higher score

                    date_candidates.append((date_tuple, score, position_score))
            except (ValueError, TypeError, IndexError):
                continue

    # If no dates found in text, use path information
    if not date_candidates:
        if path_year and path_month:
            return f"{path_year}-{path_month}-01"
        elif path_year:
            return f"{path_year}-01-01"
        return ""

    # Sort by score (prioritizing path matches), then by position in text
    date_candidates.sort(key=lambda x: (x[1], x[2]), reverse=True)

    # Return the best match in YYYY-MM-DD format
    year, month, day = date_candidates[0][0]
    return f"{year}-{month}-{day}"

####  Extract source and main content

In [6]:
def extract_source(article_dict, text, language):
    reference_sections = ["Reference", "Referenz", "Quelle", "Source"]
    for section in reference_sections:
        if section in article_dict['sections']:
            source_text = article_dict['sections'][section]
            if source_text.strip():
                return source_text.strip()

    source_patterns = [
        r'(?:Source|Quelle):\s*([^\.]+)',
        r'(?:By|Von|Author|Autor):\s*([^\.]+)',
        r'(?:Copyright|©)\s*([^\.]+)',
        r'(?:Published by|Veröffentlicht von):\s*([^\.]+)'
    ]

    for pattern in source_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1).strip()

    eth_units = [
        r'(ETH[\s-]Zürich[^\.;,]*(?:Kommunikation|Communication|Department|Departement|Abteilung)[^\.;,]*)',
        r'(ETH[\s-]Zurich[^\.;,]*(?:Communication|Department|Unit|Division)[^\.;,]*)',
        r'((?:Hochschulkommunikation|University Communication)[^\.;,]*ETH[^\.;,]*)'
    ]

    for pattern in eth_units:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1).strip()

    if article_dict['original_filename']:
        domain_match = re.search(r'(?:www\.)?([a-z0-9-]+)\.(?:com|org|edu|ch|de)', article_dict['original_filename'], re.IGNORECASE)
        if domain_match:
            domain = domain_match.group(1).lower()
            if 'ethz' in domain:
                return "ETH Zürich" if language == "de" else "ETH Zurich"
            elif 'uzh' in domain:
                return "Universität Zürich" if language == "de" else "University of Zurich"
            else:
                domain = domain.replace('-', ' ').title()
                return domain

    if "ETH" in text:
        return "ETH Zürich, Hochschulkommunikation" if language == "de" else "ETH Zurich, University Communications"

    return "Unbekannte Quelle" if language == "de" else "Unknown source"

def extract_main_content(article_dict):
    skip_sections = ["Reference", "Referenz", "Quelle", "Source"]

    if len(article_dict['sections']) == 1:
        section_name = list(article_dict['sections'].keys())[0]
        if section_name not in skip_sections:
            return article_dict['sections'][section_name]

    content_parts = []

    for section_name, content in article_dict['sections'].items():
        if section_name not in skip_sections and content.strip():
            if len(article_dict['sections']) > 1:
                content_parts.append(f"{section_name}: {content}")
            else:
                content_parts.append(content)

    return "\n\n".join(content_parts)

#### Extract named entities

*  Better filtering for time expressions: I've expanded the list of time-related terms to catch false positives like "Ende Juli" that should not be considered entities.
*  Improved entity validation: The is_valid_entity() function provides more comprehensive checks to filter out invalid entities like "Jahren Gemeindepräsident".
*  Handling of overlapping entities: The algorithm now prioritizes longer, more specific entities over their substrings (e.g., prioritizing "Department of Computer Science" over just "Computer Science").
*  Added contextual patterns: Recognition of organizational structures like "Department of X" or "Institute for Y" that are common in academic settings.
*  Scoring system: Entities are now scored based on frequency, length, and relevance to ETH, not just on the order they appear in the text.
*  Better acronym detection: The function now handles academic acronyms better, while filtering out Roman numerals.
*  More specific ETH entities: Added entities specific to ETH Zurich that are likely to appear in your text.




In [7]:
import re
import string
from collections import Counter

def extract_named_entities(text, language):
    """
    Extract named entities from text with improved accuracy.

    Args:
        text (str): Text to extract entities from
        language (str): Language code 'en' or 'de'

    Returns:
        list: Up to 10 unique named entities
    """
    if not text:
        return []

    # Normalize text
    normalized_text = text.replace('\n', ' ').replace('  ', ' ')

    # Common words, time expressions, and phrases to exclude
    stop_words = set([
        "The", "This", "That", "These", "Those", "They", "Their", "And", "But", "However",
        "From", "Both", "When", "Then", "In", "On", "At", "All", "Every", "Each", "Many",
        "Few", "Some", "Other", "Another", "Such", "More", "Most", "Less", "Least",

        "Der", "Die", "Das", "Diese", "Dieser", "Dieses", "Jene", "Und", "Aber", "Jedoch",
        "Von", "Beide", "Wenn", "Dann", "In", "Auf", "Bei", "Alle", "Jeder", "Jede", "Jedes",
        "Viele", "Wenige", "Einige", "Andere", "Solche", "Mehr", "Meiste", "Weniger", "Wenigste"
    ])

    # Time expressions to exclude
    time_expressions = set([
        "January", "February", "March", "April", "May", "June", "July",
        "August", "September", "October", "November", "December",
        "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday",
        "Today", "Tomorrow", "Yesterday", "Morning", "Afternoon", "Evening", "Night",

        "Januar", "Februar", "März", "April", "Mai", "Juni", "Juli",
        "August", "September", "Oktober", "November", "Dezember",
        "Montag", "Dienstag", "Mittwoch", "Donnerstag", "Freitag", "Samstag", "Sonntag",
        "Heute", "Morgen", "Gestern", "Vormittag", "Nachmittag", "Abend", "Nacht",

        "Ende", "Anfang", "Mitte", "Begin", "End", "Middle", "Start",
        "Year", "Month", "Week", "Day", "Hour", "Minute", "Second",
        "Jahr", "Monat", "Woche", "Tag", "Stunde", "Minute", "Sekunde",
        "Jahren", "Monaten", "Wochen", "Tagen", "Stunden", "Minuten", "Sekunden"
    ])

    # Prepositions and articles to exclude
    prepositions = set([
        "in", "on", "at", "by", "for", "with", "from", "to", "of", "over", "under",
        "between", "among", "through", "during", "after", "before", "until", "about",
        "in", "auf", "an", "bei", "für", "mit", "von", "zu", "über", "unter",
        "zwischen", "durch", "während", "nach", "vor", "bis", "über"
    ])

    articles = set(["a", "an", "the", "ein", "eine", "einer", "eines", "einem", "einen", "der", "die", "das", "den", "dem", "des"])

    # Function to check if a potential entity is valid
    def is_valid_entity(entity):
        words = entity.split()

        # Check minimum length
        if len(entity) < 4 or len(words) < 2:
            return False

        # Check if it's all caps (likely an acronym)
        if entity.isupper() and len(entity) <= 5:
            return True

        # Check for time expressions
        if any(time_word in entity for time_word in time_expressions):
            return False

        # Check for starting with prepositions or articles
        if words[0].lower() in prepositions or words[0].lower() in articles:
            return False

        # Check if ending with common suffixes that shouldn't be entity boundaries
        invalid_suffixes = ["from", "with", "for", "by", "von", "mit", "für", "bei"]
        if words[-1].lower() in invalid_suffixes:
            return False

        # Check if it contains numbers
        if any(char.isdigit() for char in entity):
            return False

        # Check if it's a stopword
        if entity in stop_words:
            return False

        return True

    # Potential entities container
    potential_entities = []

    # 1. Look for organization names (2-4 capitalized words in sequence)
    org_pattern = r'\b([A-Z][a-zäöüÄÖÜß]+(?:[ \-][A-Z][a-zäöüÄÖÜß]+){1,3})\b'
    org_matches = re.findall(org_pattern, normalized_text)

    for match in org_matches:
        if is_valid_entity(match):
            potential_entities.append(match)

    # 2. Look for acronyms (2-5 uppercase letters)
    acronym_pattern = r'\b([A-Z]{2,5})\b'
    acronym_matches = re.findall(acronym_pattern, normalized_text)

    for match in acronym_matches:
        if len(match) >= 2 and not match.lower() in ['ii', 'iii', 'iv', 'vi', 'vii']:
            potential_entities.append(match)

    # 3. ETH-specific named entities with high priority
    eth_specific = []
    if language == "de":
        eth_specific = [
            "ETH Zürich", "ETH-Zürich", "ETH", "Universität Zürich", "UZH",
            "Hönggerberg", "ASVZ", "Polyterrasse", "Polybahn", "VSETH", "ETHZ"
        ]
    else:
        eth_specific = [
            "ETH Zurich", "ETH", "University of Zurich", "UZH",
            "Hönggerberg", "ASVZ", "Polyterrasse", "Polybahn", "VSETH", "ETHZ"
        ]

    for entity in eth_specific:
        if entity in normalized_text:
            potential_entities.append(entity)

    # 4. Look for people names with titles
    titles = ["Prof", "Professor", "Dr", "Professorin", "Doktor", "PD", "Dozent", "Dozentin"]

    for title in titles:
        # Pattern for "Title. FirstName LastName" or "Title FirstName LastName"
        name_pattern = f"{title}\\.?\\s+([A-Z][a-zäöüÄÖÜß]+(?:\\s+[A-Z][a-zäöüÄÖÜß]+){{1,2}})"
        prof_matches = re.findall(name_pattern, normalized_text)

        for match in prof_matches:
            if not any(word.lower() in prepositions for word in match.split()):
                potential_entities.append(f"{title}. {match}")

    # 5. Check for other potential entities with the pattern "Department of X" or "Institute for Y"
    entity_prefixes = [
        "Department of", "Institute of", "School of", "Faculty of", "Center for",
        "Institut für", "Abteilung für", "Departement für", "Fakultät für", "Zentrum für"
    ]

    for prefix in entity_prefixes:
        prefix_pattern = f"{prefix} ([A-Z][a-zA-ZäöüÄÖÜß ]+?)(?:\\s+[a-z]|\\.|,|$)"
        prefix_matches = re.findall(prefix_pattern, normalized_text)

        for match in prefix_matches:
            if len(match) > 3 and not match.strip() in time_expressions:
                potential_entities.append(f"{prefix} {match.strip()}")

    # Weighting and filtering entities
    entity_counter = Counter(potential_entities)

    # Filter out invalid entities and sort by frequency and length
    filtered_entities = []
    for entity, count in entity_counter.items():
        # Higher weight for repeated entities
        score = count * 2

        # Higher weight for longer entities (more specific)
        score += min(len(entity.split()), 3)

        # Higher weight for ETH-specific entities
        if any(eth_term in entity for eth_term in eth_specific):
            score += 5

        filtered_entities.append((entity, score))

    # Sort by score and get unique entities
    sorted_entities = sorted(filtered_entities, key=lambda x: x[1], reverse=True)

    # Deduplicate and handle overlapping entities
    unique_entities = []
    seen_substrings = set()

    for entity, _ in sorted_entities:
        entity_lower = entity.lower()

        # Skip if this entity is a substring of an already seen entity
        if any(entity_lower in seen.lower() for seen in seen_substrings):
            continue

        # If this entity contains any already seen entities, remove those
        for seen in list(seen_substrings):
            if seen.lower() in entity_lower:
                seen_substrings.remove(seen)
                if seen in unique_entities:
                    unique_entities.remove(seen)

        unique_entities.append(entity)
        seen_substrings.add(entity)

        if len(unique_entities) >= 10:
            break

    return unique_entities

#### Extract topics

In [8]:
def extract_topics(text, language):
    if not text:
        return ["University News"]

    text_lower = text.lower()

    topics_keywords = [
        ("Infrastructure", [
            ("karte", 3), ("card", 3), ("ausweis", 3), ("identification", 3),
            ("gebäude", 2), ("building", 2), ("raum", 1), ("room", 1),
            ("campus", 2), ("hönggerberg", 3), ("zentrum", 1), ("center", 1)
        ]),

        ("Technology & Innovation", [
            ("technologie", 3), ("technology", 3), ("innovation", 3),
            ("digital", 2), ("software", 2), ("computer", 2), ("app", 2),
            ("rfid", 3), ("chip", 2), ("elektronisch", 1), ("electronic", 1),
            ("entwicklung", 1), ("development", 1), ("programmier", 2), ("coding", 2)
        ]),

        ("Research", [
            ("forschung", 3), ("research", 3), ("wissenschaft", 3), ("science", 3),
            ("studie", 2), ("study", 2), ("experiment", 2), ("projekt", 1), ("project", 1),
            ("entdeckung", 2), ("discovery", 2), ("publikation", 2), ("publication", 2)
        ]),

        ("Education", [
            ("ausbildung", 3), ("education", 3), ("studium", 3), ("studies", 3),
            ("student", 3), ("studierend", 3), ("lehre", 3), ("teaching", 3),
            ("vorlesung", 2), ("lecture", 2), ("kurs", 2), ("course", 2),
            ("prüfung", 2), ("exam", 2), ("seminar", 2), ("unterricht", 2)
        ]),

        ("Finance", [
            ("finanzen", 3), ("finance", 3), ("kosten", 3), ("costs", 3),
            ("preis", 3), ("price", 3), ("erhöhung", 2), ("increase", 2),
            ("budget", 3), ("geld", 2), ("money", 2), ("zahlung", 1), ("payment", 1)
        ]),

        ("University Administration", [
            ("verwaltung", 3), ("administration", 3), ("leitung", 2), ("management", 2),
            ("präsident", 2), ("president", 2), ("rektor", 2), ("rector", 2),
            ("direktor", 2), ("director", 2), ("strategie", 2), ("strategy", 2)
        ]),

        ("Campus Life", [
            ("campus", 3), ("student", 2), ("mensa", 3), ("cafeteria", 3),
            ("essen", 2), ("food", 2), ("veranstaltung", 2), ("event", 2),
            ("freizeit", 2), ("leisure", 2), ("sport", 2), ("asvz", 3)
        ]),

        ("International", [
            ("international", 3), ("global", 3), ("weltweit", 2), ("worldwide", 2),
            ("ausland", 2), ("foreign", 2), ("kooperation", 2), ("cooperation", 2),
            ("austausch", 2), ("exchange", 2), ("partner", 2)
        ]),

        ("Sustainability", [
            ("nachhaltig", 3), ("sustainable", 3), ("umwelt", 3), ("environment", 3),
            ("klima", 3), ("climate", 3), ("grün", 2), ("green", 2),
            ("energie", 2), ("energy", 2), ("ressource", 2), ("resource", 2)
        ]),

        ("Weather & Environment", [
            ("wetter", 3), ("weather", 3), ("sturm", 3), ("storm", 3),
            ("umwelt", 2), ("environment", 2), ("klima", 2), ("climate", 2),
            ("regen", 2), ("rain", 2), ("wind", 2), ("temperatur", 2), ("temperature", 2)
        ]),

        ("Communication", [
            ("kommunikation", 3), ("communication", 3), ("mitteilung", 3), ("announcement", 3),
            ("information", 2), ("bericht", 2), ("report", 2), ("news", 3),
            ("presse", 2), ("press", 2), ("media", 2), ("medien", 2)
        ]),

        ("Catering & Food", [
            ("mensa", 3), ("cafeteria", 3), ("essen", 3), ("food", 3),
            ("verpflegung", 3), ("catering", 3), ("restaurant", 2),
            ("speise", 2), ("meal", 2), ("menü", 2), ("menu", 2)
        ]),

        ("Staff", [
            ("mitarbeiter", 3), ("staff", 3), ("personal", 3), ("employee", 3),
            ("anstellung", 2), ("employment", 2), ("arbeit", 2), ("work", 2),
            ("position", 2), ("stelle", 2), ("job", 2)
        ]),

        ("COVID-19", [
            ("covid", 3), ("corona", 3), ("pandemic", 3), ("pandemie", 3),
            ("lockdown", 3), ("virus", 2), ("impfung", 2), ("vaccination", 2),
            ("maske", 2), ("mask", 2), ("abstand", 2), ("distance", 2)
        ])
    ]

    topic_scores = {}

    for topic_name, keywords in topics_keywords:
        score = 0
        for keyword, weight in keywords:
            count = text_lower.count(keyword)
            if count > 0:
                score += count * weight

        if score > 0:
            topic_scores[topic_name] = score

    if not topic_scores:
        return ["University News"]

    sorted_topics = sorted(topic_scores.items(), key=lambda x: x[1], reverse=True)
    return [topic for topic, score in sorted_topics[:6]]


#### keyword extaction

*   Multiple extraction methods: Combines TF-IDF, TextRank, and KeyBERT for more robust keyword extraction
*   Fallback mechanisms: If any advanced method fails, the system gracefully falls back to simpler methods
*   Priority-based merging: Keywords from more sophisticated methods get higher priority
*   Language support: Maintains support for both English and German texts
*   Multi-word phrase detection: Preserves your special handling for important multi-word terms
*   Better preprocessing: Uses NLTK's tokenization and stopword removal





In [9]:
#!pip install nltk==3.8.1 --quiet
#import nltk
#nltk.download('punkt')
#nltk.download('stopwords')
#nltk.download('wordnet')

In [13]:
#!pip install keybert

  Using cached keybert-0.9.0-py3-none-any.whl.metadata (15 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.4.5.8-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.2.1.3-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.5.147-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.6.1.9-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.3.1.170-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nvjitlink_cu12-12.4.127-py3-none-manylin

In [11]:
#def ensure_nltk_resources(): [__import__('nltk').download(r, quiet=True) for r in ['punkt', 'stopwords', 'wordnet']]
#ensure_nltk_resources()

In [14]:
import string
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from keybert import KeyBERT
import numpy as np
from nltk.tokenize import sent_tokenize
import networkx as nx

# Download necessary NLTK resources (only need to run once)
#nltk.download('punkt', quiet=True)
#nltk.download('stopwords', quiet=True)
#nltk.download('wordnet', quiet=True)

def extract_keywords(text, language):
    """
    Extract keywords from text using multiple advanced NLP techniques.

    Args:
        text (str): Input text for keyword extraction
        language (str): Language code 'en' or 'de'

    Returns:
        list: Up to 7 unique keywords
    """
    if not text:
        return ["ETH Zurich"]

    # Prepare fallback keywords
    eth_keywords = ["ETH Zurich", "University", "Research", "Science", "Campus", "Education"]

    # Initialize language-specific resources
    if language == "de":
        try:
            stop_words = set(stopwords.words('german'))
        except:
            # Fallback if NLTK resources not available
            stop_words = set([
                "der", "die", "das", "und", "oder", "aber", "weil", "als", "wenn", "als", "aber", "nicht",
                "während", "bei", "für", "mit", "über", "gegen", "zwischen", "durch",
                "von", "zu", "in", "auf", "dass", "diese", "auch", "dann", "mehr", "andere"
            ])

        # German to English mapping for certain important terms
        de_to_en = {
            "eth zürich": "ETH Zurich",
            "eth-karte": "ETH Card",
            "karte": "Card",
            "ausweis": "Identification Card",
            "studierende": "Students",
            "studenten": "Students",
            "student": "Student",
            "mitarbeitende": "Staff",
            "mitarbeiter": "Staff",
            "personal": "Personnel",
            "design": "Design",
            "erscheinungsbild": "Visual Identity",
            "forschung": "Research",
            "wissenschaft": "Science",
            "preiserhöhung": "Price Increase",
            "kosten": "Costs",
            "mensa": "Cafeteria",
            "verpflegung": "Catering",
            "unwetter": "Storm",
            "sturm": "Storm",
            "wetter": "Weather",
            "nachhaltigkeit": "Sustainability",
            "umwelt": "Environment",
            "klima": "Climate",
            "gebäude": "Building",
            "campus": "Campus",
            "hönggerberg": "Hönggerberg",
            "zentrum": "Campus Center",
            "technologie": "Technology",
            "innovation": "Innovation",
            "digital": "Digital",
            "lehre": "Teaching",
            "bildung": "Education",
            "kommunikation": "Communication"
        }
    else:  # English
        try:
            stop_words = set(stopwords.words('english'))
        except:
            # Fallback if NLTK resources not available
            stop_words = set([
                "the", "and", "or", "but", "because", "as", "if", "when", "than", "but", "not",
                "during", "at", "by", "for", "with", "about", "against", "between", "into",
                "through", "from", "to", "in", "on", "that", "this", "these", "those", "also"
            ])
        de_to_en = {}  # Not needed for English

    # Add custom stopwords
    stop_words.update(["also", "like", "would", "could", "should", "make", "want"])

    # Pre-process text
    text_lower = text.lower()
    text_no_punct = text_lower.translate(str.maketrans('', '', string.punctuation))

    # Special handling for multi-word terms
    multi_word_terms = {
        "eth zürich": "ETH Zurich",
        "eth-karte": "ETH Card",
        "elektronische karte": "Electronic ID",
        "corporate design": "Corporate Design",
        "neue design": "New Design",
        "universität zürich": "University of Zurich",
        "hönggerberg campus": "Hönggerberg Campus"
    }

    # Extract multi-word terms
    special_keywords = []
    for term, translation in multi_word_terms.items():
        if term in text_lower:
            special_keywords.append(translation)

    # Tokenize and clean text
    words = word_tokenize(text_no_punct)
    filtered_words = [w for w in words if w not in stop_words and len(w) > 3 and not w.isdigit()]

    # 1. TF-IDF based extraction
    try:
        # Create a document from filtered words for TF-IDF
        processed_text = ' '.join(filtered_words)

        # If text is too short, skip TF-IDF
        if len(processed_text.split()) < 5:
            tfidf_keywords = []
        else:
            # Create TF-IDF vectorizer
            vectorizer = TfidfVectorizer(max_features=20,
                                        ngram_range=(1, 2),
                                        stop_words=list(stop_words))

            # Fit and transform the text
            tfidf_matrix = vectorizer.fit_transform([processed_text])

            # Get feature names and scores
            feature_names = vectorizer.get_feature_names_out()
            scores = tfidf_matrix.toarray()[0]

            # Create keyword-score pairs and sort
            keyword_scores = [(feature_names[i], scores[i]) for i in range(len(feature_names))]
            keyword_scores.sort(key=lambda x: x[1], reverse=True)

            # Extract top keywords
            tfidf_keywords = [k.capitalize() for k, _ in keyword_scores[:10]]
    except Exception:
        # Fallback if TF-IDF fails
        tfidf_keywords = []

    # 2. TextRank-based extraction
    try:
        # Only apply TextRank if text has enough sentences
        sentences = sent_tokenize(text)

        if len(sentences) > 2:
            # Create a graph
            graph = nx.Graph()

            # Add nodes (words)
            unique_filtered_words = list(set(filtered_words))
            graph.add_nodes_from(unique_filtered_words)

            # Add edges (co-occurrences)
            window_size = 4
            for i in range(len(filtered_words) - window_size + 1):
                window = filtered_words[i:i+window_size]
                for j in range(len(window)):
                    for k in range(j+1, len(window)):
                        if window[j] != window[k]:
                            if graph.has_edge(window[j], window[k]):
                                graph[window[j]][window[k]]['weight'] += 1
                            else:
                                graph.add_edge(window[j], window[k], weight=1)

            # Run PageRank
            scores = nx.pagerank(graph)

            # Sort words by score
            sorted_words = sorted(scores.items(), key=lambda x: x[1], reverse=True)

            # Extract top TextRank keywords
            textrank_keywords = [word.capitalize() for word, _ in sorted_words[:10]]
        else:
            textrank_keywords = []
    except Exception:
        # Fallback if TextRank fails
        textrank_keywords = []

    # 3. KeyBERT-based extraction (if available)
    try:
        # Initialize KeyBERT
        kw_model = KeyBERT()

        # Extract keywords with KeyBERT
        keybert_results = kw_model.extract_keywords(
            text,
            keyphrase_ngram_range=(1, 2),
            stop_words=list(stop_words),
            top_n=10
        )

        # Process results
        keybert_keywords = [keyword.capitalize() for keyword, _ in keybert_results]
    except Exception:
        # Fallback if KeyBERT fails or is not available
        keybert_keywords = []

    # 4. Traditional frequency-based extraction (as backup)
    word_freq = {}
    for word in filtered_words:
        # Apply language-specific translations for German
        if language == "de" and word in de_to_en:
            word = de_to_en[word]
        word_freq[word] = word_freq.get(word, 0) + 1

    sorted_words = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)
    frequency_keywords = [word.capitalize() for word, _ in sorted_words[:10]]

    # Combine all keyword methods with priority
    all_keywords = []

    # Add special multi-word terms first (highest priority)
    all_keywords.extend(special_keywords)

    # Add keywords from advanced methods
    if keybert_keywords:
        all_keywords.extend(keybert_keywords)

    if tfidf_keywords:
        all_keywords.extend(tfidf_keywords)

    if textrank_keywords:
        all_keywords.extend(textrank_keywords)

    # Add frequency-based keywords as backup
    all_keywords.extend(frequency_keywords)

    # Add default ETH keywords at the end (lowest priority)
    all_keywords.extend(eth_keywords)

    # Remove duplicates while preserving order
    unique_keywords = []
    seen = set()
    for kw in all_keywords:
        kw_lower = kw.lower()
        if kw_lower not in seen and not any(kw_lower == sw for sw in stop_words) and len(kw) > 3:
            seen.add(kw_lower)
            unique_keywords.append(kw)

    return unique_keywords[:7]

#### Generate summary

*   Content-Based Scoring: Instead of just taking the first few sentences, this
function scores sentences based on the importance of the words they contain.
*   Term Frequency: Words that appear more frequently in the document are likely more important to the topic, so sentences containing these words get higher scores.
*   Position Bias: Recognizes that introductory sentences often contain key information, but doesn't exclusively select them.
*   Length Normalization: Prevents bias towards very long sentences by normalizing scores based on sentence length.
*   NLTK Integration: Uses NLTK for better sentence tokenization if available, with a regex fallback.
*   Original Flow Preservation: After selecting the highest-scoring sentences, it arranges them in their original order to maintain the logical flow of the text.
*   Flexible Selection: Ensures we get at least one sentence even if it exceeds the maximum length to provide some summary.

In [15]:
import re
import numpy as np
import string
from collections import Counter
from heapq import nlargest
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords

def generate_summary(text, max_length=250):
    """
    Generate a summary using a hybrid extractive method that weighs sentences
    based on importance rather than just taking the first few sentences.

    Args:
        text (str): The text to summarize
        max_length (int): Maximum length of the summary in characters

    Returns:
        str: The generated summary
    """
    if not text or len(text) <= max_length:
        return text

    # Tokenize sentences using NLTK if available, fallback to regex if not
    try:
        sentences = sent_tokenize(text)
    except:
        # Fallback to regex sentence splitting (less accurate but doesn't require nltk)
        sentence_pattern = r'(?<=[.!?])\s+'
        sentences = re.split(sentence_pattern, text)
        # Filter out empty sentences
        sentences = [s.strip() for s in sentences if s.strip()]

    # Simple case: if we only have a few short sentences, just return them
    if len(sentences) <= 3 and sum(len(s) for s in sentences) <= max_length:
        return " ".join(sentences)

    # Get stopwords if NLTK is available, otherwise use a basic list
    try:
        stop_words = set(stopwords.words('english'))
    except:
        stop_words = {
            "the", "a", "an", "and", "but", "if", "or", "because", "as", "what",
            "which", "this", "that", "these", "those", "then", "just", "so", "than",
            "such", "when", "who", "how", "where", "why", "is", "are", "was", "were",
            "be", "been", "being", "have", "has", "had", "having", "do", "does", "did",
            "doing", "to", "from", "in", "out", "on", "off", "over", "under", "again"
        }

    # Clean and tokenize the text for scoring
    def clean_text(text):
        # Remove punctuation and convert to lowercase
        text = text.lower()
        text = text.translate(str.maketrans('', '', string.punctuation))
        # Remove numbers
        text = re.sub(r'\d+', '', text)
        # Tokenize and remove stopwords
        words = text.split()
        return [word for word in words if word not in stop_words and len(word) > 1]

    words = clean_text(text)

    # Calculate term frequency
    word_freq = Counter(words)
    max_freq = max(word_freq.values(), default=1)

    # Normalize word frequency
    for word in word_freq:
        word_freq[word] = word_freq[word] / max_freq

    # Score sentences based on term frequency, position, and length
    sentence_scores = {}
    for i, sentence in enumerate(sentences):
        words_in_sentence = clean_text(sentence)

        # Skip very short sentences (often not informative)
        if len(words_in_sentence) < 3:
            continue

        # Score based on word frequency
        tf_score = sum(word_freq.get(word, 0) for word in words_in_sentence)

        # Position bias: earlier sentences often contain key information
        position_score = 1.0 / (i + 1) if i < 3 else 1.0 / (i + 2)

        # Length normalization: avoid bias towards longer sentences
        length_factor = min(1.0, 5.0 / len(words_in_sentence)) if len(words_in_sentence) > 0 else 0

        # Final score combines all factors
        final_score = (tf_score * 0.6) + (position_score * 0.3) + (length_factor * 0.1)
        sentence_scores[sentence] = final_score

    # Find the best sentences while respecting max_length
    summary = ""

    # Sort sentences by score
    ranked_sentences = sorted(sentence_scores.items(), key=lambda x: x[1], reverse=True)

    # Take highest-scored sentences that fit within max_length
    current_length = 0
    selected_sentences = []

    for sentence, _ in ranked_sentences:
        if current_length + len(sentence) + 1 <= max_length:  # +1 for space
            selected_sentences.append(sentence)
            current_length += len(sentence) + 1
        else:
            # If we haven't selected any sentences yet, take at least one
            if not selected_sentences:
                selected_sentences.append(sentence)
            break

    # Sort selected sentences by their original position to maintain flow
    sentence_positions = {s: sentences.index(s) for s in selected_sentences}
    selected_sentences.sort(key=lambda s: sentence_positions[s])

    # Join sentences to form summary
    summary = " ".join(selected_sentences)

    # Add ellipsis if we've truncated the text
    if len(summary) < len(text) and summary:
        summary += "..."

    return summary

#### Extract content type

In [16]:
def extract_document_type(text, language):
    """
    Determine the document type based on content analysis.
    """
    text_lower = text.lower()

    # Document type patterns
    type_patterns = {
        "News Article": [
            r'news', r'artikel', r'bericht', r'mitteilung', r'press release',
            r'medienmitteilung', r'ankündigung', r'announcement'
        ],
        "Research Publication": [
            r'study', r'studie', r'research', r'forschung', r'publication',
            r'publikation', r'journal', r'paper', r'doi', r'published in', r'veröffentlicht in'
        ],
        "Event Announcement": [
            r'event', r'veranstaltung', r'seminar', r'workshop', r'conference',
            r'konferenz', r'invitation', r'einladung', r'upcoming', r'registration'
        ],
        "Policy Update": [
            r'policy', r'richtlinie', r'regulation', r'regulierung', r'guideline',
            r'leitfaden', r'rule', r'regel', r'procedure', r'prozedur'
        ],
        "Interview": [
            r'interview', r'conversation', r'gespräch', r'qa', r'q&a',
            r'fragen und antworten', r'discusses', r'diskutiert'
        ],
        "Campus Notice": [
            r'notice', r'hinweis', r'reminder', r'erinnerung', r'announcement',
            r'ankündigung', r'attention', r'achtung', r'important information'
        ]
    }

    type_scores = {}
    for doc_type, patterns in type_patterns.items():
        score = 0
        for pattern in patterns:
            matches = re.findall(pattern, text_lower)
            score += len(matches)
        if score > 0:
            type_scores[doc_type] = score

    if not type_scores:
        return "News Article"  # Default

    # Return the document type with the highest score
    return max(type_scores.items(), key=lambda x: x[1])[0]

#### Anlyze content structure

In [17]:
def analyze_content_structure(text):
    """
    Analyze the content structure to identify sections, paragraphs, and formatting.
    """
    if not text:
        return {}

    # Count paragraphs
    paragraphs = [p for p in text.split('\n\n') if p.strip()]
    paragraph_count = len(paragraphs)

    # Count bullet points and numbered lists
    bullet_pattern = r'^\s*[\*\-•]\s+'
    bullet_points = sum(1 for line in text.split('\n') if re.match(bullet_pattern, line))

    numbered_pattern = r'^\s*\d+\.\s+'
    numbered_items = sum(1 for line in text.split('\n') if re.match(numbered_pattern, line))

    # Detect if there are headings
    heading_pattern = r'^#+\s+'
    headings = sum(1 for line in text.split('\n') if re.match(heading_pattern, line))


    # Check for links
    link_pattern = r'\[([^\]]+)\]\(([^)]+)\)'
    links = len(re.findall(link_pattern, text))

    # Structure analysis
    structure_analysis = {
        "paragraph_count": paragraph_count,
        "bullet_points": bullet_points,
        "numbered_items": numbered_items,
        "headings": headings,
        "links": links
    }

    return structure_analysis

#### Generate additional metadata like doc length, readability, sentiment

In [18]:
def generate_rich_metadata(text, language):
    """
    Generate rich metadata for the article with standardized English fields.
    """
    if not text:
        return {
            "document_length_words": 0,
            "readability_score": "unknown",
            "sentiment": "neutral",
            "audience_type": ["Students", "Staff"],
            "context_tags": ["ETH Internal"]
        }

    # Count words
    words = re.findall(r'\b\w+\b', text)
    word_count = len(words)

    # Determine readability (standardized to English)
    sentences = re.split(r'[.!?]+', text)
    sentence_count = len([s for s in sentences if s.strip()])

    avg_words_per_sentence = word_count / max(sentence_count, 1)
    long_words = len([w for w in words if len(w) > 6])
    long_word_percentage = (long_words / max(word_count, 1)) * 100

    if avg_words_per_sentence > 25 or long_word_percentage > 30:
        readability = "complex"
    elif avg_words_per_sentence > 15 or long_word_percentage > 20:
        readability = "moderately complex"
    else:
        readability = "easy to read"

    # Improved sentiment analysis with more keywords
    positive_patterns = [
        r'\b(?:gut|besser|positiv|erfolgreich|vorteil|nutzen|förder|erfreut|freude|verbessert)',
        r'\b(?:good|better|positive|successful|advantage|benefit|promote|pleased|improve|happy)'
    ]

    negative_patterns = [
        r'\b(?:schlecht|problem|negativ|schwierig|nachteil|kritisch|belastung|sorge|verschlechter)',
        r'\b(?:bad|problem|negative|difficult|disadvantage|critical|burden|worry|worsen)'
    ]

    positive_count = 0
    for pattern in positive_patterns:
        positive_count += len(re.findall(pattern, text.lower()))

    negative_count = 0
    for pattern in negative_patterns:
        negative_count += len(re.findall(pattern, text.lower()))

    # Calculate sentiment ratio
    total = positive_count + negative_count
    if total == 0:
        sentiment = "neutral"
    elif positive_count > negative_count * 2:
        sentiment = "positive"
    elif negative_count > positive_count * 2:
        sentiment = "negative"
    elif positive_count > negative_count:
        sentiment = "slightly positive"
    elif negative_count > positive_count:
        sentiment = "slightly negative"
    else:
        sentiment = "neutral"

    # Determine audience type (standardized to English)
    audience_type = []

    # More specific patterns for different audience types
    audience_patterns = [
        ("Students", [r'\bstud(?:ent|ierend|ium|ies)', r'\bschule\b', r'\buniversity\b']),
        ("Staff", [r'\bmitarbeit|\bpersonal|\bstaff|\bemployee|\bangestellt']),
        ("Faculty", [r'\bprofessor|\bdozent|\bfaculty|\blectur|\bdocent']),
        ("Researchers", [r'\bforsch|\bresearch|\bwissenschaft|\bscience|\blabor|\blab\b']),
        ("Administration", [r'\bverwaltung|\badministration|\bleitung|\bmanagement']),
        ("General Public", [r'\böffentlich|\bpublic|\ballgemein|\bgeneral|\bcommunity|\bgesellschaft'])
    ]

    for audience, patterns in audience_patterns:
        for pattern in patterns:
            if re.search(pattern, text.lower()):
                audience_type.append(audience)
                break

    # Default audience if none detected
    if not audience_type:
        audience_type = ["Students", "Staff"]

    # Context tags (standardized to English)
    context_tags = []

    # More comprehensive context tagging system
    context_patterns = [
        ("Financial", [r'\bfinan|\bkosten|\bbudget|\bcost|\bprice|\bpreis|\bgeld|\bmoney']),
        ("Catering", [r'\bmensa|\bessen|\bverpfleg|\bfood|\bdining|\bcafeteria|\bmahlzeit|\bmeal']),
        ("COVID-19", [r'\bcorona|\bcovid|\bpandemie|\bpandemic|\blockdown|\bvirus']),
        ("Sustainability", [r'\bnachhaltig|\bsustainable|\benvironment|\bumwelt|\bklima|\bclimate']),
        ("Infrastructure", [r'\bgebäude|\bbuilding|\binfrastruktur|\bcampus|\braum|\bspace']),
        ("Technology", [r'\btechnologie|\btechnology|\bdigital|\bsoftware|\brfid|\bapp']),
        ("Research", [r'\bforschung|\bresearch|\bwissenschaft|\bscience|\bstudie|\bstudy']),
        ("Education", [r'\bausbildung|\beducation|\bstudium|\bstudies|\blehre|\bteaching']),
        ("Administrative", [r'\bverwaltung|\badministration|\bmanagement|\bleitung|\bpolicy']),
        ("Communications", [r'\bkommunikation|\bcommunication|\bmitteilung|\bannouncement']),
        ("Events", [r'\bveranstaltung|\bevent|\bkonferenz|\bconference|\bmeeting|\bseminar']),
        ("Weather", [r'\bwetter|\bweather|\bsturm|\bstorm|\bregen|\brain|\btemperatur']),
        ("International", [r'\binternational|\bglobal|\bweltweit|\bworldwide|\bausland']),
        ("Career", [r'\bkarriere|\bcareer|\bjob|\bstelle|\bposition|\bbewerbung|\bapplication'])
    ]

    # Check for context tags
    for tag, patterns in context_patterns:
        for pattern in patterns:
            if re.search(pattern, text.lower()):
                context_tags.append(tag)
                break

    # Check if ETH-related
    if re.search(r'\beth|\bethz|\beidgenössische|\bpoly', text.lower()):
        context_tags.append("ETH Internal")

    # Limit to most relevant tags (max 4)
    if len(context_tags) > 4:
        context_tags = context_tags[:4]
    elif not context_tags:
        context_tags = ["University News"]

    # Extract document type
    document_type = extract_document_type(text, language)

    # Extract citation information if it appears to be a research publication
    #citation_info = {}
    #if document_type == "Research Publication" or "Research" in context_tags:
    #    citation_info = extract_citation_info(text)

    # Extract temporal references for events
    #temporal_info = {}
    #if document_type == "Event Announcement" or "Events" in context_tags:
    #    temporal_info = extract_temporal_references(text)

    # Extract semantic entities for better context
    #semantic_entities = extract_semantic_entities(text)

    # Generate text complexity metrics
    word_lengths = [len(w) for w in words if w]
    avg_word_length = sum(word_lengths) / max(len(word_lengths), 1)

    # Text complexity metrics
    complexity_metrics = {
        "avg_words_per_sentence": round(avg_words_per_sentence, 2),
        "avg_word_length": round(avg_word_length, 2),
        "long_word_percentage": round(long_word_percentage, 2)
    }

    # Create rich metadata with all fields in English
    rich_metadata = {
        "document_length_words": word_count,
        "readability_score": readability,
        "sentiment": sentiment,
        "document_type": document_type,
        "audience_type": audience_type,
        "context_tags": context_tags,
        "complexity_metrics": complexity_metrics
    }

    # Add additional metadata based on document type
    #if citation_info:
    #    rich_metadata["citation_info"] = citation_info

    #if temporal_info and any(temporal_info.values()):
    #    rich_metadata["temporal_info"] = temporal_info

    #if semantic_entities and any(semantic_entities.values()):
    #    rich_metadata["semantic_entities"] = semantic_entities

    return rich_metadata

#### Load markdown files

In [19]:

def load_markdown_files(input_dir):
    """
    Load all markdown files from the input directory.
    Args:
        input_dir (str): Path to the directory containing .md files
    Returns:
        list: List of tuples (file_path, file_name, file_content)
    """
    markdown_files = []

    try:
        input_path = Path(input_dir)
        if not input_path.exists():
            logger.error(f"Input directory does not exist: {input_dir}")
            return markdown_files

        # Recursively find all .md files
        for file_path in input_path.glob('**/*.md'):
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = f.read()
                    file_name = file_path.name
                    markdown_files.append((file_path, file_name, content))
                    logger.info(f"Loaded file: {file_path}")
            except Exception as e:
                logger.error(f"Error reading file {file_path}: {str(e)}")

        logger.info(f"Loaded {len(markdown_files)} markdown files")
        return markdown_files

    except Exception as e:
        logger.error(f"Error loading markdown files: {str(e)}")
        return markdown_files

#### Save

In [20]:
def save_processed_article(processed_article, input_dir, output_dir, filepath):
    """
    Save processed article to JSON file in the same directory structure as the input.
    Args:
        processed_article (dict): Processed article data
        input_dir (str): Base input directory
        output_dir (str): Base output directory
        filepath (Path): Original file path
    """
    try:
        # Get the relative path from the input directory
        input_path = Path(input_dir)
        relative_path = filepath.relative_to(input_path)

        # Create the same directory structure in the output directory
        output_file_dir = Path(output_dir) / relative_path.parent
        output_file_dir.mkdir(parents=True, exist_ok=True)

        # Create output file path with .json extension
        output_file = output_file_dir / (filepath.stem + '.json')

        # Write JSON with proper indentation and UTF-8 encoding
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(processed_article, f, ensure_ascii=False, indent=2)

        logger.info(f"Saved processed article to {output_file}")

    except Exception as e:
        logger.error(f"Error saving processed article {filepath}: {str(e)}")

#### Batch process

In [21]:

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

def batch_process(input_dir, output_dir, max_workers=4):
    """
    Process multiple markdown files in parallel.
    Args:
        input_dir (str): Base input directory
        output_dir (str): Base output directory
        max_workers (int): Maximum number of parallel workers
    """
    logger.info(f"Starting batch processing with {max_workers} workers")

    # Load markdown files with full paths
    input_files = []
    input_path = Path(input_dir)

    # Find all .md files in news-qa-ethz1/HKNews directory and subdirectories
    for file_path in input_path.glob('**/HKNews/**/*.md'):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()
                input_files.append((file_path, content))
                logger.info(f"Loaded file: {file_path}")
        except Exception as e:
            logger.error(f"Error reading file {file_path}: {str(e)}")

    logger.info(f"Loaded {len(input_files)} markdown files")

    # Define the processing function inside batch_process so it can access variables
    def process_file(file_info):
        file_path, content = file_info
        try:
            # Process the article
            processed_article = process_article(content, file_path.name, file_path)

            # Save the processed article
            save_processed_article(processed_article, input_dir, output_dir, file_path)

            return True
        except Exception as e:
            logger.error(f"Error processing {file_path}: {str(e)}")
            return False

    # Process files in parallel
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(tqdm(
            executor.map(process_file, input_files),
            total=len(input_files),
            desc="Processing articles"
        ))

    # Count successes and failures
    processed_count = sum(results)
    error_count = len(input_files) - processed_count

    logger.info(f"Batch processing completed. Processed: {processed_count}, Errors: {error_count}")

    return processed_count, error_count


#### Create search index

In [22]:
def create_search_index(processed_dir, index_file):
    """
    Create a simple search index from processed articles.
    Args:
        processed_dir (str): Directory containing processed JSON files
        index_file (str): Output index file path
    """
    try:
        processed_path = Path(processed_dir)

        if not processed_path.exists():
            logger.error(f"Processed directory does not exist: {processed_dir}")
            return False

        # Initialize search index
        search_index = {
            "articles": [],
            "keywords": {},
            "topics": {},
            "entities": {},
            "metadata": {
                "total_articles": 0,
                "languages": {},
                "document_types": {}
            }
        }

        # Process all JSON files
        json_files = list(processed_path.glob('**/*.json'))

        for json_file in tqdm(json_files, desc="Building search index"):
            try:
                with open(json_file, 'r', encoding='utf-8') as f:
                    article = json.load(f)

                # Add to articles list with essential info
                article_info = {
                    "id": json_file.stem,
                    "title": article.get("title", ""),
                    "language": article.get("language", "unknown"),
                    "date": article.get("date", ""),
                    "summary": article.get("summary", ""),
                    "path": str(json_file.relative_to(processed_path))
                }

                search_index["articles"].append(article_info)

                # Update metadata counters
                search_index["metadata"]["total_articles"] += 1

                # Count languages
                lang = article.get("language", "unknown")
                search_index["metadata"]["languages"][lang] = search_index["metadata"]["languages"].get(lang, 0) + 1

                # Count document types
                doc_type = article.get("rich_metadata", {}).get("document_type", "Unknown")
                search_index["metadata"]["document_types"][doc_type] = search_index["metadata"]["document_types"].get(doc_type, 0) + 1

                # Index keywords
                for keyword in article.get("keywords", []):
                    if keyword not in search_index["keywords"]:
                        search_index["keywords"][keyword] = []
                    search_index["keywords"][keyword].append(article_info["id"])

                # Index topics
                for topic in article.get("topics", []):
                    if topic not in search_index["topics"]:
                        search_index["topics"][topic] = []
                    search_index["topics"][topic].append(article_info["id"])

                # Index named entities
                for entity in article.get("named_entities", []):
                    if entity not in search_index["entities"]:
                        search_index["entities"][entity] = []
                    search_index["entities"][entity].append(article_info["id"])

            except Exception as e:
                logger.error(f"Error indexing file {json_file}: {str(e)}")

        # Save the index
        with open(index_file, 'w', encoding='utf-8') as f:
            json.dump(search_index, f, ensure_ascii=False, indent=2)

        logger.info(f"Search index created successfully with {search_index['metadata']['total_articles']} articles")
        return True

    except Exception as e:
        logger.error(f"Error creating search index: {str(e)}")
        return False

#### Main processing function

In [23]:
# Main article processing function
# exclude extract_citation_info, extract_temporal_references, extract_semantic_entities
def process_article(markdown_text, filename, filepath):
    """
    Process a single article through all cleaning steps.
    Args:
        markdown_text (str): The content of the .md file
        filename (str): The name of the file (e.g., "article.md")
        filepath (Path or str): Full path to the .md file, used to extract date if needed
    Returns:
        dict: Structured metadata and article info
    """
    try:
        # Basic cleaning
        cleaned_text = basic_text_cleaning(markdown_text)

        # Extract article structure
        article_structure = extract_article_structure(cleaned_text)

        # Extract main content (keep in original language)
        main_content = extract_main_content(article_structure)

        # Detect language
        language = detect_language(main_content or cleaned_text)

        # Extract title (keep in original language)
        title = extract_title(article_structure, filename)

        # Extract date (standardized format, fallback to folder structure)
        date = extract_date(main_content or cleaned_text, filepath=filepath)

        # Extract source (keep in original language)
        source = extract_source(article_structure, main_content or cleaned_text, language)

        # Extract named entities (keep in original language)
        named_entities = extract_named_entities(main_content, language)

        # Extract topics (standardized to English)
        topics = extract_topics(main_content, language)

        # Extract keywords (standardized to English)
        keywords = extract_keywords(main_content, language)

        # Generate summary (keep in original language)
        summary = generate_summary(main_content)

        # Generate rich metadata (standardized to English)
        rich_metadata = generate_rich_metadata(main_content, language)

        # Analyze content structure
        content_structure = analyze_content_structure(main_content)

        # Final structured document
        processed_article = {
            "language": language,
            "title": title,
            "date": date,
            "source": source,
            "main_content": main_content,
            "named_entities": named_entities,
            "topics": topics,
            "keywords": keywords,
            "summary": summary,
            "content_structure": content_structure,
            "rich_metadata": rich_metadata
        }

        return processed_article

    except Exception as e:
        logger.error(f"Error processing {filename}: {str(e)}")

        # Return a minimal valid structure in case of failure
        return {
            "language": "unknown",
            "title": filename.replace('.md', '').replace('-', ' ').title(),
            "date": "",
            "source": "ETH Zurich",
            "main_content": "",
            "named_entities": [],
            "topics": ["University News"],
            "keywords": ["ETH Zurich"],
            "summary": "",
            "content_structure": {
                "paragraph_count": 0,
                "bullet_points": 0,
                "numbered_items": 0,
                "headings": 0,
                "links": 0
                }
            ,
            "rich_metadata": {
                "document_length_words": 0,
                "readability_score": "unknown",
                "sentiment": "neutral",
                "document_type": "News Article",
                "audience_type": ["Students", "Staff"],
                "context_tags": ["ETH Internal"]
            }
        }

#### Run script

In [33]:
def run_processing_for_github(base_input_dir, base_output_dir, max_workers=4):
    """
    Run ETH news processing for GitHub repository structure.

    Args:
        base_input_dir (str): Path to the directory containing news-qa-ethz1
        base_output_dir (str): Path where processed files should be saved
        max_workers (int, optional): Number of parallel workers
    """
    # Look for HKNews directory within the repository
    input_dir = Path(base_input_dir)
    logger.info(f"Looking for HKNews in: {input_dir}")

    # Process files in parallel
    processed_count, error_count = batch_process(base_input_dir, base_output_dir, max_workers)

    logger.info(f"Processing completed. Processed: {processed_count}, Errors: {error_count}")
    logger.info("ETH news processing completed successfully")

    return processed_count, error_count


base_dir = '/content/news-qa-ethz1'
output_dir = '/content/news-qa-ethz1/'  # Same as input to keep files in place
processed_count, error_count = run_processing_for_github(base_dir, output_dir)

Processing articles:   0%|          | 0/4390 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Processing articles: 100%|██████████| 4390/4390 [1:37:04<00:00,  1.33s/it]


#### Commit

In [36]:
# Navigate to your repo (if not already there)
%cd /content/news-qa-ethz1

# Track all newly created JSON files
!git add HKNews/**/*.json

# Commit the changes with a clear message
!git commit -m "🔄 Add processed article metadata as JSON files"

# Push the changes to your GitHub fork
!git push origin main

/content/news-qa-ethz1
[main 06f6b254] 🔄 Add processed article metadata as JSON files
 4358 files changed, 51980 insertions(+), 174196 deletions(-)
Enumerating objects: 8342, done.
Counting objects: 100% (8342/8342), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4876/4876), done.
Writing objects: 100% (4877/4877), 3.19 MiB | 3.95 MiB/s, done.
Total 4877 (delta 3516), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3516/3516), completed with 3418 local objects.
remote: This repository moved. Please use the new location:
remote:   https://github.com/Ddannia/news-qa-ethz1.git
To https://github.com/ddannia/news-qa-ethz1.git
   a77591af..06f6b254  main -> main


# Inspect processed files

In [37]:
import re
from pathlib import Path

# Your base news folder
hknews_path = Path("/content/news-qa-ethz1/HKNews")

# Search term (adjust this)
search_term = "Harvesting Drinking Water From Humidity"  # ← your keyword/title (case-insensitive)

# Supported file types
file_extensions = [".html", ".md", ".json"]

# Compile regex (case-insensitive)
pattern = re.compile(re.escape(search_term), re.IGNORECASE)

# Search all relevant files
matches = []
for ext in file_extensions:
    for file in hknews_path.rglob(f"*{ext}"):
        try:
            with open(file, 'r', encoding='utf-8') as f:
                content = f.read()
                if pattern.search(content):
                    matches.append(file)
        except:
            pass  # Skip unreadable files

# Show results
print(f"✅ Found {len(matches)} matching files for '{search_term}':\n")
for match in matches:
    print("-", match)

# Optional: Load and preview the first match
if matches:
    print("\n📄 Preview of first match:")
    with open(matches[0], 'r', encoding='utf-8') as f:
        print(f.read()[:1000])  # show first 1000 characters

✅ Found 1 matching files for 'Harvesting Drinking Water From Humidity':

- /content/news-qa-ethz1/HKNews/en_news_events/2021/06/harvesting-drinking-water-from-humidity.json

📄 Preview of first match:
{
  "language": "unknown",
  "title": "Harvesting Drinking Water From Humidity",
  "date": "2021-06-01",
  "source": "Haechler I, Park H, Schnoering G, Gulich T, Rohner M, Tripathy A, Milionis A, Schutzius TM, Poulikakos D: Exploiting radiative cooling for uninterrupted 24-hour water harvesting from the atmosphere. Science Advances, 23 June 2021, doi: 10.1126/sciadv.abf3978",
  "main_content": "Self-cooling and protection from radiation: Fresh water is scarce in many parts of the world and must be obtained at great expense. Communities near the ocean can desalinate sea water for this purpose, but doing so requires a large amount of energy. Further away from the coast, practically often the only remaining option is to condense atmospheric humidity through cooling, either through processes t

### save as csv

In [38]:
import json
import pandas as pd
from pathlib import Path

# Path to your repo
hknews_path = Path("/content/news-qa-ethz1/HKNews")

# Find all JSON files recursively
json_files = list(hknews_path.rglob("*.json"))
print(f"✅ Found {len(json_files)} JSON metadata files.")

# Load into list
all_articles = []
for file in json_files:
    try:
        with open(file, 'r', encoding='utf-8') as f:
            data = json.load(f)
            data['source_file'] = str(file.relative_to(hknews_path))  # keep path info
            all_articles.append(data)
    except Exception as e:
        print(f"⚠️ Skipped {file}: {e}")

✅ Found 4390 JSON metadata files.


In [39]:
# Flatten rich metadata fields if needed
for a in all_articles:
    rich = a.pop('rich_metadata', {})
    a.update(rich)

# Create DataFrame
df = pd.DataFrame(all_articles)

# Save as CSV
csv_path = "/content/news_metadata.csv"
df.to_csv(csv_path, index=False)
print(f"📁 Saved CSV to: {csv_path}")

📁 Saved CSV to: /content/news_metadata.csv


In [40]:
from google.colab import files

# Download both
files.download("/content/news_metadata.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Commit notebook changes

In [54]:
!cd /content/news-qa-ethz1/

In [58]:
import os
os.getcwd()

'/content/news-qa-ethz1'

In [65]:
!ls /content

news_metadata.csv  news-qa-ethz1  sample_data


In [61]:
!find /content -name "*.ipynb"

/content/news-qa-ethz1/notebooks/1_2_Multilingual_Text_Preprocessing_and_Cleaning.ipynb
/content/news-qa-ethz1/notebooks/.ipynb_checkpoints/02_hybrid_parsing-checkpoint.ipynb
/content/news-qa-ethz1/notebooks/.ipynb_checkpoints/01_html_parsing_comparison-checkpoint.ipynb
/content/news-qa-ethz1/notebooks/02_hybrid_parsing.ipynb
/content/news-qa-ethz1/notebooks/01_html_parsing_comparison.ipynb


In [60]:
!mv "/content/1.2_Multilingual Text Preprocessing and Cleaning.ipynb" "/content/news-qa-ethz1/notebooks/1_2_Multilingual_Text_Preprocessing_and_Cleaning.ipynb"

mv: cannot stat '/content/1.2_Multilingual Text Preprocessing and Cleaning.ipynb': No such file or directory


In [63]:
%cd /content/news-qa-ethz1

!git status  # Just to confirm Git sees the changes

!git add notebooks/1_2_Multilingual_Text_Preprocessing_and_Cleaning.ipynb
!git commit -m "Update: improved multilingual text preprocessing notebook"
!git push

/content/news-qa-ethz1
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add/rm <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	deleted:    scripts/process_eth_news.py

no changes added to commit (use "git add" and/or "git commit -a")
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add/rm <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	deleted:    scripts/process_eth_news.py

no changes added to commit (use "git add" and/or "git commit -a")
Everything up-to-date


# Archive

## Load

### load all articles

In [32]:
import os

for root, dirs, files in os.walk(os.getcwd()):
    if 'HKNews' in dirs:
        print(f"Found HKNews at: {os.path.join(root, 'HKNews')}")

        # Check if it contains the expected language folders
        hknews_path = os.path.join(root, 'HKNews')
        print(f"Contents of HKNews: {os.listdir(hknews_path)}")

Found HKNews at: /content/news-qa-ethz1/HKNews
Contents of HKNews: ['en_news_events', 'de_news_events', 'de_internal', 'en_internal']


In [ ]:
from getpass import getpass
token = getpass("🔐 Enter your GitHub token")

!git clone https://{token}@github.com/ddannia/news-qa-ethz1.git /content/news-qa-ethz1

🔐 Enter your GitHub token··········
Cloning into '/content/news-qa-ethz1'...
remote: Enumerating objects: 23731, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 23731 (delta 23), reused 10 (delta 10), pack-reused 23698 (from 1)
Receiving objects: 100% (23731/23731), 19.24 MiB | 19.28 MiB/s, done.
Resolving deltas: 100% (14506/14506), done.
Updating files: 100% (8803/8803), done.


In [4]:
from pathlib import Path
import os
import json

# 1. Set the base HKNews directory
hknews_base = Path("/content/news-qa-ethz1/HKNews")

# 2. Recursively find all .md files under the base directory
md_file_paths = list(hknews_base.rglob("*.md"))
print(f"✅ Found {len(md_file_paths)} markdown files.")

# 3. Read each markdown file into a dictionary
parsed_articles = {}

for path in md_file_paths:
    try:
        with open(path, 'r', encoding='utf-8') as f:
            parsed_articles[str(path)] = f.read()
    except Exception as e:
        print(f"⚠️ Failed to read {path}: {e}")

✅ Found 4390 markdown files.


In [ ]:
# Pick one example to display (e.g., the first one)
example_filename = list(parsed_articles.keys())[4]
example_content = parsed_articles[example_filename]

print(f"📄 Filename: {example_filename}\n")
print(example_content)

📄 Filename: /content/news-qa-ethz1/HKNews/en_news_events/2021/06/mixed-cultures-for-a-greater-yield.md

# mixed-cultures-for-a-greater-yield

**Source:** en_news_events/2021/06/mixed-cultures-for-a-greater-yield.html

## Applying an ecological principle

Monocultures dominate arable land today, with vast areas given over to single elite varieties that promise a high yield. But planting arable land with just one type of crop has its disadvantages: these areas are easy game for fungal and insect pests, posing a threat to crops. To keep pests at bay, farmers are having to use resistant varieties and various pesticides.

Mixed cultures present a potential alternative to monocultures. Rather than having large expanses of land planted with just one species or variety, several species or varieties are sown alongside each other. However, as little research has been done into this method, especially from an agricultural perspective, mixed cultures are rare in arable farming.

A team led by ETH 

In [ ]:
# Install spaCy
!pip install spacy

# Download the English model (small)
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 60.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
!pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 13.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=3b005e2e6b6b463fdaa70eb2b815ecb6035c46c523437ef73f4b38fc3d78b8b8
  Stored in directory: /root/.cache/pip/wheels/0a/f2/b2/e5ca405801e05eb7c8ed5b3b4bcf1fcabcd6272c167640072e
Successfully built langdetect


In [ ]:
!pip install nltk


In [ ]:
import os
import re
import json
import pandas as pd
import unicodedata
import datetime
import argparse
from typing import Dict, List, Any, Tuple
from pathlib import Path
import logging

# For language detection
import langdetect
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 42  # For reproducible results

# Additional libraries for NLP
import spacy
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download necessary NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("text_processing.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("TextProcessor")

# Load spaCy models for named entity recognition
try:
    nlp_en = spacy.load("en_core_web_sm")
    nlp_de = spacy.load("de_core_news_sm")
    spacy_models_available = True
except OSError:
    logger.warning("spaCy models not found. Named entity recognition will be limited.")
    logger.info("To install spaCy models, run: python -m spacy download en_core_web_sm de_core_news_sm")
    spacy_models_available = False

class TextProcessor:
    """
    Process multilingual markdown files containing news articles.
    Handles both German and English texts with language-specific processing.
    """

    def __init__(self):
        self.stopwords_en = set(stopwords.words('english'))
        self.stopwords_de = set(stopwords.words('german'))

        # Additional German stopwords
        self.stopwords_de.update(['der', 'die', 'das', 'und', 'in', 'zu', 'den', 'dem', 'mit', 'von'])

        # Common date patterns for standardization
        self.date_patterns = [
            # German format: DD.MM.YYYY
            (r'(\d{1,2})\.(\d{1,2})\.(\d{4})', r'\3-\2-\1'),
            # English format: MM/DD/YYYY
            (r'(\d{1,2})/(\d{1,2})/(\d{4})', r'\3-\1-\2'),
            # Other common formats
            (r'(\d{1,2})-(\d{1,2})-(\d{4})', r'\3-\2-\1'),
        ]

    def read_markdown_file(self, file_path: str) -> str:
        """Read the content of a markdown file."""
        try:
            with open(file_path, 'r', encoding='utf-8') as file:
                return file.read()
        except UnicodeDecodeError:
            # Try another encoding if UTF-8 fails
            with open(file_path, 'r', encoding='latin-1') as file:
                return file.read()
        except Exception as e:
            logger.error(f"Error reading file {file_path}: {e}")
            return ""

    def detect_language(self, text: str) -> str:
        """
        Detect the language of the text.
        Returns 'de' for German, 'en' for English, 'unknown' otherwise.
        """
        try:
            # Use the first paragraph that isn't a header for detection
            paragraphs = [p for p in text.split('\n\n') if not p.startswith('#')]
            if not paragraphs:
                return "unknown"

            sample_text = paragraphs[0]
            lang = detect(sample_text)

            # Map to our simplified language codes
            if lang == 'de':
                return 'de'
            elif lang in ['en', 'eng']:
                return 'en'
            else:
                return lang
        except Exception as e:
            logger.warning(f"Language detection failed: {e}")
            return "unknown"

    def normalize_unicode(self, text: str) -> str:
        """Normalize Unicode characters."""
        return unicodedata.normalize('NFKC', text)

    def normalize_whitespace(self, text: str) -> str:
        """Remove extra spaces and normalize line breaks."""
        # Replace multiple spaces with a single space
        text = re.sub(r' +', ' ', text)
        # Normalize multiple line breaks
        text = re.sub(r'\n{3,}', '\n\n', text)
        return text.strip()

    def standardize_dates(self, text: str) -> str:
        """Standardize various date formats to ISO format (YYYY-MM-DD)."""
        for pattern, replacement in self.date_patterns:
            text = re.sub(pattern, replacement, text)
        return text

    def process_german_text(self, text: str) -> str:
        """Apply German-specific text processing."""
        # Normalize umlauts if needed
        text = text.replace('ae', 'ä').replace('oe', 'ö').replace('ue', 'ü')

        # Handle compound words if needed
        # This is a simplified approach, more complex processing might be needed

        return text

    def extract_named_entities(self, text: str, language: str) -> List[Dict[str, str]]:
        """Extract named entities using spaCy."""
        if not spacy_models_available:
            return []

        try:
            # Use the appropriate model based on language
            nlp = nlp_de if language == 'de' else nlp_en
            doc = nlp(text)

            # Extract entities
            entities = []
            for ent in doc.ents:
                entities.append({
                    'text': ent.text,
                    'type': ent.label_,
                    'start': ent.start_char,
                    'end': ent.end_char
                })

            return entities
        except Exception as e:
            logger.warning(f"Named entity extraction failed: {e}")
            return []

    def extract_keywords(self, text: str, language: str, num_keywords: int = 10) -> List[str]:
        """
        Extract key terms from the text based on frequency and importance.
        """
        # Choose stopwords based on language
        stop_words = self.stopwords_de if language == 'de' else self.stopwords_en

        # Tokenize
        words = word_tokenize(text.lower())

        # Remove stopwords and short words
        filtered_words = [word for word in words
                          if word.isalpha() and
                          word not in stop_words and
                          len(word) > 2]

        # Count word frequencies
        word_freq = {}
        for word in filtered_words:
            word_freq[word] = word_freq.get(word, 0) + 1

        # Sort by frequency
        sorted_keywords = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)

        # Return top keywords
        return [word for word, _ in sorted_keywords[:num_keywords]]

    def parse_markdown_content(self, content: str, filename: str) -> Dict[str, Any]:
        """
        Parse markdown content and extract structured data.
        """
        # Extract original HTML filename from the first header
        header_match = re.search(r'^#\s+(.+\.html)', content, re.MULTILINE)
        original_filename = header_match.group(1) if header_match else filename

        # Remove the header line
        content_without_header = re.sub(r'^#\s+.+\.html\s*\n', '', content)

        # Extract sections
        sections = {}
        section_matches = re.findall(r'##\s+(.+)\s*\n([\s\S]*?)(?=\n##\s+|\n?$)', content_without_header)

        for section_title, section_content in section_matches:
            section_title = section_title.strip()
            section_content = self.normalize_whitespace(section_content)
            sections[section_title] = section_content

        # Find main content section
        main_content = ""
        main_section_title = ""

        # Look for section titles that might indicate main content
        main_content_indicators = [
            'main content', 'content', 'main', 'body', 'article',  # English
            'hauptinhalt', 'inhalt', 'hauptteil', 'artikel'        # German
        ]

        # First, try to find by title
        for title, content in sections.items():
            if any(indicator in title.lower() for indicator in main_content_indicators):
                main_content = content
                main_section_title = title
                break

        # If not found, use the longest section
        if not main_content:
            max_length = 0
            for title, content in sections.items():
                if len(content) > max_length:
                    max_length = len(content)
                    main_content = content
                    main_section_title = title

        # If still no sections found, use the whole content without the header
        if not main_content:
            main_content = self.normalize_whitespace(content_without_header)
            main_section_title = "Full Content"

        # Detect language
        language = self.detect_language(main_content)

        # Get title
        title = original_filename.replace('.html', '').replace('-', ' ')
        title = ' '.join(word.capitalize() for word in title.split())

        # Extract summary if available
        summary_keys = ['Summary', 'Abstract', 'Zusammenfassung', 'Kurzbeschreibung']
        summary = next((sections[key] for key in summary_keys if key in sections), "")

        # Extract references if available
        reference_keys = ['References', 'Referenzen', 'Quellen', 'Sources']
        references = next((sections[key] for key in reference_keys if key in sections), "")

        # Clean the main content based on language
        cleaned_text = self.normalize_unicode(main_content)
        cleaned_text = self.normalize_whitespace(cleaned_text)
        cleaned_text = self.standardize_dates(cleaned_text)

        if language == 'de':
            cleaned_text = self.process_german_text(cleaned_text)

        # Create basic metadata
        result = {
            "filename": original_filename,
            "title": title,
            "language": language,
            "summary": summary,
            "main_content": main_content,
            "main_section_title": main_section_title,
            "references": references,
            "all_sections": list(sections.keys()),
            "cleaned_text": cleaned_text,
            "word_count": len(cleaned_text.split()),
            "extraction_date": datetime.datetime.now().isoformat(),
        }

        # Add advanced metadata
        result["keywords"] = self.extract_keywords(cleaned_text, language)
        result["named_entities"] = self.extract_named_entities(cleaned_text, language)

        # Add a unique identifier for the document
        result["doc_id"] = f"{language}-{original_filename.replace('.html', '')}"

        return result

    def process_file(self, file_path: str) -> Dict[str, Any]:
        """Process a single markdown file."""
        filename = os.path.basename(file_path)
        content = self.read_markdown_file(file_path)

        if not content:
            logger.warning(f"No content found in {file_path}")
            return {}

        try:
            result = self.parse_markdown_content(content, filename)
            logger.info(f"Successfully processed {filename} ({result['language']})")
            return result
        except Exception as e:
            logger.error(f"Error processing {filename}: {e}")
            return {}

    def process_directory(self, directory_path: str) -> List[Dict[str, Any]]:
        """Process all markdown files in the given directory."""
        results = []

        try:
            file_paths = [os.path.join(directory_path, f) for f in os.listdir(directory_path)
                         if f.endswith('.md')]

            logger.info(f"Found {len(file_paths)} markdown files in {directory_path}")

            for file_path in file_paths:
                result = self.process_file(file_path)
                if result:
                    results.append(result)

            return results
        except Exception as e:
            logger.error(f"Error processing directory {directory_path}: {e}")
            return []

    def save_results(self, results: List[Dict[str, Any]], output_dir: str,
                     save_json: bool = True, save_csv: bool = True, save_sqlite: bool = False) -> None:
        """Save processing results in various formats."""
        os.makedirs(output_dir, exist_ok=True)

        # Count by language
        languages = {}
        for result in results:
            lang = result.get('language', 'unknown')
            languages[lang] = languages.get(lang, 0) + 1

        language_counts = ", ".join([f"{lang}: {count}" for lang, count in languages.items()])
        logger.info(f"Processed {len(results)} articles ({language_counts})")

        if save_json:
            json_path = os.path.join(output_dir, 'processed_articles.json')
            with open(json_path, 'w', encoding='utf-8') as f:
                json.dump(results, f, ensure_ascii=False, indent=2)
            logger.info(f"Saved JSON to {json_path}")

        if save_csv:
            # For CSV, we'll flatten nested structures
            flattened_results = []
            for result in results:
                flat_result = result.copy()
                # Convert lists to strings
                for key, value in flat_result.items():
                    if isinstance(value, list):
                        flat_result[key] = json.dumps(value)
                flattened_results.append(flat_result)

            csv_path = os.path.join(output_dir, 'processed_articles.csv')
            df = pd.DataFrame(flattened_results)
            df.to_csv(csv_path, index=False, encoding='utf-8')
            logger.info(f"Saved CSV to {csv_path}")

        if save_sqlite:
            import sqlite3
            db_path = os.path.join(output_dir, 'articles.db')

            # Convert data for SQLite
            flattened_results = []
            for result in results:
                flat_result = result.copy()
                # Convert lists and dicts to JSON strings
                for key, value in flat_result.items():
                    if isinstance(value, (list, dict)):
                        flat_result[key] = json.dumps(value)
                flattened_results.append(flat_result)

            # Create DataFrame and save to SQLite
            df = pd.DataFrame(flattened_results)

            # Connect to SQLite database
            conn = sqlite3.connect(db_path)
            df.to_sql('articles', conn, if_exists='replace', index=False)

            # Create indices for faster retrieval
            cursor = conn.cursor()
            cursor.execute("CREATE INDEX IF NOT EXISTS idx_language ON articles (language)")
            cursor.execute("CREATE INDEX IF NOT EXISTS idx_title ON articles (title)")
            cursor.execute("CREATE INDEX IF NOT EXISTS idx_doc_id ON articles (doc_id)")
            conn.commit()
            conn.close()

            logger.info(f"Saved SQLite database to {db_path}")


def main(input_dir: str, output_dir: str,
         save_json: bool = True, save_csv: bool = True, save_sqlite: bool = False) -> None:
    """
    Process all markdown files in the input directory and save results to the output directory.

    Args:
        input_dir: Directory containing markdown files to process
        output_dir: Directory to save processed results
        save_json: Whether to save results as JSON
        save_csv: Whether to save results as CSV
        save_sqlite: Whether to save results in SQLite database
    """
    processor = TextProcessor()

    logger.info(f"Starting processing of markdown files in {input_dir}")
    results = processor.process_directory(input_dir)

    if results:
        processor.save_results(results, output_dir, save_json, save_csv, save_sqlite)
        logger.info(f"Processing complete. Results saved to {output_dir}")
    else:
        logger.error("No results generated. Check input directory and file formats.")

def main(input_dir: str, output_dir: str,
         save_json: bool = True, save_csv: bool = True, save_sqlite: bool = False) -> None:
    """
    Process all markdown files in the input directory and save results to the output directory.

    Args:
        input_dir: Directory containing markdown files to process
        output_dir: Directory to save processed results
        save_json: Whether to save results as JSON
        save_csv: Whether to save results as CSV
        save_sqlite: Whether to save results in SQLite database
    """
    processor = TextProcessor()

    logger.info(f"Starting processing of markdown files in {input_dir}")
    results = processor.process_directory(input_dir)

    if results:
        processor.save_results(results, output_dir, save_json, save_csv, save_sqlite)
        logger.info(f"Processing complete. Results saved to {output_dir}")
    else:
        logger.error("No results generated. Check input directory and file formats.")


[nltk_data] Error downloading 'punkt' from
[nltk_data]     <https://raw.githubusercontent.com/nltk/nltk_data/gh-
[nltk_data]     pages/packages/tokenizers/punkt.zip>:   HTTP Error
[nltk_data]     429: Too Many Requests


In [ ]:

if name == "main":
    parser = argparse.ArgumentParser(description="Process multilingual markdown news articles")
    parser.add_argument("--input", "-i", required=True, help="Input directory containing markdown files")
    parser.add_argument("--output", "-o", required=True, help="Output directory for processed files")
    parser.add_argument("--json", action="store_true", default=True, help="Save results as JSON")
    parser.add_argument("--csv", action="store_true", default=True, help="Save results as CSV")
    parser.add_argument("--sqlite", action="store_true", default=False, help="Save results in SQLite database")

    args = parser.parse_args()

    main(args.input, args.output, args.json, args.csv, args.sqlite)


In [ ]:
# Import necessary libraries
import sys
import argparse
from IPython.display import display
import nltk
nltk.download('punkt', download_dir='/usr/local/nltk_data')

# Override sys.argv with your desired arguments
sys.argv = ['step1_2_script.ipynb',
            '--input', 'news-qa-ethz1/notebooks/parsed_markdown',
            '--output', 'news-qa-ethz1/notebooks/processed_data']

# Create the parser and parse arguments
parser = argparse.ArgumentParser(description="Process multilingual markdown news articles")
parser.add_argument("--input", "-i", required=True, help="Input directory containing markdown files")
parser.add_argument("--output", "-o", required=True, help="Output directory for processed files")
parser.add_argument("--json", action="store_true", default=True, help="Save results as JSON")
parser.add_argument("--csv", action="store_true", default=True, help="Save results as CSV")
parser.add_argument("--sqlite", action="store_true", default=False, help="Save results in SQLite database")

args = parser.parse_args()

# Display the args to confirm they're correct
display(args)

# Call the main function with the parsed arguments
main(args.input, args.output, args.json, args.csv, args.sqlite)

[nltk_data] Downloading package punkt to /usr/local/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Namespace(input='news-qa-ethz1/notebooks/parsed_markdown', output='news-qa-ethz1/notebooks/processed_data', json=True, csv=True, sqlite=False)

ERROR:TextProcessor:Error processing die-eth-karte-erhaelt-ein-neues-design.md: 
**********************************************************************
  Resource punkt_tab not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('punkt_tab')
  
  For more information see: https://www.nltk.org/data.html

  Attempted to load tokenizers/punkt_tab/english/

  Searched in:
    - '/root/nltk_data'
    - '/usr/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
    - '/usr/local/nltk_data'
**********************************************************************

ERROR:TextProcessor:Error processing detecting-storms-thanks-to-gps.md: 
**********************************************************************
  Resource punkt_tab not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nlt

In [ ]:
!ls news-qa-ethz1/notebooks/

01_html_parsing_comparison.ipynb			parsed_markdown
02_hybrid_parsing.ipynb					step1_2_script.ipynb
1_2_Multilingual_Text_Preprocessing_and_Cleaning.ipynb


## Trial functions for Processing

## version 1

In [ ]:
def basic_text_cleaning(text):
    """
    Perform basic text cleaning operations.

    Args:
        text (str): Raw text input

    Returns:
        str: Cleaned text
    """
    import re
    import unicodedata

    # Normalize Unicode characters
    text = unicodedata.normalize('NFKC', text)

    # Replace multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text)

    # Remove extra line breaks, but preserve paragraph breaks (double line breaks)
    text = re.sub(r'\n{3,}', '\n\n', text)

    # Strip leading/trailing whitespace
    text = text.strip()

    return text

# Apply text cleaning to all parsed articles
cleaned_articles = {}

for filename, content in parsed_articles.items():
    cleaned_articles[filename] = basic_text_cleaning(content)

# Show a before/after comparison for the first article
if parsed_articles:
    sample_filename = list(parsed_articles.keys())[0]
    print("BEFORE CLEANING:")
    print("-" * 50)
    print(parsed_articles[sample_filename][:500] + "...")  # Display first 500 chars
    print("\nAFTER CLEANING:")
    print("-" * 50)
    print(cleaned_articles[sample_filename][:500] + "...")  # Display first 500 chars

    # Print some statistics
    before_chars = sum(len(text) for text in parsed_articles.values())
    after_chars = sum(len(text) for text in cleaned_articles.values())
    print(f"\nTotal characters before cleaning: {before_chars}")
    print(f"Total characters after cleaning: {after_chars}")
    print(f"Character reduction: {before_chars - after_chars} ({(before_chars - after_chars) / before_chars * 100:.2f}%)")

BEFORE CLEANING:
--------------------------------------------------
# die-eth-karte-erhaelt-ein-neues-design.html

## Main article

In diesem Sommer erhalten Angehörige der ETH Zürich eine neue ETH-Karte. Die Karte erscheint dabei im neuen Design: Das 2008 eingeführte, grüne Erscheinungsbild wird ersetzt durch ein Blau aus dem Corporate Design der ETH Zürich.

Die Karte ist mit einer weiterentwickelten Version des bisher verwendeten RFID-Chips ausgestattet. Die elektronischen Funktionen der ETH-Karte und das Kartenmanagementsystem sind dieselben wie bisher. Eben...

AFTER CLEANING:
--------------------------------------------------
# die-eth-karte-erhaelt-ein-neues-design.html ## Main article In diesem Sommer erhalten Angehörige der ETH Zürich eine neue ETH-Karte. Die Karte erscheint dabei im neuen Design: Das 2008 eingeführte, grüne Erscheinungsbild wird ersetzt durch ein Blau aus dem Corporate Design der ETH Zürich. Die Karte ist mit einer weiterentwickelten Version des bisher verwen

In [ ]:
def examine_files(parsed_articles):
    """
    Examine the first few articles to understand their structure
    """
    print("\nExamining the first file content:")
    first_file = list(parsed_articles.keys())[1]
    content = parsed_articles[first_file]

    # Print the first 500 characters
    print(f"First 500 characters of {first_file}:")
    print(content[:500])

    # Print the file structure
    print("\nFile structure:")
    lines = content.split('\n')
    for i, line in enumerate(lines[:10]):  # Print first 10 lines
        print(f"Line {i}: {line}")

    # Check if there are any sections
    section_pattern = r'## (.+?)\n'
    sections = re.findall(section_pattern, content)
    print(f"\nFound {len(sections)} sections: {sections}")

    return

examine_files(parsed_articles)



Examining the first file content:
First 500 characters of detecting-storms-thanks-to-gps.md:
# detecting-storms-thanks-to-gps.html

## In brief

An exceptionally severe storm swept over Zurich on 13 July 2021 shortly before 2 a.m.: howling squalls, constant lightning and torrential rain woke people up with a start. Benedikt Soja, Professor of Space Geodesy, also got little sleep that night. “It was one of the most severe storms I’ve ever witnessed. I woke up in the middle of the night and could see the storm raging through the window,” he remembers.

The scale of the storm was evident t

File structure:
Line 0: # detecting-storms-thanks-to-gps.html
Line 1: 
Line 2: ## In brief
Line 3: 
Line 4: An exceptionally severe storm swept over Zurich on 13 July 2021 shortly before 2 a.m.: howling squalls, constant lightning and torrential rain woke people up with a start. Benedikt Soja, Professor of Space Geodesy, also got little sleep that night. “It was one of the most severe storms I’ve ever

## version 2

In [ ]:
!pip install textstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.3/105.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.4/939.4 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 49.8 MB/s eta 0:00:00


In [ ]:
import os
import re
import unicodedata
import json
import pandas as pd
from langdetect import detect
import spacy
from tqdm import tqdm
from datetime import datetime
import textstat
from nltk.sentiment import SentimentIntensityAnalyzer

# Function 1: Basic Text Cleaning
def basic_text_cleaning(text):
    """
    Perform basic text cleaning operations.
    """
    # Normalize Unicode characters
    text = unicodedata.normalize('NFKC', text)

    # Replace multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text)

    # Remove extra line breaks, but preserve paragraph breaks
    text = re.sub(r'\n{3,}', '\n\n', text)

    # Strip leading/trailing whitespace
    text = text.strip()

    return text

# Function 2: Language Detection
def detect_language(text):
    """
    Detect the language of the given text.
    """
    try:
        # Use only the first 1000 characters for faster detection
        sample = text[:1000]
        language = detect(sample)
        return language
    except:
        return 'unknown'

# Function 3: Extract Article Structure
def extract_article_structure(markdown_text):
    """
    Extract article structure from markdown text with better section detection
    """
    # Initialize article structure
    article = {
        'original_filename': '',
        'sections': {},
    }

    # Extract original filename from the header line (first line with # prefix)
    header_match = re.search(r'^# (.+?)$', markdown_text, re.MULTILINE)
    if header_match:
        article['original_filename'] = header_match.group(1).strip()

    # Extract sections
    # Look for sections starting with ## and ending at the next ## or the end of text
    section_pattern = r'## (.+?)\n(.*?)(?=\n## |\Z)'
    section_matches = re.finditer(section_pattern, markdown_text, re.DOTALL)

    for match in section_matches:
        section_title = match.group(1).strip()
        section_content = match.group(2).strip()
        article['sections'][section_title] = section_content

    # If no sections were found, treat the entire content as one section
    if not article['sections']:
        # Skip the header line if it exists
        content = markdown_text
        if header_match:
            header_end = header_match.end()
            content = markdown_text[header_end:].strip()

        if content:
            article['sections']['Main'] = content

    return article
# Function 4: Extract Title and Date
def extract_title_date(article_dict):
    """
    Extract title and publication date from article structure.
    """
    # Typically title might be in first section or in a specific section
    title = ""
    date_str = ""

    # Common section names that might contain titles
    title_sections = ['Title', 'Headline', 'In brief']

    for section_name in title_sections:
        if section_name in article_dict['sections']:
            # Take first line as title
            content = article_dict['sections'][section_name]
            lines = content.split('\n')
            if lines:
                title = lines[0].strip()
                break

    # If no title found in specific sections, use first non-empty line from any section
    if not title:
        for content in article_dict['sections'].values():
            lines = content.split('\n')
            for line in lines:
                if line.strip():
                    title = line.strip()
                    break
            if title:
                break

    # Extract date from text using regex patterns
    date_patterns = [
        r'(\d{1,2})[./](\d{1,2})[./](\d{2,4})',  # DD/MM/YYYY or MM/DD/YYYY
        r'(\d{4})-(\d{1,2})-(\d{1,2})',          # YYYY-MM-DD
        r'(\d{1,2})\.(\d{1,2})\.(\d{2,4})'       # DD.MM.YYYY (German format)
    ]

    for section_content in article_dict['sections'].values():
        for pattern in date_patterns:
            date_match = re.search(pattern, section_content)
            if date_match:
                date_str = date_match.group(0)
                # Format the date as YYYY-MM-DD
                parts = re.split(r'[./\-]', date_str)
                if len(parts) == 3:
                    # Determine year, month, day based on pattern
                    if len(parts[2]) == 4:  # DD/MM/YYYY
                        date_str = f"{parts[2]}-{parts[1].zfill(2)}-{parts[0].zfill(2)}"
                    elif len(parts[0]) == 4:  # YYYY-MM-DD
                        date_str = f"{parts[0]}-{parts[1].zfill(2)}-{parts[2].zfill(2)}"
                    else:  # Default to YYYY-MM-DD
                        if int(parts[2]) > 31:  # Year is last
                            date_str = f"{parts[2]}-{parts[1].zfill(2)}-{parts[0].zfill(2)}"
                        else:  # Assume European format DD.MM.YYYY
                            # For 2-digit years, assume 20xx
                            year = parts[2] if len(parts[2]) == 4 else f"20{parts[2]}"
                            date_str = f"{year}-{parts[1].zfill(2)}-{parts[0].zfill(2)}"
                break
        if date_str:
            break

    return title, date_str

# Function 5: Extract Source
def extract_source(article_dict):
    """
    Extract the likely source of the article.
    """
    source = ""

    # Check for source in specific sections
    source_sections = ['Source', 'Reference', 'About', 'Author']

    for section_name in source_sections:
        if section_name in article_dict['sections']:
            source = article_dict['sections'][section_name].strip()
            break

    # Look for patterns like "Source:", "From:", etc.
    if not source:
        source_patterns = [
            r'(?:Source|Quelle|Von|From|By):\s*([^\n]+)',
            r'(?:Copyright|©)\s*([^\n]+)'
        ]

        for content in article_dict['sections'].values():
            for pattern in source_patterns:
                source_match = re.search(pattern, content)
                if source_match:
                    source = source_match.group(1).strip()
                    break
            if source:
                break

    # Extract filename as last resort
    if not source and article_dict['original_filename']:
        # Extract domain or organization from filename
        filename = article_dict['original_filename']
        domain_match = re.search(r'(\w+)\.(?:com|org|edu|de|ch)', filename)
        if domain_match:
            source = domain_match.group(1).capitalize()

    # Default source if nothing found
    if not source:
        source = "Unknown source"

    return source

# Function 6: Extract Main Content
def extract_main_content(article_dict):
    """
    Extract and combine the main content sections.
    """
    # Skip these sections when building main content
    skip_sections = ['References', 'Source', 'About', 'Author', 'Copyright']

    # Combine content from all relevant sections
    main_content_parts = []

    for section_name, content in article_dict['sections'].items():
        if section_name not in skip_sections and content.strip():
            main_content_parts.append(content)

    main_content = " ".join(main_content_parts)

    # If there's original_filename but no content, use that as fallback
    if not main_content and article_dict.get('original_filename'):
        main_content = f"Content related to {article_dict['original_filename']}"

    return main_content

# Function 7: Extract Named Entities
def extract_named_entities(text, language):
    """
    Extract named entities from text using spaCy.
    """
    entities = []

    try:
        # Load appropriate language model
        if language == 'de':
            nlp = spacy.load('de_core_news_sm')
        elif language == 'en':
            nlp = spacy.load('en_core_web_sm')
        else:
            return entities

        # Process only first 10000 characters to avoid memory issues
        doc = nlp(text[:10000])

        # Extract named entities and keep only unique ones
        entity_set = set()
        for ent in doc.ents:
            if ent.text not in entity_set:
                entity_set.add(ent.text)
                entities.append(ent.text)
    except Exception as e:
        print(f"Error extracting entities: {e}")

    return entities[:10]  # Limit to top 10 entities

# Function 8: Extract Topics
def extract_topics(text, language):
    """
    Extract topics from the text.
    """
    # Custom topic extraction based on keywords and patterns
    topics = set()

    # Topic keywords for German
    de_topics = {
        "Preiserhöhungen": ["preis", "erhöhung", "teuer", "kosten"],
        "Hochschulgastronomie": ["mensa", "catering", "verpflegung", "gastro"],
        "Inflation": ["inflation", "teuerung", "anstieg"],
        "Catering": ["catering", "gastronomie", "verpflegung"],
        "Coronapandemie": ["corona", "pandemie", "covid"],
        "Subventionen": ["subvention", "unterstützung", "förderung"]
    }

    # Topic keywords for English
    en_topics = {
        "Price Increases": ["price", "increase", "expensive", "cost"],
        "University Catering": ["cafeteria", "catering", "food", "dining"],
        "Inflation": ["inflation", "price hike", "rising costs"],
        "Catering": ["catering", "food service", "dining"],
        "Coronavirus": ["corona", "pandemic", "covid"],
        "Subsidies": ["subsidy", "subsidies", "support", "funding"]
    }

    # Select topic dictionary based on language
    topic_dict = de_topics if language == 'de' else en_topics

    # Check text for keywords related to each topic
    for topic, keywords in topic_dict.items():
        for keyword in keywords:
            if keyword.lower() in text.lower():
                topics.add(topic)
                break

    return list(topics)[:6]  # Limit to top 6 topics

# Function 9: Extract Keywords
def extract_keywords(text, language, n=7):
    """
    Extract important keywords from the text.
    """
    try:
        # Load appropriate language model
        if language == 'de':
            nlp = spacy.load('de_core_news_sm')
        elif language == 'en':
            nlp = spacy.load('en_core_web_sm')
        else:
            return []

        # Process the text
        doc = nlp(text[:10000])  # Limit to avoid memory issues

        # Remove stopwords and punctuation
        words = [token.text for token in doc if not token.is_stop and not token.is_punct and len(token.text) > 2]

        # Count word frequencies
        from collections import Counter
        word_freq = Counter(words)

        # Return top n keywords
        keywords = [word for word, _ in word_freq.most_common(n)]
        return keywords
    except:
        return []

# Function 10: Generate Summary
def generate_summary(text, max_words=50):
    """
    Generate a more comprehensive summary.
    """
    # Split text into sentences
    sentences = re.split(r'(?<=[.!?])\s+', text)

    # Take first few sentences until we reach max_words
    summary = ""
    word_count = 0

    for sentence in sentences:
        words = sentence.split()
        if word_count + len(words) <= max_words:
            summary += sentence + " "
            word_count += len(words)
        else:
            break

    return summary.strip()

# Function 11: Generate Rich Metadata
def generate_rich_metadata(text, language):
    """
    Generate rich metadata for the article.
    """
    # Install required packages
    try:
        import nltk
        nltk.download('vader_lexicon', quiet=True)
    except ImportError:
        import sys
        !{sys.executable} -m pip install nltk textstat
        import nltk
        nltk.download('vader_lexicon', quiet=True)

    import textstat
    from nltk.sentiment import SentimentIntensityAnalyzer

    # Count words
    word_count = len(text.split())

    # Determine readability
    if language == 'de':
        readability = "leicht verständlich"  # Simplified for German
    else:
        reading_ease = textstat.flesch_reading_ease(text)
        if reading_ease > 70:
            readability = "easy to read"
        elif reading_ease > 50:
            readability = "moderately readable"
        else:
            readability = "difficult to read"

    # Sentiment analysis
    sentiment = "neutral"
    try:
        sia = SentimentIntensityAnalyzer()
        sentiment_score = sia.polarity_scores(text)

        if sentiment_score['compound'] > 0.2:
            sentiment = "positiv" if language == 'de' else "positive"
        elif sentiment_score['compound'] < -0.2:
            sentiment = "negativ" if language == 'de' else "negative"
        else:
            sentiment = "neutral"

        # Add intensity for more nuance
        if abs(sentiment_score['compound']) > 0.5:
            if language == 'de':
                sentiment = "stark " + sentiment if sentiment != "neutral" else sentiment
            else:
                sentiment = "strongly " + sentiment if sentiment != "neutral" else sentiment
        elif 0.2 < abs(sentiment_score['compound']) < 0.5:
            if language == 'de':
                sentiment = "neutral bis " + sentiment if sentiment != "neutral" else sentiment
            else:
                sentiment = "neutral to " + sentiment if sentiment != "neutral" else sentiment
    except:
        pass

    # Check for compound words and umlauts (German)
    contains_compound_words = False
    contains_umlauts = False

    if language == 'de':
        # Check for compound words (simplified)
        long_words = [word for word in text.split() if len(word) > 15]
        contains_compound_words = len(long_words) > 0

        # Check for umlauts
        contains_umlauts = bool(re.search(r'[äöüÄÖÜß]', text))

    # Determine audience type and context tags
    audience_types = []
    context_tags = []

    # Common audience types
    audience_keywords = {
        'Studierende': ['student', 'studierend', 'studenten'],
        'Mitarbeitende': ['mitarbeiter', 'personal', 'staff', 'faculty'],
        'Hochschulleitung': ['leitung', 'präsident', 'rektor', 'leadership'],
        'Forschende': ['forscher', 'wissenschaft', 'research'],
        'Öffentlichkeit': ['public', 'öffentlich']
    }

    # Context tags
    context_keywords = {
        'Finanzielle Lage': ['finanz', 'kosten', 'budget'],
        'Verpflegung': ['mensa', 'essen', 'food', 'dining'],
        'Corona-Folgen': ['corona', 'covid', 'pandemic'],
        'ETH intern': ['eth', 'zürich', 'intern'],
        'Forschung': ['forschung', 'research', 'studie'],
        'Lehre': ['lehre', 'unterricht', 'kurs', 'course'],
        'Nachhaltigkeit': ['nachhaltig', 'sustainable', 'umwelt']
    }

    # Check for audience keywords
    for audience, keywords in audience_keywords.items():
        for keyword in keywords:
            if keyword.lower() in text.lower():
                audience_types.append(audience)
                break

    # Check for context keywords
    for tag, keywords in context_keywords.items():
        for keyword in keywords:
            if keyword.lower() in text.lower():
                context_tags.append(tag)
                break

    # Limit audience types and context tags
    audience_types = audience_types[:3]
    context_tags = context_tags[:4]

    # Create placeholder for embedding vector
    embedding_vector = "[...]"

    # Build rich metadata dictionary
    rich_metadata = {
        "document_length_words": word_count,
        "readability_score": readability,
        "sentiment": sentiment,
        "contains_compound_words": contains_compound_words,
        "contains_umlauts": contains_umlauts,
        "embedding_vector": embedding_vector,
        "audience_type": audience_types,
        "context_tags": context_tags
    }

    return rich_metadata

# Main processing function
def process_article(markdown_text, filename):
    """
    Process a single article through all cleaning steps with better debugging
    """
    print(f"\nProcessing file: {filename}")
    print(f"Original content length: {len(markdown_text)} characters")

    # Step 1: Clean the text
    cleaned_text = basic_text_cleaning(markdown_text)
    print(f"Cleaned text length: {len(cleaned_text)} characters")

    # Step 2: Extract article structure
    article_structure = extract_article_structure(cleaned_text)
    print(f"Extracted sections: {list(article_structure['sections'].keys())}")

    # Step 3: Combine all sections' content
    all_content = " ".join(article_structure['sections'].values())
    print(f"Combined content length: {len(all_content)} characters")

    # Step 4: Detect language (with fallback)
    language = "unknown"
    if all_content:
        try:
            language = detect_language(all_content)
            print(f"Detected language: {language}")
        except:
            # If language detection fails, fall back to German (most likely)
            language = "de"
            print("Language detection failed, defaulting to German")
    else:
        print("WARNING: No content extracted for language detection")

    # Step 5: Extract title (with fallback)
    title = ""
    # First try to use the original filename
    if article_structure.get('original_filename'):
        # Convert filename to title format
        title_from_filename = article_structure['original_filename']
        # Remove file extension if present
        title_from_filename = re.sub(r'\.(html|md)$', '', title_from_filename)
        # Replace hyphens and underscores with spaces
        title_from_filename = title_from_filename.replace('-', ' ').replace('_', ' ')
        # Capitalize first letter of each word
        title = ' '.join(word.capitalize() for word in title_from_filename.split())

    # If no title from filename, try to extract from content
    if not title:
        # Extract first non-empty line from any section
        for content in article_structure['sections'].values():
            lines = [line for line in content.split('\n') if line.strip()]
            if lines:
                title = lines[0].strip()
                break

    print(f"Extracted title: {title[:50]}...")

    # Step 6: Extract main content (ensuring it's not empty)
    main_content = extract_main_content(article_structure)
    if not main_content:
        main_content = cleaned_text  # Fall back to entire cleaned text
        print("WARNING: Falling back to full content")
    else:
        print(f"Main content length: {len(main_content)} characters")

    # Rest of processing...

    # Create final structured document
    processed_article = {
        "language": language,
        "title": title,
        "date": "",  # We'll fill this separately
        "source": "ETH Zürich",  # Default source
        "main_content": main_content,
        "named_entities": [],
        "topics": [],
        "keywords": [],
        "summary": main_content[:200] + "..." if len(main_content) > 200 else main_content,
        "rich_metadata": {
            "document_length_words": len(main_content.split()),
            "readability_score": "leicht verständlich" if language == "de" else "easy to read",
            "sentiment": "neutral",
            "contains_compound_words": bool(re.search(r'\w{15,}', main_content)) if language == "de" else False,
            "contains_umlauts": bool(re.search(r'[äöüÄÖÜß]', main_content)),
            "embedding_vector": "[...]",
            "audience_type": ["Studierende", "Mitarbeitende"],
            "context_tags": ["ETH intern"]
        }
    }

    return processed_article

# Main execution script
def process_all_articles(parsed_articles):
    """
    Process all articles and return a dataset.
    """
    processed_data = {}

    # Install required packages if not already installed
    try:
        import spacy
        import nltk
        import textstat
        # Download language models if needed
        if not spacy.util.is_package('en_core_web_sm'):
            print("Downloading English language model...")
            spacy.cli.download('en_core_web_sm')
        if not spacy.util.is_package('de_core_news_sm'):
            print("Downloading German language model...")
            spacy.cli.download('de_core_news_sm')
    except ImportError:
        print("Installing required packages...")
        import sys
        !{sys.executable} -m pip install spacy langdetect tqdm nltk textstat
        import spacy
        print("Downloading language models...")
        !{sys.executable} -m spacy download en_core_web_sm
        !{sys.executable} -m spacy download de_core_news_sm

    # Process each article
    print(f"Processing {len(parsed_articles)} articles...")
    for filename, content in tqdm(parsed_articles.items()):
        try:
            processed = process_article(content, filename)
            processed_data[filename] = processed
        except Exception as e:
            print(f"Error processing {filename}: {e}")

    # Save full data to JSON (including nested structures)
    with open('processed_articles.json', 'w', encoding='utf-8') as f:
        json.dump(processed_data, f, ensure_ascii=False, indent=2)

    print(f"Processing complete. Saved {len(processed_data)} articles to JSON.")

    return processed_data



In [ ]:
# Execute the processing
folder_path = 'news-qa-ethz1/notebooks/parsed_markdown'

# Ensure the folder exists
if not os.path.exists(folder_path):
    raise FileNotFoundError(f"Folder not found: {folder_path}")

# List all files in the folder
files = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
print(f"Found {len(files)} files.")

# Read all files into a dictionary
parsed_articles = {}

for file in files:
    full_path = os.path.join(folder_path, file)
    with open(full_path, 'r', encoding='utf-8') as f:
        parsed_articles[file] = f.read()

# Process all articles
processed_data = process_all_articles(parsed_articles)

# Print some statistics
languages = [article["language"] for article in processed_data.values()]
print(f"\nLanguage distribution:")
for lang in set(languages):
    count = languages.count(lang)
    print(f"  - {lang}: {count} articles ({count/len(languages)*100:.1f}%)")

titles = [article["title"] for article in processed_data.values() if article["title"]]
print(f"\nNumber of articles with title: {len(titles)}")

dates = [article["date"] for article in processed_data.values() if article["date"]]
print(f"Number of articles with date: {len(dates)}")

# Preview one processed article
if processed_data:
    sample_key = list(processed_data.keys())[0]
    print(f"\nSample processed article ('{sample_key}'):")
    print(json.dumps(processed_data[sample_key], indent=2, ensure_ascii=False)[:500] + "...")

Found 4 files.
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Processing 4 articles...


100%|██████████| 4/4 [00:00<00:00, 12.51it/s]

Processing complete. Saved 4 articles to JSON.

Language distribution:
  - unknown: 4 articles (100.0%)

Number of articles with title: 0
Number of articles with date: 0

Sample processed article ('die-eth-karte-erhaelt-ein-neues-design.md'):
{
  "language": "unknown",
  "title": "",
  "date": "",
  "source": "Ethz",
  "main_content": "",
  "named_entities": [],
  "topics": [],
  "keywords": [],
  "summary": "",
  "rich_metadata": {
    "document_length_words": 0,
    "readability_score": "easy to read",
    "sentiment": "neutral",
    "contains_compound_words": false,
    "contains_umlauts": false,
    "embedding_vector": "[...]",
    "audience_type": [],
    "context_tags": []
  }
}...


## version 3

In [ ]:
import os
import re
import unicodedata
import json
import pandas as pd
from langdetect import detect
from tqdm import tqdm
import datetime

# Function 1: Basic Text Cleaning
def basic_text_cleaning(text):
    """
    Perform basic text cleaning operations.
    """
    # Normalize Unicode characters
    text = unicodedata.normalize('NFKC', text)

    # Replace multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text)

    # Remove extra line breaks, but preserve paragraph breaks
    text = re.sub(r'\n{3,}', '\n\n', text)

    # Strip leading/trailing whitespace
    text = text.strip()

    return text

# Function 2: Language Detection
def detect_language(text):
    """
    Detect the language of the given text.
    """
    try:
        # Use only the first 1000 characters for faster detection
        sample = text[:1000]
        language = detect(sample)
        return language
    except:
        # Default to German if detection fails
        return "de"

# Function 3: Extract Article Structure
def extract_article_structure(markdown_text):
    """
    Extract article structure from markdown text.
    """
    # Initialize article structure
    article = {
        'original_filename': '',
        'sections': {},
    }

    # Extract original filename from the header line
    header_match = re.search(r'^# (.+?)$', markdown_text, re.MULTILINE)
    if header_match:
        article['original_filename'] = header_match.group(1).strip()

    # Extract sections
    section_pattern = r'## (.+?)\n(.*?)(?=\n## |\Z)'
    section_matches = re.finditer(section_pattern, markdown_text, re.DOTALL)

    for match in section_matches:
        section_title = match.group(1).strip()
        section_content = match.group(2).strip()
        article['sections'][section_title] = section_content

    return article

# Function 4: Extract Title
def extract_title(article_dict, filename):
    """
    Extract title from the article structure or filename.
    """
    title = ""

    # Try to extract from sections like "In brief" or "Main article"
    title_sections = ["In brief", "Main article", "Title", "Headline"]

    for section_name in title_sections:
        if section_name in article_dict['sections']:
            content = article_dict['sections'][section_name]
            lines = content.split('\n')
            if lines:
                # Take first line as title
                title = lines[0].strip()
                break

    # If no title found, use filename
    if not title and article_dict['original_filename']:
        # Clean up the filename to create a title
        clean_filename = article_dict['original_filename'].replace('.html', '')
        # Replace hyphens with spaces and capitalize words
        title = ' '.join(word.capitalize() for word in clean_filename.split('-'))

    # If still no title, use the markdown filename
    if not title:
        clean_filename = filename.replace('.md', '')
        title = ' '.join(word.capitalize() for word in clean_filename.split('-'))

    return title

# Function 5: Extract Date
def extract_date(text):
    """
    Extract date from text.
    """
    # Various date patterns
    date_patterns = [
        r'(\d{1,2})\.(\d{1,2})\.(\d{4})',  # DD.MM.YYYY (German)
        r'(\d{1,2})[/\.](\d{1,2})[/\.](\d{4})',  # DD/MM/YYYY or MM/DD/YYYY
        r'(\d{4})-(\d{1,2})-(\d{1,2})',  # YYYY-MM-DD
        r'(\d{1,2})(?:st|nd|rd|th)? (?:of )?([A-Za-z]+)[,]? (\d{4})'  # 13th July 2021
    ]

    months = {
        'january': '01', 'february': '02', 'march': '03', 'april': '04',
        'may': '05', 'june': '06', 'july': '07', 'august': '08',
        'september': '09', 'october': '10', 'november': '11', 'december': '12',
        'jan': '01', 'feb': '02', 'mar': '03', 'apr': '04', 'jun': '06',
        'jul': '07', 'aug': '08', 'sep': '09', 'oct': '10', 'nov': '11', 'dec': '12'
    }

    for pattern in date_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            if len(match.groups()) == 3:
                if pattern == r'(\d{4})-(\d{1,2})-(\d{1,2})':  # YYYY-MM-DD
                    year, month, day = match.groups()
                elif pattern == r'(\d{1,2})(?:st|nd|rd|th)? (?:of )?([A-Za-z]+)[,]? (\d{4})':  # 13th July 2021
                    day, month_name, year = match.groups()
                    month = months.get(month_name.lower(), '01')
                else:  # DD.MM.YYYY or similar
                    day, month, year = match.groups()

                # Format as YYYY-MM-DD
                return f"{year}-{month.zfill(2)}-{day.zfill(2)}"

    # No date found, return current date
    today = datetime.datetime.now()
    return today.strftime('%Y-%m-%d')

# Function 6: Extract Source
def extract_source(article_dict):
    """
    Extract the source of the article.
    """
    # Check for "Reference" section
    if "Reference" in article_dict['sections']:
        return article_dict['sections']["Reference"]

    # Check for original filename
    if article_dict['original_filename']:
        if "eth" in article_dict['original_filename'].lower():
            return "ETH Zürich, interne Mitteilung / Hochschulkommunikation"

    # Default source
    return "ETH Zürich"

# Function 7: Extract Main Content
def extract_main_content(article_dict):
    """
    Extract the main content of the article.
    """
    # Skip these sections
    skip_sections = ["Reference", "Author", "Copyright"]

    # Combine all relevant sections
    content_sections = []

    for section_name, content in article_dict['sections'].items():
        if section_name not in skip_sections:
            content_sections.append(content)

    return " ".join(content_sections)

# Function 8: Extract Named Entities
def extract_named_entities(text, language):
    """
    Extract named entities from text.
    """
    # For simplicity, extract potential entities using regex patterns
    entities = []

    # Pattern for capitalized words (potential entities)
    entity_pattern = r'\b[A-Z][a-zäöüÄÖÜß]+(?:\s+[A-Z][a-zäöüÄÖÜß]+)*'

    # Find all matches
    matches = re.findall(entity_pattern, text)

    # Filter out common words and duplicates
    common_words = ["Der", "Die", "Das", "Ein", "Eine", "Eines", "The", "A", "An", "In", "On", "At"]
    unique_entities = set()

    for match in matches:
        if match not in common_words and len(match) > 1:
            unique_entities.add(match)

    # Find specific entities (e.g., ETH Zürich)
    specific_entities = ["ETH Zürich", "ETH", "Zürich", "Universität Zürich", "UZH"]
    for entity in specific_entities:
        if entity in text and entity not in unique_entities:
            unique_entities.add(entity)

    # Convert to list and limit
    entities = list(unique_entities)[:10]

    return entities

# Function 9: Extract Topics
def extract_topics(text, language):
    """
    Extract topics from text.
    """
    topics = []

    # Topic keywords (bilingual)
    topic_keywords = {
        "Hochschulgastronomie": ["mensa", "catering", "verpflegung", "gastro", "cafeteria"],
        "Forschung": ["forschung", "research", "wissenschaft", "science", "studie", "study"],
        "Technologie": ["technologie", "technology", "innovation", "entwicklung", "development"],
        "Hochschulpolitik": ["politik", "policy", "strategie", "strategy", "leitung", "management"],
        "Studium": ["studium", "studies", "studierenden", "students", "lehre", "teaching"],
        "Nachhaltigkeit": ["nachhaltig", "sustainable", "umwelt", "environment", "klima", "climate"],
        "Digitalisierung": ["digital", "computer", "software", "hardware", "daten", "data"],
        "Infrastruktur": ["infrastruktur", "infrastructure", "gebäude", "building", "campus"],
        "Internationales": ["international", "global", "weltweit", "worldwide", "ausland", "abroad"],
        "Preiserhöhungen": ["preis", "price", "kosten", "costs", "erhöhung", "increase"]
    }

    # Check for keywords in text
    for topic, keywords in topic_keywords.items():
        for keyword in keywords:
            if keyword.lower() in text.lower():
                topics.append(topic)
                break

    # If no topics found, add default topics
    if not topics:
        if "ETH-Karte" in text:
            topics = ["Infrastruktur", "Digitalisierung", "Hochschulpolitik"]
        elif "GPS" in text or "storm" in text.lower():
            topics = ["Forschung", "Technologie", "Nachhaltigkeit"]
        else:
            topics = ["ETH Zürich", "Hochschulpolitik"]

    return topics[:6]  # Limit to 6 topics

# Function 10: Extract Keywords
def extract_keywords(text, language):
    """
    Extract keywords from text.
    """
    # Simple keyword extraction using word frequency
    # Remove common stop words
    stop_words_de = ["der", "die", "das", "und", "in", "ist", "von", "den", "des", "mit", "zu", "für"]
    stop_words_en = ["the", "and", "in", "is", "of", "to", "that", "for", "on", "with", "as", "by"]

    stop_words = stop_words_de + stop_words_en

    # Tokenize and clean
    words = re.findall(r'\b\w+\b', text.lower())

    # Remove stop words and short words
    filtered_words = [word for word in words if word not in stop_words and len(word) > 3]

    # Count frequencies
    word_count = {}
    for word in filtered_words:
        if word in word_count:
            word_count[word] += 1
        else:
            word_count[word] = 1

    # Sort by frequency
    sorted_words = sorted(word_count.items(), key=lambda x: x[1], reverse=True)

    # Extract top keywords
    keywords = [word for word, count in sorted_words[:7]]

    # Add ETH Zürich as a keyword if relevant
    if "eth" in text.lower() and "ETH" not in keywords and "Zürich" not in keywords:
        keywords.insert(0, "ETH Zürich")

    return keywords

# Function 11: Generate Summary
def generate_summary(text, max_length=200):
    """
    Generate a summary of the text.
    """
    # Split into sentences
    sentences = re.split(r'(?<=[.!?])\s+', text)

    # Take first few sentences
    summary = ""
    for sentence in sentences:
        if len(summary) + len(sentence) <= max_length:
            summary += sentence + " "
        else:
            break

    summary = summary.strip()

    # Add ellipsis if truncated
    if len(summary) < len(text):
        summary += "..."

    return summary

# Function 12: Generate Rich Metadata
def generate_rich_metadata(text, language):
    """
    Generate rich metadata for the article.
    """
    # Count words
    words = re.findall(r'\b\w+\b', text)
    word_count = len(words)

    # Determine readability
    if language == 'de':
        readability = "leicht verständlich"
    else:
        readability = "easy to read"

    # Determine sentiment (simple heuristic)
    positive_words = ["gut", "besser", "positiv", "erfolgreich", "good", "better", "positive", "successful"]
    negative_words = ["schlecht", "problematisch", "negativ", "schwierig", "bad", "problematic", "negative", "difficult"]

    positive_count = sum(1 for word in positive_words if word.lower() in text.lower())
    negative_count = sum(1 for word in negative_words if word.lower() in text.lower())

    if positive_count > negative_count:
        sentiment = "positiv" if language == "de" else "positive"
    elif negative_count > positive_count:
        sentiment = "negativ" if language == "de" else "negative"
    else:
        sentiment = "neutral"

    # Check for compound words and umlauts
    contains_compound_words = bool(re.search(r'\b\w{15,}\b', text))
    contains_umlauts = bool(re.search(r'[äöüÄÖÜß]', text))

    # Determine audience type
    audience_type = []

    if re.search(r'\bstud(ent|ierend)', text.lower()):
        audience_type.append("Studierende")

    if re.search(r'\bmitarbeit|personal|staff\b', text.lower()):
        audience_type.append("Mitarbeitende")

    if re.search(r'\bleitung|direktion|management\b', text.lower()):
        audience_type.append("Hochschulleitung")

    # Default audience if none detected
    if not audience_type:
        audience_type = ["Studierende", "Mitarbeitende"]

    # Context tags
    context_tags = []

    if re.search(r'\bfinanz|kosten|budget\b', text.lower()):
        context_tags.append("Finanzielle Lage")

    if re.search(r'\bmensa|essen|verpflegung\b', text.lower()):
        context_tags.append("Verpflegung")

    if re.search(r'\bcorona|covid|pandemie\b', text.lower()):
        context_tags.append("Corona-Folgen")

    if re.search(r'\beth zürich|hochschule\b', text.lower()):
        context_tags.append("ETH intern")

    # Default context tag if none detected
    if not context_tags:
        context_tags.append("ETH intern")

    # Create rich metadata
    rich_metadata = {
        "document_length_words": word_count,
        "readability_score": readability,
        "sentiment": sentiment,
        "contains_compound_words": contains_compound_words,
        "contains_umlauts": contains_umlauts,
        "embedding_vector": "[...]",
        "audience_type": audience_type,
        "context_tags": context_tags
    }

    return rich_metadata

# Main processing function
def process_article(markdown_text, filename):
    """
    Process a single article through all cleaning steps.
    """
    # Step 1: Clean the text
    cleaned_text = basic_text_cleaning(markdown_text)

    # Step 2: Extract article structure
    article_structure = extract_article_structure(cleaned_text)

    # Step 3: Combine all sections' content
    all_content = " ".join(article_structure['sections'].values())

    # Step 4: Detect language
    language = detect_language(all_content)

    # Step 5: Extract title
    title = extract_title(article_structure, filename)

    # Step 6: Extract date
    date = extract_date(all_content)

    # Step 7: Extract source
    source = extract_source(article_structure)

    # Step 8: Extract main content
    main_content = extract_main_content(article_structure)

    # Step 9: Extract named entities
    named_entities = extract_named_entities(all_content, language)

    # Step 10: Extract topics
    topics = extract_topics(all_content, language)

    # Step 11: Extract keywords
    keywords = extract_keywords(all_content, language)

    # Step 12: Generate summary
    summary = generate_summary(main_content)

    # Step 13: Generate rich metadata
    rich_metadata = generate_rich_metadata(all_content, language)

    # Step 14: Create final structured document
    processed_article = {
        "language": language,
        "title": title,
        "date": date,
        "source": source,
        "main_content": main_content,
        "named_entities": named_entities,
        "topics": topics,
        "keywords": keywords,
        "summary": summary,
        "rich_metadata": rich_metadata
    }

    return processed_article

# Main execution function
def process_all_articles(parsed_articles):
    """
    Process all articles and return a dataset.
    """
    processed_data = {}

    # Process each article
    print(f"Processing {len(parsed_articles)} articles...")
    for filename, content in tqdm(parsed_articles.items()):
        try:
            processed = process_article(content, filename)
            processed_data[filename] = processed
        except Exception as e:
            print(f"Error processing {filename}: {e}")

    # Save full data to JSON
    with open('processed_articles.json', 'w', encoding='utf-8') as f:
        json.dump(processed_data, f, ensure_ascii=False, indent=2)

    print(f"Processing complete. Saved {len(processed_data)} articles to JSON.")

    return processed_data

# Execute the processing
folder_path = 'news-qa-ethz1/notebooks/parsed_markdown'

# Ensure the folder exists
if not os.path.exists(folder_path):
    raise FileNotFoundError(f"Folder not found: {folder_path}")

# List all files in the folder
files = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
print(f"Found {len(files)} files.")

# Read all files into a dictionary
parsed_articles = {}

for file in files:
    full_path = os.path.join(folder_path, file)
    with open(full_path, 'r', encoding='utf-8') as f:
        parsed_articles[file] = f.read()

# Process all articles
processed_data = process_all_articles(parsed_articles)

# Print some statistics
languages = [article["language"] for article in processed_data.values()]
print(f"\nLanguage distribution:")
for lang in set(languages):
    count = languages.count(lang)
    print(f"  - {lang}: {count} articles ({count/len(languages)*100:.1f}%)")

# Print a sample of the processed data
if processed_data:
    sample_key = list(processed_data.keys())[0]
    print(f"\nSample processed article ('{sample_key}'):")
    print(json.dumps(processed_data[sample_key], indent=2, ensure_ascii=False))

Found 4 files.
Processing 4 articles...


100%|██████████| 4/4 [00:00<00:00, 773.50it/s]

Processing complete. Saved 4 articles to JSON.

Language distribution:
  - de: 4 articles (100.0%)

Sample processed article ('die-eth-karte-erhaelt-ein-neues-design.md'):
{
  "language": "de",
  "title": "Die Eth Karte Erhaelt Ein Neues Design ## main article in diesem sommer erhalten angehörige der eth zürich eine neue eth Karte. die karte erscheint dabei im neuen design: das 2008 eingeführte, grüne erscheinungsbild wird ersetzt durch ein blau aus dem corporate design der eth zürich. die karte ist mit einer weiterentwickelten version des bisher verwendeten rfid Chips ausgestattet. die elektronischen funktionen der eth Karte und das kartenmanagementsystem sind dieselben wie bisher. ebenso bleiben die informationen auf der eth Karte gleich: eth Logo, mitarbeiterfoto, name, geburtsdatum, gültigkeitsdauer, berufliche rolle, organisationale zuordnung, identifikationsnummer und eventuell asvz Berechtigung. der austausch der alten durch die neue eth Karte beginnt ab juli für studierende und

## version 4: better text extraction

In [ ]:
import os
import re
import unicodedata
import json
from langdetect import detect
from tqdm import tqdm
import datetime

# Function 1: Basic Text Cleaning
def basic_text_cleaning(text):
    """
    Perform basic text cleaning operations.
    """
    # Normalize Unicode characters
    text = unicodedata.normalize('NFKC', text)

    # Strip leading/trailing whitespace
    text = text.strip()

    return text

# Function 2: Language Detection
def detect_language(text):
    """
    Detect the language of the given text.
    """
    try:
        # Use only the first 1000 characters for faster detection
        sample = text[:1000]
        language = detect(sample)
        return language
    except:
        # Default to German if detection fails
        return "de"

# Function 3: Extract Article Structure
def extract_article_structure(markdown_text):
    """
    Extract article structure from markdown text.
    """
    # Initialize article structure
    article = {
        'original_filename': '',
        'sections': {},
    }

    # Extract original filename from the header line
    header_match = re.search(r'^# (.+?)$', markdown_text, re.MULTILINE)
    if header_match:
        article['original_filename'] = header_match.group(1).strip()

    # Extract sections using an improved regex pattern
    # This looks for ## Section headers and captures all content until the next section
    lines = markdown_text.split('\n')
    current_section = None
    section_content = []

    for line in lines:
        if line.startswith('## '):
            # Save previous section if exists
            if current_section is not None:
                article['sections'][current_section] = '\n'.join(section_content).strip()

            # Start new section
            current_section = line[3:].strip()  # Remove '## ' prefix
            section_content = []
        elif current_section is not None:
            section_content.append(line)

    # Save the last section
    if current_section is not None and section_content:
        article['sections'][current_section] = '\n'.join(section_content).strip()

    return article

# Function 4: Extract Title
def extract_title(article_dict, filename):
    """
    Extract title from the article structure or filename.
    """
    # Try to create title from original filename
    if article_dict['original_filename']:
        # Clean up the filename to create a title
        clean_filename = article_dict['original_filename'].replace('.html', '')
        # Replace hyphens with spaces and capitalize words
        words = clean_filename.split('-')
        title = ' '.join(word.capitalize() for word in words)
        return title

    # If no original filename, use the markdown filename
    clean_filename = filename.replace('.md', '')
    words = clean_filename.split('-')
    title = ' '.join(word.capitalize() for word in words)
    return title

# Function 5: Extract Date
def extract_date(text):
    """
    Extract date from text.
    """
    # Various date patterns
    date_patterns = [
        r'(\d{1,2})\.(\d{1,2})\.(\d{4})',  # DD.MM.YYYY (German)
        r'(\d{1,2})[/\.](\d{1,2})[/\.](\d{4})',  # DD/MM/YYYY or MM/DD/YYYY
        r'(\d{4})-(\d{1,2})-(\d{1,2})',  # YYYY-MM-DD
        r'(\d{1,2})(?:st|nd|rd|th)? (?:of )?([A-Za-z]+)[,]? (\d{4})'  # 13th July 2021
    ]

    months = {
        'january': '01', 'february': '02', 'march': '03', 'april': '04',
        'may': '05', 'june': '06', 'july': '07', 'august': '08',
        'september': '09', 'october': '10', 'november': '11', 'december': '12',
        'jan': '01', 'feb': '02', 'mar': '03', 'apr': '04', 'jun': '06',
        'jul': '07', 'aug': '08', 'sep': '09', 'oct': '10', 'nov': '11', 'dec': '12'
    }

    for pattern in date_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            if len(match.groups()) == 3:
                if pattern == r'(\d{4})-(\d{1,2})-(\d{1,2})':  # YYYY-MM-DD
                    year, month, day = match.groups()
                elif pattern == r'(\d{1,2})(?:st|nd|rd|th)? (?:of )?([A-Za-z]+)[,]? (\d{4})':  # 13th July 2021
                    day, month_name, year = match.groups()
                    month = months.get(month_name.lower(), '01')
                else:  # DD.MM.YYYY or similar
                    day, month, year = match.groups()

                # Format as YYYY-MM-DD
                return f"{year}-{month.zfill(2)}-{day.zfill(2)}"

    # Look for year mentions (e.g., "October 2015")
    year_pattern = r'([A-Za-z]+)\s+(\d{4})'
    year_match = re.search(year_pattern, text, re.IGNORECASE)
    if year_match:
        month_name, year = year_match.groups()
        month = months.get(month_name.lower(), '01')
        return f"{year}-{month}-01"

    # If there's a 4-digit year mentioned anywhere
    year_only = re.search(r'\b(20\d{2})\b', text)
    if year_only:
        return f"{year_only.group(1)}-01-01"

    # No date found, return empty string
    return ""

# Function 6: Extract Source
def extract_source(article_dict):
    """
    Extract the source of the article.
    """
    # Check for "Reference" section
    if "Reference" in article_dict['sections']:
        return article_dict['sections']["Reference"]

    # Default source for ETH articles
    return "ETH Zürich, interne Mitteilung / Hochschulkommunikation"

# Function 7: Extract Main Content
def extract_main_content(article_dict):
    """
    Extract the main content of the article.
    """
    # Skip these sections
    skip_sections = ["Reference"]

    # Combine all relevant sections
    content_parts = []

    for section_name, content in article_dict['sections'].items():
        if section_name not in skip_sections:
            content_parts.append(content)

    # Join with double newlines to separate sections
    return " ".join(content_parts)

# Function 8: Extract Named Entities
def extract_named_entities(text, language):
    """
    Extract named entities from text.
    """
    entities = []

    # Common ETH-related entities
    eth_entities = ["ETH Zürich", "ETH", "Universität Zürich", "UZH"]
    for entity in eth_entities:
        if entity in text and entity not in entities:
            entities.append(entity)

    # Look for other capitalized terms that could be entities
    entity_pattern = r'\b[A-Z][a-zA-Z-]+(?: [A-Z][a-zA-Z-]+)*'
    matches = re.findall(entity_pattern, text)

    # Filter and add unique entities
    for match in matches:
        # Skip short or common words
        if len(match) > 3 and match not in ["Der", "Die", "Das", "Ein", "Eine", "The", "This", "That"] and match not in entities:
            entities.append(match)

    # Look for specialized terms
    if "ETH-Karte" in text:
        entities.append("ETH-Karte")
    if "RFID" in text:
        entities.append("RFID-Chip")
    if "ASVZ" in text:
        entities.append("ASVZ")

    # Return up to 10 entities
    return entities[:10]

# Function 9: Extract Topics
def extract_topics(text, language):
    """
    Extract topics from text.
    """
    topics = []

    # Check for various topics based on keywords
    if re.search(r'karte|card|rfid', text.lower()):
        topics.append("Infrastruktur")
        topics.append("Digitalisierung")

    if re.search(r'student|studier|mitarbeit', text.lower()):
        topics.append("Hochschulpolitik")

    if re.search(r'preis|kosten|gebühr', text.lower()):
        topics.append("Preiserhöhungen")
        topics.append("Finanzielle Lage")

    if re.search(r'mensa|essen|verpfleg|gastro', text.lower()):
        topics.append("Hochschulgastronomie")
        topics.append("Verpflegung")

    if re.search(r'forsch|research|wissenschaft', text.lower()):
        topics.append("Forschung")
        topics.append("Wissenschaft")

    if re.search(r'tech|data|gps|signal', text.lower()):
        topics.append("Technologie")
        topics.append("Innovation")

    if re.search(r'storm|wetter|climate|klima', text.lower()):
        topics.append("Umwelt")
        topics.append("Wetter")

    # Add default topic if none found
    if not topics:
        topics.append("ETH Zürich")
        topics.append("Hochschulpolitik")

    # Remove duplicates and return up to 6 topics
    return list(dict.fromkeys(topics))[:6]

# Function 10: Extract Keywords
def extract_keywords(text, language):
    """
    Extract keywords from text.
    """
    keywords = []

    # Look for specific keywords based on content
    if "ETH" in text:
        keywords.append("ETH Zürich")

    if re.search(r'karte|card', text.lower()):
        keywords.append("ETH-Karte")
        keywords.append("Identifikation")

    if "RFID" in text:
        keywords.append("RFID-Chip")

    if re.search(r'design|erscheinungsbild', text.lower()):
        keywords.append("Design")
        keywords.append("Corporate Design")

    if re.search(r'student|studier', text.lower()):
        keywords.append("Studierende")

    if re.search(r'mitarbeit|personal', text.lower()):
        keywords.append("Mitarbeitende")

    if re.search(r'gps|geodesy|signal', text.lower()):
        keywords.append("GPS")
        keywords.append("Geodäsie")

    if re.search(r'storm|sturm|wetter', text.lower()):
        keywords.append("Unwetter")
        keywords.append("Wetterphänomene")

    if re.search(r'preis|kosten', text.lower()):
        keywords.append("Preisanpassung")
        keywords.append("Kostensteigerung")

    if re.search(r'nachhaltig|sustainable', text.lower()):
        keywords.append("Nachhaltigkeit")

    # Add more default keywords if needed
    if len(keywords) < 7:
        additional = ["Universität", "Hochschule", "Innovation", "Bildung", "Forschung", "Wissenschaft"]
        keywords.extend(additional[:(7-len(keywords))])

    # Return up to 7 keywords
    return keywords[:7]

# Function 11: Generate Summary
def generate_summary(text, max_length=200):
    """
    Generate a summary of the text.
    """
    if not text:
        return ""

    # Take the first few sentences, up to max_length
    sentences = re.split(r'(?<=[.!?])\s+', text)

    summary = ""
    for sentence in sentences:
        if len(summary) + len(sentence) <= max_length:
            summary += sentence + " "
        else:
            break

    summary = summary.strip()

    # Add ellipsis if truncated
    if len(summary) < len(text) and summary:
        summary += "..."

    return summary

# Function 12: Generate Rich Metadata
def generate_rich_metadata(text, language):
    """
    Generate rich metadata for the article.
    """
    # Count words
    words = re.findall(r'\b\w+\b', text)
    word_count = len(words)

    # Determine readability
    readability = "leicht verständlich" if language == 'de' else "easy to read"

    # Check for sentiment (simple approach)
    sentiment = "neutral"

    # Check for compound words and umlauts
    contains_compound_words = bool(re.search(r'\b\w{15,}\b', text))
    contains_umlauts = bool(re.search(r'[äöüÄÖÜß]', text))

    # Determine audience type
    audience_type = []

    if re.search(r'student|studier', text.lower()):
        audience_type.append("Studierende")

    if re.search(r'mitarbeit|personal|staff', text.lower()):
        audience_type.append("Mitarbeitende")

    if re.search(r'leitung|direktion|management', text.lower()):
        audience_type.append("Hochschulleitung")

    # Default audience if none detected
    if not audience_type:
        audience_type = ["Studierende", "Mitarbeitende"]

    # Context tags
    context_tags = []

    if re.search(r'finan|kosten|budget', text.lower()):
        context_tags.append("Finanzielle Lage")

    if re.search(r'mensa|essen|verpfleg', text.lower()):
        context_tags.append("Verpflegung")

    if re.search(r'corona|covid|pandemie', text.lower()):
        context_tags.append("Corona-Folgen")

    # Add default ETH intern tag
    context_tags.append("ETH intern")

    # Create rich metadata
    rich_metadata = {
        "document_length_words": word_count,
        "readability_score": readability,
        "sentiment": sentiment,
        "contains_compound_words": contains_compound_words,
        "contains_umlauts": contains_umlauts,
        "embedding_vector": "[...]",
        "audience_type": audience_type,
        "context_tags": context_tags
    }

    return rich_metadata

# Main processing function
def process_article(markdown_text, filename):
    """
    Process a single article through all cleaning steps.
    """
    # Step 1: Clean the text
    cleaned_text = basic_text_cleaning(markdown_text)

    # Step 2: Extract article structure
    article_structure = extract_article_structure(cleaned_text)

    # Step 3: Extract main content (this was the issue before)
    main_content = extract_main_content(article_structure)

    # Step 4: Detect language
    language = detect_language(main_content or cleaned_text)

    # Step 5: Extract title
    title = extract_title(article_structure, filename)

    # Step 6: Extract date
    date = extract_date(main_content)

    # Step 7: Extract source
    source = extract_source(article_structure)

    # Step 8: Extract named entities
    named_entities = extract_named_entities(main_content, language)

    # Step 9: Extract topics
    topics = extract_topics(main_content, language)

    # Step 10: Extract keywords
    keywords = extract_keywords(main_content, language)

    # Step 11: Generate summary
    summary = generate_summary(main_content)

    # Step 12: Generate rich metadata
    rich_metadata = generate_rich_metadata(main_content, language)

    # Step 13: Create final structured document
    processed_article = {
        "language": language,
        "title": title,
        "date": date,
        "source": source,
        "main_content": main_content,
        "named_entities": named_entities,
        "topics": topics,
        "keywords": keywords,
        "summary": summary,
        "rich_metadata": rich_metadata
    }

    return processed_article

# Main execution function
def process_all_articles(parsed_articles):
    """
    Process all articles and return a dataset.
    """
    processed_data = {}

    # Process each article
    print(f"Processing {len(parsed_articles)} articles...")
    for filename, content in tqdm(parsed_articles.items()):
        try:
            processed = process_article(content, filename)
            processed_data[filename] = processed
        except Exception as e:
            print(f"Error processing {filename}: {e}")

    # Save full data to JSON
    with open('processed_articles.json', 'w', encoding='utf-8') as f:
        json.dump(processed_data, f, ensure_ascii=False, indent=2)

    print(f"Processing complete. Saved {len(processed_data)} articles to JSON.")

    return processed_data

# Execute the processing
folder_path = 'news-qa-ethz1/notebooks/parsed_markdown'

# Ensure the folder exists
if not os.path.exists(folder_path):
    raise FileNotFoundError(f"Folder not found: {folder_path}")

# List all files in the folder
files = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
print(f"Found {len(files)} files.")

# Read all files into a dictionary
parsed_articles = {}

for file in files:
    full_path = os.path.join(folder_path, file)
    with open(full_path, 'r', encoding='utf-8') as f:
        parsed_articles[file] = f.read()

# Process all articles
processed_data = process_all_articles(parsed_articles)

# Print some statistics
languages = [article["language"] for article in processed_data.values()]
print(f"\nLanguage distribution:")
for lang in set(languages):
    count = languages.count(lang)
    print(f"  - {lang}: {count} articles ({count/len(languages)*100:.1f}%)")

# Print a sample of the processed data
if processed_data:
    sample_key = list(processed_data.keys())[0]
    print(f"\nSample processed article ('{sample_key}'):")
    sample_article = processed_data[sample_key]

    # Print a prettier version with truncated main_content
    pretty_sample = sample_article.copy()
    if pretty_sample["main_content"] and len(pretty_sample["main_content"]) > 100:
        pretty_sample["main_content"] = pretty_sample["main_content"][:100] + "..."

    print(json.dumps(pretty_sample, indent=2, ensure_ascii=False))

Found 4 files.
Processing 4 articles...


100%|██████████| 4/4 [00:00<00:00, 105.68it/s]

Processing complete. Saved 4 articles to JSON.

Language distribution:
  - de: 2 articles (50.0%)
  - en: 2 articles (50.0%)

Sample processed article ('die-eth-karte-erhaelt-ein-neues-design.md'):
{
  "language": "de",
  "title": "Die Eth Karte Erhaelt Ein Neues Design",
  "date": "2008-01-01",
  "source": "ETH Zürich, interne Mitteilung / Hochschulkommunikation",
  "main_content": "In diesem Sommer erhalten Angehörige der ETH Zürich eine neue ETH-Karte. Die Karte erscheint dabei i...",
  "named_entities": [
    "ETH Zürich",
    "ETH",
    "Universität Zürich",
    "UZH",
    "Sommer",
    "Angeh",
    "ETH-Karte",
    "Die Karte",
    "Design",
    "Erscheinungsbild"
  ],
  "topics": [
    "Infrastruktur",
    "Digitalisierung",
    "Hochschulpolitik",
    "Hochschulgastronomie",
    "Verpflegung"
  ],
  "keywords": [
    "ETH Zürich",
    "ETH-Karte",
    "Identifikation",
    "RFID-Chip",
    "Design",
    "Corporate Design",
    "Studierende"
  ],
  "summary": "In diesem Sommer e

In [ ]:
# Print a sample of the processed data
if processed_data:
    sample_key = list(processed_data.keys())[1]
    print(f"\nSample processed article ('{sample_key}'):")
    sample_article = processed_data[sample_key]

    # Print a prettier version with truncated main_content
    pretty_sample = sample_article.copy()
    if pretty_sample["main_content"] and len(pretty_sample["main_content"]) > 100:
        pretty_sample["main_content"] = pretty_sample["main_content"][:100] + "..."

    print(json.dumps(pretty_sample, indent=2, ensure_ascii=False))


Sample processed article ('detecting-storms-thanks-to-gps.md'):
{
  "language": "en",
  "title": "Detecting Storms Thanks To Gps",
  "date": "2021-07-13",
  "source": "Space geodesy is a field of geodesy that addresses the measurement and mapping of large areas, particularly of the earth, using space technology. The main goal of space geodesy is to gain precise information about the shape, size and movement of the earth.\n\nGPS is a decisive component of space geodesy. GPS satellites can be used to determine user positions on the earth with a high level of precision. This is used in many applications such as navigation, surveying and geographic information systems.\n\nAichinger-Rosenberger M, Aregger M, Kopp J, Soja B: Detecting Signatures of Convective Storm Events in GNSS-SNR: Two Case Studies from Summer 2021 in Switzerland. Geophysical Research Letters 2023, 50. doi: 10.1029/2023GL104916",
  "main_content": "An exceptionally severe storm swept over Zurich on 13 July 2021 shortly b

In [ ]:
# Print a sample of the processed data
if processed_data:
    sample_key = list(processed_data.keys())[2]
    print(f"\nSample processed article ('{sample_key}'):")
    sample_article = processed_data[sample_key]

    # Print a prettier version with truncated main_content
    pretty_sample = sample_article.copy()
    if pretty_sample["main_content"] and len(pretty_sample["main_content"]) > 100:
        pretty_sample["main_content"] = pretty_sample["main_content"][:100] + "..."

    print(json.dumps(pretty_sample, indent=2, ensure_ascii=False))


Sample processed article ('in-memory-of-konrad-steffen.md'):
{
  "language": "en",
  "title": "In Memory Of Konrad Steffen",
  "date": "2012-01-01",
  "source": "ETH Zürich, interne Mitteilung / Hochschulkommunikation",
  "main_content": "He was not only spellbound by Greenland’s wildness and beauty, but also driven by concern for its fu...",
  "named_entities": [
    "ETH",
    "Greenland",
    "Konrad Steffen",
    "Photograph",
    "ETH Zurich",
    "Giulia Marthaler",
    "Tall",
    "Anyone",
    "At ETH Zurich",
    "Koni"
  ],
  "topics": [
    "Forschung",
    "Wissenschaft",
    "Umwelt",
    "Wetter"
  ],
  "keywords": [
    "ETH Zürich",
    "Universität",
    "Hochschule",
    "Innovation",
    "Bildung",
    "Forschung",
    "Wissenschaft"
  ],
  "summary": "",
  "rich_metadata": {
    "document_length_words": 917,
    "readability_score": "easy to read",
    "sentiment": "neutral",
    "contains_compound_words": false,
    "contains_umlauts": true,
    "embedding_vector"

In [ ]:
# Print a sample of the processed data
if processed_data:
    sample_key = list(processed_data.keys())[3]
    print(f"\nSample processed article ('{sample_key}'):")
    sample_article = processed_data[sample_key]

    # Print a prettier version with truncated main_content
    pretty_sample = sample_article.copy()
    if pretty_sample["main_content"] and len(pretty_sample["main_content"]) > 100:
        pretty_sample["main_content"] = pretty_sample["main_content"][:100] + "..."

    print(json.dumps(pretty_sample, indent=2, ensure_ascii=False))


Sample processed article ('erc-advanced-grants.md'):
{
  "language": "de",
  "title": "Erc Advanced Grants",
  "date": "2020-01-01",
  "source": "ETH Zürich, interne Mitteilung / Hochschulkommunikation",
  "main_content": "Die ERC Advanced Grants gehören zu den begehrtesten Auszeichnungen im europäischen Forschungsraum. M...",
  "named_entities": [
    "ETH Zürich",
    "ETH",
    "Die ERC Advanced Grants",
    "Auszeichnungen",
    "Forschungsraum",
    "Europ",
    "Forschungsrat",
    "Projekte",
    "Spitzenforschenden",
    "Renommee"
  ],
  "topics": [
    "Hochschulgastronomie",
    "Verpflegung",
    "Forschung",
    "Wissenschaft",
    "Technologie",
    "Innovation"
  ],
  "keywords": [
    "ETH Zürich",
    "Universität",
    "Hochschule",
    "Innovation",
    "Bildung",
    "Forschung",
    "Wissenschaft"
  ],
  "summary": "Die ERC Advanced Grants gehören zu den begehrtesten Auszeichnungen im europäischen Forschungsraum....",
  "rich_metadata": {
    "document_length_word

## version 5:
- modify the script to implement this approach by translating the topics, keywords, and context_tags to English

- generalized script

In [ ]:
import os
import re
import unicodedata
import json
from langdetect import detect
from tqdm import tqdm
import datetime
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Function 1: Basic Text Cleaning
def basic_text_cleaning(text):
    """
    Perform basic text cleaning operations.
    """
    # Normalize Unicode characters
    text = unicodedata.normalize('NFKC', text)

    # Strip leading/trailing whitespace
    text = text.strip()

    return text

# Function 2: Language Detection
def detect_language(text):
    """
    Detect the language of the given text.
    """
    if not text:
        return "unknown"

    try:
        # Use only the first 1000 characters for faster detection
        sample = text[:1000]
        language = detect(sample)
        return language
    except:
        # Default to unknown if detection fails
        return "unknown"

# Function 3: Extract Article Structure
def extract_article_structure(markdown_text):
    """
    Extract article structure from markdown text.
    """
    # Initialize article structure
    article = {
        'original_filename': '',
        'sections': {},
    }

    # Extract original filename from the header line
    header_match = re.search(r'^# (.+?)$', markdown_text, re.MULTILINE)
    if header_match:
        article['original_filename'] = header_match.group(1).strip()

    # Extract sections using line-by-line parsing
    lines = markdown_text.split('\n')
    current_section = None
    section_content = []

    for line in lines:
        if line.startswith('## '):
            # Save previous section if exists
            if current_section is not None:
                article['sections'][current_section] = '\n'.join(section_content).strip()

            # Start new section
            current_section = line[3:].strip()  # Remove '## ' prefix
            section_content = []
        elif current_section is not None:
            section_content.append(line)

    # Save the last section
    if current_section is not None and section_content:
        article['sections'][current_section] = '\n'.join(section_content).strip()

    return article

# Function 4: Extract Title
def extract_title(article_dict, filename):
    """
    Extract title from the article structure or filename.
    Keep in original language.
    """
    # First try to extract from content sections
    title_sections = ["Title", "Headline", "In brief", "Main article", "Titel", "Überschrift", "Kurz gefasst"]

    for section_name in title_sections:
        if section_name in article_dict['sections']:
            content = article_dict['sections'][section_name]
            lines = content.split('\n')
            if lines:
                # Take first non-empty line as title
                for line in lines:
                    if line.strip():
                        return line.strip()

    # If no title found in sections, try original filename
    if article_dict['original_filename']:
        # Clean up the filename to create a title
        clean_filename = article_dict['original_filename'].replace('.html', '')
        # Replace hyphens with spaces and capitalize words
        words = clean_filename.split('-')
        title = ' '.join(word.capitalize() for word in words)
        return title

    # If no original filename, use the markdown filename
    clean_filename = filename.replace('.md', '')
    words = clean_filename.split('-')
    title = ' '.join(word.capitalize() for word in words)
    return title

# Function 5: Extract Date
def extract_date(text):
    """
    Extract date from text and standardize to YYYY-MM-DD format.
    """
    # Various date patterns
    date_patterns = [
        r'(\d{1,2})\.(\d{1,2})\.(\d{4})',  # DD.MM.YYYY (German)
        r'(\d{1,2})[/\.](\d{1,2})[/\.](\d{4})',  # DD/MM/YYYY or MM/DD/YYYY
        r'(\d{4})-(\d{1,2})-(\d{1,2})',  # YYYY-MM-DD
        r'(\d{1,2})(?:st|nd|rd|th)? (?:of )?([A-Za-z]+)[,]? (\d{4})'  # 13th July 2021
    ]

    months = {
        'january': '01', 'february': '02', 'march': '03', 'april': '04',
        'may': '05', 'june': '06', 'july': '07', 'august': '08',
        'september': '09', 'october': '10', 'november': '11', 'december': '12',
        'jan': '01', 'feb': '02', 'mar': '03', 'apr': '04', 'jun': '06',
        'jul': '07', 'aug': '08', 'sep': '09', 'oct': '10', 'nov': '11', 'dec': '12',
        # German months
        'januar': '01', 'februar': '02', 'märz': '03', 'april': '04',
        'mai': '05', 'juni': '06', 'juli': '07', 'august': '08',
        'september': '09', 'oktober': '10', 'november': '11', 'dezember': '12'
    }

    for pattern in date_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            if len(match.groups()) == 3:
                if pattern == r'(\d{4})-(\d{1,2})-(\d{1,2})':  # YYYY-MM-DD
                    year, month, day = match.groups()
                elif pattern == r'(\d{1,2})(?:st|nd|rd|th)? (?:of )?([A-Za-z]+)[,]? (\d{4})':  # 13th July 2021
                    day, month_name, year = match.groups()
                    month = months.get(month_name.lower(), '01')
                else:  # DD.MM.YYYY or similar
                    day, month, year = match.groups()

                # Format as YYYY-MM-DD
                return f"{year}-{month.zfill(2)}-{day.zfill(2)}"

    # Look for month year formats (e.g., "Oktober 2015", "July 2021")
    month_year_pattern = r'([A-Za-z]+)\s+(\d{4})'
    month_year_match = re.search(month_year_pattern, text, re.IGNORECASE)
    if month_year_match:
        month_name, year = month_year_match.groups()
        month = months.get(month_name.lower(), '01')
        return f"{year}-{month}-01"

    # If there's a 4-digit year mentioned anywhere
    year_only = re.search(r'\b(20\d{2})\b', text)
    if year_only:
        return f"{year_only.group(1)}-01-01"

    # No date found, return empty string
    return ""

# Function 6: Extract Source
def extract_source(article_dict, language):
    """
    Extract the source of the article.
    Keep in original language.
    """
    # Check for "Reference" section
    reference_sections = ["Reference", "Referenz", "Quelle", "Source"]
    for section in reference_sections:
        if section in article_dict['sections']:
            return article_dict['sections'][section]

    # Default source based on language
    if language == "de":
        return "ETH Zürich, interne Mitteilung / Hochschulkommunikation"
    else:
        return "ETH Zurich, internal communication / University Communications"

# Function 7: Extract Main Content
def extract_main_content(article_dict):
    """
    Extract the main content of the article.
    Keep in original language.
    """
    # Skip these sections
    skip_sections = ["Reference", "Referenz", "Quelle", "Source"]

    # Combine all relevant sections
    content_parts = []

    for section_name, content in article_dict['sections'].items():
        if section_name not in skip_sections and content.strip():
            content_parts.append(content)

    # Join with spaces to create flowing text
    return " ".join(content_parts)

# Function 8: Extract Named Entities
def extract_named_entities(text, language):
    """
    Extract named entities from text.
    Keep in original language.
    """
    if not text:
        return []

    entities = []

    # Common ETH-related entities
    eth_entities = ["ETH Zürich", "ETH", "Universität Zürich", "UZH"] if language == "de" else ["ETH Zurich", "ETH", "University of Zurich", "UZH"]
    for entity in eth_entities:
        if entity in text and entity not in entities:
            entities.append(entity)

    # Look for other capitalized terms that could be entities
    entity_pattern = r'\b[A-Z][a-zA-Z-]+(?: [A-Z][a-zA-Z-]+){0,2}'
    matches = re.findall(entity_pattern, text)

    # Filter and add unique entities
    common_words = ["Der", "Die", "Das", "Ein", "Eine", "The", "This", "That", "And", "But", "For", "They", "She", "He"]
    for match in matches:
        if (len(match) > 3 and
            match not in common_words and
            match not in entities and
            not re.match(r'^[A-Z][a-z]+ [A-Z][a-z]+$', match)):  # Skip simple Name Surname patterns
            entities.append(match)

    # Look for specialized terms in respective languages
    if language == "de":
        special_terms = ["ETH-Karte", "RFID-Chip", "ASVZ", "Mensa", "Hönggerberg"]
    else:
        special_terms = ["ETH Card", "RFID Chip", "ASVZ", "Cafeteria", "Hönggerberg"]

    for term in special_terms:
        if term in text and term not in entities:
            entities.append(term)

    # Return up to 10 entities
    return entities[:10]

# Function 9: Extract Topics (Standardized to English)
def extract_topics(text, language):
    """
    Extract topics from text and standardize to English.
    """
    topics = []

    # Define topic detection rules with English output
    topic_rules = [
        # Infrastructure and Cards
        (r'karte|card|rfid|chip|ausweis|identification', "Infrastructure"),
        (r'student|studier|studium|studies|course', "Education"),
        (r'mitarbeit|staff|personal|employee', "Staff"),
        (r'preis|kosten|gebühr|price|cost|fee', "Finance"),
        (r'mensa|essen|verpfleg|gastro|food|dining|cafeteria', "Catering"),
        (r'forsch|research|wissenschaft|science', "Research"),
        (r'tech|data|gps|signal|digital', "Technology"),
        (r'storm|wetter|climate|klima|unwetter', "Environment"),
        (r'kommunikation|communication|mitteilung|announcement', "Communication"),
        (r'design|erscheinungsbild|corporate', "Corporate Design"),
        (r'nachhaltig|sustainable|umwelt|environment', "Sustainability"),
        (r'international|global|worldwide|weltweit', "International")
    ]

    # Check for topics based on rules
    for pattern, topic in topic_rules:
        if re.search(pattern, text.lower()):
            topics.append(topic)

    # Add default topic if none found
    if not topics:
        topics.append("University News")

    # Remove duplicates and return up to 6 topics
    return list(dict.fromkeys(topics))[:6]

# Function 10: Extract Keywords (Standardized to English)
def extract_keywords(text, language):
    """
    Extract keywords from text and standardize to English.
    """
    # Translation dictionary for common terms
    de_to_en = {
        "eth zürich": "ETH Zurich",
        "eth-karte": "ETH Card",
        "karte": "Card",
        "ausweis": "Identification",
        "studierende": "Students",
        "mitarbeitende": "Staff",
        "design": "Design",
        "erscheinungsbild": "Corporate Design",
        "forschung": "Research",
        "wissenschaft": "Science",
        "preiserhöhung": "Price Increase",
        "kosten": "Costs",
        "mensa": "Cafeteria",
        "verpflegung": "Catering",
        "unwetter": "Severe Weather",
        "sturm": "Storm",
        "nachhaltigkeit": "Sustainability"
    }

    keywords = []

    # Check for specific content and add appropriate keywords in English
    if language == "de":
        # For German content, translate common keywords
        if "ETH" in text:
            keywords.append("ETH Zurich")

        if re.search(r'karte|card', text.lower()):
            keywords.append("ETH Card")
            keywords.append("Identification")

        if "RFID" in text:
            keywords.append("RFID Chip")

        if re.search(r'design|erscheinungsbild', text.lower()):
            keywords.append("Design")
            keywords.append("Corporate Identity")

        if re.search(r'student|studier', text.lower()):
            keywords.append("Students")

        if re.search(r'mitarbeit|personal', text.lower()):
            keywords.append("Staff")

        if re.search(r'preis|kosten', text.lower()):
            keywords.append("Price Adjustment")
            keywords.append("Cost Increase")

        if re.search(r'nachhaltig', text.lower()):
            keywords.append("Sustainability")
    else:
        # For English content, use English keywords directly
        if "ETH" in text:
            keywords.append("ETH Zurich")

        if re.search(r'card|identification', text.lower()):
            keywords.append("ETH Card")
            keywords.append("Identification")

        if "RFID" in text:
            keywords.append("RFID Chip")

        if re.search(r'design|corporate', text.lower()):
            keywords.append("Design")
            keywords.append("Corporate Identity")

        if re.search(r'student', text.lower()):
            keywords.append("Students")

        if re.search(r'staff|employee', text.lower()):
            keywords.append("Staff")

        if re.search(r'price|cost', text.lower()):
            keywords.append("Price Adjustment")
            keywords.append("Cost Increase")

        if re.search(r'sustainable', text.lower()):
            keywords.append("Sustainability")

    # Check for GPS/Storm content in either language
    if re.search(r'gps|geodesy|geodäsie', text.lower()):
        keywords.append("GPS")
        keywords.append("Geodesy")

    if re.search(r'storm|sturm|weather|wetter', text.lower()):
        keywords.append("Weather")
        keywords.append("Meteorology")

    # Add more default keywords if needed
    if len(keywords) < 7:
        additional = ["University", "Higher Education", "Innovation", "Education", "Research", "Science"]
        keywords.extend(additional[:(7-len(keywords))])

    # Return up to 7 keywords
    return list(dict.fromkeys(keywords))[:7]

# Function 11: Generate Summary
def generate_summary(text, max_length=200):
    """
    Generate a summary of the text.
    Keep in original language.
    """
    if not text:
        return ""

    # Take the first few sentences, up to max_length
    sentences = re.split(r'(?<=[.!?])\s+', text)

    summary = ""
    for sentence in sentences:
        if len(summary) + len(sentence) <= max_length:
            summary += sentence + " "
        else:
            break

    summary = summary.strip()

    # Add ellipsis if truncated
    if len(summary) < len(text) and summary:
        summary += "..."

    return summary

# Function 12: Generate Rich Metadata (Standardized to English)
def generate_rich_metadata(text, language):
    """
    Generate rich metadata for the article with standardized English fields.
    """
    if not text:
        return {
            "document_length_words": 0,
            "readability_score": "easy to read",
            "sentiment": "neutral",
            "contains_compound_words": False,
            "contains_umlauts": False,
            "embedding_vector": "[...]",
            "audience_type": ["Students", "Staff"],
            "context_tags": ["ETH Internal"]
        }

    # Count words
    words = re.findall(r'\b\w+\b', text)
    word_count = len(words)

    # Determine readability (standardized to English)
    if word_count > 500:
        readability = "moderately complex"
    elif word_count > 200:
        readability = "easy to read"
    else:
        readability = "very easy to read"

    # Check for sentiment (simple approach, standardized to English)
    positive_patterns = r'gut|besser|positiv|erfolgreich|vorteil|good|better|positive|successful|advantage'
    negative_patterns = r'schlecht|problem|negativ|schwierig|nachteil|bad|problem|negative|difficult|disadvantage'

    positive_matches = len(re.findall(positive_patterns, text.lower()))
    negative_matches = len(re.findall(negative_patterns, text.lower()))

    if positive_matches > negative_matches * 2:
        sentiment = "positive"
    elif negative_matches > positive_matches * 2:
        sentiment = "negative"
    elif positive_matches > negative_matches:
        sentiment = "slightly positive"
    elif negative_matches > positive_matches:
        sentiment = "slightly negative"
    else:
        sentiment = "neutral"

    # Check for compound words and umlauts
    contains_compound_words = bool(re.search(r'\b\w{15,}\b', text))
    contains_umlauts = bool(re.search(r'[äöüÄÖÜß]', text))

    # Determine audience type (standardized to English)
    audience_type = []

    if re.search(r'student|studier|studierende', text.lower()):
        audience_type.append("Students")

    if re.search(r'mitarbeit|personal|staff|employee', text.lower()):
        audience_type.append("Staff")

    if re.search(r'leitung|direktion|management|leadership', text.lower()):
        audience_type.append("Leadership")

    if re.search(r'forsch|research|wissenschaft|science', text.lower()):
        audience_type.append("Researchers")

    # Default audience if none detected
    if not audience_type:
        audience_type = ["Students", "Staff"]

    # Context tags (standardized to English)
    context_tags = []

    if re.search(r'finan|kosten|budget|cost|price', text.lower()):
        context_tags.append("Financial")

    if re.search(r'mensa|essen|verpfleg|food|dining|cafeteria', text.lower()):
        context_tags.append("Catering")

    if re.search(r'corona|covid|pandemie|pandemic', text.lower()):
        context_tags.append("COVID-19")

    if re.search(r'nachhaltig|sustainable|environment|umwelt', text.lower()):
        context_tags.append("Sustainability")

    # Add default ETH internal tag
    context_tags.append("ETH Internal")

    # Create rich metadata with all fields in English
    rich_metadata = {
        "document_length_words": word_count,
        "readability_score": readability,
        "sentiment": sentiment,
        "contains_compound_words": contains_compound_words,
        "contains_umlauts": contains_umlauts,
        "embedding_vector": "[...]",
        "audience_type": audience_type,
        "context_tags": context_tags
    }

    return rich_metadata

# Main processing function
def process_article(markdown_text, filename):
    """
    Process a single article through all cleaning steps.
    """
    try:
        # Step a: Basic cleaning
        cleaned_text = basic_text_cleaning(markdown_text)

        # Step b: Extract article structure
        article_structure = extract_article_structure(cleaned_text)

        # Step c: Extract main content (keep in original language)
        main_content = extract_main_content(article_structure)

        # Step d: Detect language
        language = detect_language(main_content or cleaned_text)

        # Step e: Extract title (keep in original language)
        title = extract_title(article_structure, filename)

        # Step f: Extract date (standardized format)
        date = extract_date(main_content or cleaned_text)

        # Step g: Extract source (keep in original language)
        source = extract_source(article_structure, language)

        # Step h: Extract named entities (keep in original language)
        named_entities = extract_named_entities(main_content, language)

        # Step i: Extract topics (standardized to English)
        topics = extract_topics(main_content, language)

        # Step j: Extract keywords (standardized to English)
        keywords = extract_keywords(main_content, language)

        # Step k: Generate summary (keep in original language)
        summary = generate_summary(main_content)

        # Step l: Generate rich metadata (standardized to English)
        rich_metadata = generate_rich_metadata(main_content, language)

        # Create final structured document
        processed_article = {
            "language": language,
            "title": title,
            "date": date,
            "source": source,
            "main_content": main_content,
            "named_entities": named_entities,
            "topics": topics,
            "keywords": keywords,
            "summary": summary,
            "rich_metadata": rich_metadata
        }

        return processed_article

    except Exception as e:
        logger.error(f"Error processing {filename}: {str(e)}")
        # Return a minimal valid structure in case of failure
        return {
            "language": "unknown",
            "title": filename.replace('.md', '').replace('-', ' ').title(),
            "date": "",
            "source": "ETH Zurich",
            "main_content": "",
            "named_entities": [],
            "topics": ["University News"],
            "keywords": ["ETH Zurich"],
            "summary": "",
            "rich_metadata": {
                "document_length_words": 0,
                "readability_score": "unknown",
                "sentiment": "neutral",
                "contains_compound_words": False,
                "contains_umlauts": False,
                "embedding_vector": "[...]",
                "audience_type": ["Students", "Staff"],
                "context_tags": ["ETH Internal"]
            }
        }

# Batch processing function for handling large datasets efficiently
def process_articles_in_batches(parsed_articles, batch_size=100):
    """
    Process articles in batches to handle large datasets efficiently.
    """
    all_processed = {}
    articles = list(parsed_articles.items())
    total_batches = (len(articles) + batch_size - 1) // batch_size

    logger.info(f"Processing {len(articles)} articles in {total_batches} batches of {batch_size}...")

    for batch_num in range(total_batches):
        start_idx = batch_num * batch_size
        end_idx = min((batch_num + 1) * batch_size, len(articles))

        logger.info(f"Processing batch {batch_num+1}/{total_batches} (articles {start_idx+1}-{end_idx})...")

        batch = articles[start_idx:end_idx]
        batch_results = {}

        for filename, content in tqdm(batch):
            try:
                processed = process_article(content, filename)
                batch_results[filename] = processed
            except Exception as e:
                logger.error(f"Error processing {filename}: {str(e)}")

        all_processed.update(batch_results)

        # Save intermediate results every batch
        with open(f'processed_articles_batch_{batch_num+1}.json', 'w', encoding='utf-8') as f:
            json.dump(batch_results, f, ensure_ascii=False, indent=2)

        logger.info(f"Saved batch {batch_num+1} with {len(batch_results)} articles")

    # Combine all batches into final result
    with open('processed_articles.json', 'w', encoding='utf-8') as f:
        json.dump(all_processed, f, ensure_ascii=False, indent=2)

    logger.info(f"Processing complete. Saved {len(all_processed)} articles to JSON.")

    return all_processed

# Main execution function
def main():
    """
    Main execution function.
    """
    # Define folder path
    folder_path = 'news-qa-ethz1/notebooks/parsed_markdown'

    # Ensure the folder exists
    if not os.path.exists(folder_path):
        raise FileNotFoundError(f"Folder not found: {folder_path}")

    # List all files in the folder
    files = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
    logger.info(f"Found {len(files)} files.")

    # Read all files into a dictionary
    parsed_articles = {}

    for file in tqdm(files, desc="Reading files"):
        full_path = os.path.join(folder_path, file)
        try:
            with open(full_path, 'r', encoding='utf-8') as f:
                parsed_articles[file] = f.read()
        except UnicodeDecodeError:
            # Try with different encodings if utf-8 fails
            try:
                with open(full_path, 'r', encoding='latin-1') as f:
                    parsed_articles[file] = f.read()
            except Exception as e:
                logger.error(f"Could not read {file}: {str(e)}")

    # Process all articles in batches for efficiency
    processed_data = process_articles_in_batches(parsed_articles, batch_size=100)

    # Print some statistics
    languages = [article["language"] for article in processed_data.values()]
    logger.info(f"\nLanguage distribution:")
    for lang in set(languages):
        count = languages.count(lang)
        logger.info(f"  - {lang}: {count} articles ({count/len(languages)*100:.1f}%)")

    return processed_data

# Run the main function
if __name__ == "__main__":
    processed_data = main()

100%|██████████| 4/4 [00:00<00:00, 104.64it/s]


In [ ]:
# Print a sample of the processed data
if processed_data:
    sample_key = list(processed_data.keys())[0]
    print(f"\nSample processed article ('{sample_key}'):")
    sample_article = processed_data[sample_key]

    # Print a prettier version with truncated main_content
    pretty_sample = sample_article.copy()
    if pretty_sample["main_content"] and len(pretty_sample["main_content"]) > 100:
        pretty_sample["main_content"] = pretty_sample["main_content"][:100] + "..."

    print(json.dumps(pretty_sample, indent=2, ensure_ascii=False))


Sample processed article ('die-eth-karte-erhaelt-ein-neues-design.md'):
{
  "language": "de",
  "title": "In diesem Sommer erhalten Angehörige der ETH Zürich eine neue ETH-Karte. Die Karte erscheint dabei im neuen Design: Das 2008 eingeführte, grüne Erscheinungsbild wird ersetzt durch ein Blau aus dem Corporate Design der ETH Zürich.",
  "date": "2008-01-01",
  "source": "ETH Zürich, interne Mitteilung / Hochschulkommunikation",
  "main_content": "In diesem Sommer erhalten Angehörige der ETH Zürich eine neue ETH-Karte. Die Karte erscheint dabei i...",
  "named_entities": [
    "ETH Zürich",
    "ETH",
    "Universität Zürich",
    "UZH",
    "Sommer",
    "Angeh",
    "ETH-Karte",
    "Design",
    "Erscheinungsbild",
    "Blau"
  ],
  "topics": [
    "Infrastructure",
    "Education",
    "Staff",
    "Catering",
    "Corporate Design"
  ],
  "keywords": [
    "ETH Zurich",
    "ETH Card",
    "Identification",
    "RFID Chip",
    "Design",
    "Corporate Identity",
    "Students"
 

In [ ]:
# Print a sample of the processed data
if processed_data:
    sample_key = list(processed_data.keys())[1]
    print(f"\nSample processed article ('{sample_key}'):")
    sample_article = processed_data[sample_key]

    # Print a prettier version with truncated main_content
    pretty_sample = sample_article.copy()
    if pretty_sample["main_content"] and len(pretty_sample["main_content"]) > 100:
        pretty_sample["main_content"] = pretty_sample["main_content"][:100] + "..."

    print(json.dumps(pretty_sample, indent=2, ensure_ascii=False))


Sample processed article ('detecting-storms-thanks-to-gps.md'):
{
  "language": "en",
  "title": "An exceptionally severe storm swept over Zurich on 13 July 2021 shortly before 2 a.m.: howling squalls, constant lightning and torrential rain woke people up with a start. Benedikt Soja, Professor of Space Geodesy, also got little sleep that night. “It was one of the most severe storms I’ve ever witnessed. I woke up in the middle of the night and could see the storm raging through the window,” he remembers.",
  "date": "2021-07-13",
  "source": "Space geodesy is a field of geodesy that addresses the measurement and mapping of large areas, particularly of the earth, using space technology. The main goal of space geodesy is to gain precise information about the shape, size and movement of the earth.\n\nGPS is a decisive component of space geodesy. GPS satellites can be used to determine user positions on the earth with a high level of precision. This is used in many applications such as nav

## version 6 - final version
improve the source extraction, named entity recognition, topics/keywords extraction, and context tags

In [ ]:
!pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 12.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=1cbf8b23145b33e0cdf387ce297c1a77de31b7519fde554d2da1ba5443c63c7d
  Stored in directory: /root/.cache/pip/wheels/0a/f2/b2/e5ca405801e05eb7c8ed5b3b4bcf1fcabcd6272c167640072e
Successfully built langdetect


In [ ]:
import os
import re
import unicodedata
import json
from langdetect import detect
from tqdm import tqdm
import logging
import datetime
import string

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Function 1: Basic Text Cleaning
def basic_text_cleaning(text):
    """
    Perform basic text cleaning operations.
    """
    # Normalize Unicode characters
    text = unicodedata.normalize('NFKC', text)

    # Strip leading/trailing whitespace
    text = text.strip()

    return text

# Function 2: Language Detection
def detect_language(text):
    """
    Detect the language of the given text.
    """
    if not text:
        return "unknown"

    try:
        # Use only the first 1000 characters for faster detection
        sample = text[:1000]
        language = detect(sample)
        return language
    except:
        # Default to unknown if detection fails
        return "unknown"

# Function 3: Extract Article Structure
def extract_article_structure(markdown_text):
    """
    Extract article structure from markdown text.
    """
    # Initialize article structure
    article = {
        'original_filename': '',
        'sections': {},
    }

    # Extract original filename from the header line
    header_match = re.search(r'^# (.+?)$', markdown_text, re.MULTILINE)
    if header_match:
        article['original_filename'] = header_match.group(1).strip()

    # Extract sections using line-by-line parsing
    lines = markdown_text.split('\n')
    current_section = None
    section_content = []

    for line in lines:
        if line.startswith('## '):
            # Save previous section if exists
            if current_section is not None:
                article['sections'][current_section] = '\n'.join(section_content).strip()

            # Start new section
            current_section = line[3:].strip()  # Remove '## ' prefix
            section_content = []
        elif current_section is not None:
            section_content.append(line)

    # Save the last section
    if current_section is not None and section_content:
        article['sections'][current_section] = '\n'.join(section_content).strip()

    return article

# Function 4: Extract Title
def extract_title(article_dict, filename):
    """
    Extract title from the article structure or filename.
    Keep in original language.
    """
    # First try to extract from content sections
    title_sections = ["In brief", "Main article", "Title", "Headline", "Titel", "Überschrift", "Kurz gefasst"]

    for section_name in title_sections:
        if section_name in article_dict['sections']:
            content = article_dict['sections'][section_name]
            lines = content.split('\n')
            if lines:
                # Take first non-empty line as title
                for line in lines:
                    if line.strip():
                        return line.strip()

    # If no title found in sections, try original filename
    if article_dict['original_filename']:
        # Clean up the filename to create a title
        clean_filename = article_dict['original_filename'].replace('.html', '')
        # Replace hyphens with spaces and capitalize words
        words = clean_filename.split('-')
        title = ' '.join(word.capitalize() for word in words)
        return title

    # If no original filename, use the markdown filename
    clean_filename = filename.replace('.md', '')
    words = clean_filename.split('-')
    title = ' '.join(word.capitalize() for word in words)
    return title

# Function 5: Extract Date
def extract_date(text):
    """
    Extract date from text and standardize to YYYY-MM-DD format.
    """
    # Various date patterns
    date_patterns = [
        r'(\d{1,2})\.(\d{1,2})\.(\d{4})',  # DD.MM.YYYY (German)
        r'(\d{1,2})[/\.](\d{1,2})[/\.](\d{4})',  # DD/MM/YYYY or MM/DD/YYYY
        r'(\d{4})-(\d{1,2})-(\d{1,2})',  # YYYY-MM-DD
        r'(\d{1,2})(?:st|nd|rd|th)? (?:of )?([A-Za-z]+)[,]? (\d{4})'  # 13th July 2021
    ]

    months = {
        'january': '01', 'february': '02', 'march': '03', 'april': '04',
        'may': '05', 'june': '06', 'july': '07', 'august': '08',
        'september': '09', 'october': '10', 'november': '11', 'december': '12',
        'jan': '01', 'feb': '02', 'mar': '03', 'apr': '04', 'jun': '06',
        'jul': '07', 'aug': '08', 'sep': '09', 'oct': '10', 'nov': '11', 'dec': '12',
        # German months
        'januar': '01', 'februar': '02', 'märz': '03', 'april': '04',
        'mai': '05', 'juni': '06', 'juli': '07', 'august': '08',
        'september': '09', 'oktober': '10', 'november': '11', 'dezember': '12'
    }

    for pattern in date_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            if len(match.groups()) == 3:
                if pattern == r'(\d{4})-(\d{1,2})-(\d{1,2})':  # YYYY-MM-DD
                    year, month, day = match.groups()
                elif pattern == r'(\d{1,2})(?:st|nd|rd|th)? (?:of )?([A-Za-z]+)[,]? (\d{4})':  # 13th July 2021
                    day, month_name, year = match.groups()
                    month = months.get(month_name.lower(), '01')
                else:  # DD.MM.YYYY or similar
                    day, month, year = match.groups()

                # Format as YYYY-MM-DD
                return f"{year}-{month.zfill(2)}-{day.zfill(2)}"

    # Look for month year formats (e.g., "Oktober 2015", "July 2021")
    month_year_pattern = r'([A-Za-z]+)\s+(\d{4})'
    month_year_match = re.search(month_year_pattern, text, re.IGNORECASE)
    if month_year_match:
        month_name, year = month_year_match.groups()
        month = months.get(month_name.lower(), '01')
        return f"{year}-{month}-01"

    # If there's a 4-digit year mentioned anywhere
    year_only = re.search(r'\b(20\d{2})\b', text)
    if year_only:
        return f"{year_only.group(1)}-01-01"

    # No date found, return empty string
    return ""

# IMPROVED: Function 6: Extract Source
def extract_source(article_dict, text, language):
    """
    Extract the source of the article.
    Keep in original language.
    """
    # Check in Reference section
    reference_sections = ["Reference", "Referenz", "Quelle", "Source"]
    for section in reference_sections:
        if section in article_dict['sections']:
            source_text = article_dict['sections'][section]
            if source_text.strip():
                return source_text.strip()

    # Look for specific source patterns in the text
    source_patterns = [
        r'(?:Source|Quelle):\s*([^\.]+)',
        r'(?:By|Von|Author|Autor):\s*([^\.]+)',
        r'(?:Copyright|©)\s*([^\.]+)',
        r'(?:Published by|Veröffentlicht von):\s*([^\.]+)'
    ]

    for pattern in source_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1).strip()

    # Check for ETH departments or units
    eth_units = [
        r'(ETH[\s-]Zürich[^\.;,]*(?:Kommunikation|Communication|Department|Departement|Abteilung)[^\.;,]*)',
        r'(ETH[\s-]Zurich[^\.;,]*(?:Communication|Department|Unit|Division)[^\.;,]*)',
        r'((?:Hochschulkommunikation|University Communication)[^\.;,]*ETH[^\.;,]*)'
    ]

    for pattern in eth_units:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1).strip()

    # Extract potential domains from the filename
    if article_dict['original_filename']:
        domain_match = re.search(r'(?:www\.)?([a-z0-9-]+)\.(?:com|org|edu|ch|de)', article_dict['original_filename'], re.IGNORECASE)
        if domain_match:
            domain = domain_match.group(1).lower()
            if 'ethz' in domain:
                return "ETH Zürich" if language == "de" else "ETH Zurich"
            elif 'uzh' in domain:
                return "Universität Zürich" if language == "de" else "University of Zurich"
            else:
                # Format domain as a source
                domain = domain.replace('-', ' ').title()
                return domain

    # Default source based on language and content clues
    if "ETH" in text:
        return "ETH Zürich, Hochschulkommunikation" if language == "de" else "ETH Zurich, University Communications"

    # Generic fallback
    return "Unbekannte Quelle" if language == "de" else "Unknown source"

# IMPROVED: Function 7: Extract Main Content
def extract_main_content(article_dict):
    """
    Extract the main content of the article.
    Keep in original language.
    """
    # Skip these sections
    skip_sections = ["Reference", "Referenz", "Quelle", "Source"]

    # If there's only one section and it's not in skip_sections, return it
    if len(article_dict['sections']) == 1:
        section_name = list(article_dict['sections'].keys())[0]
        if section_name not in skip_sections:
            return article_dict['sections'][section_name]

    # Combine all relevant sections
    content_parts = []

    for section_name, content in article_dict['sections'].items():
        if section_name not in skip_sections and content.strip():
            # Add section name as header if multiple sections
            if len(article_dict['sections']) > 1:
                content_parts.append(f"{section_name}: {content}")
            else:
                content_parts.append(content)

    # Join with double newlines to separate sections
    return "\n\n".join(content_parts)

# IMPROVED: Function 8: Extract Named Entities
def extract_named_entities(text, language):
    """
    Extract named entities from text.
    Keep in original language.
    """
    if not text:
        return []

    # Normalize text for better extraction
    normalized_text = text.replace('\n', ' ').replace('  ', ' ')

    entities = []

    # First look for multi-word capitalized phrases (potential organizations and people)
    # This regex looks for 2-4 capitalized words in sequence
    org_pattern = r'\b([A-Z][a-zäöüÄÖÜß]+(?:[ \-][A-Z][a-zäöüÄÖÜß]+){1,3})\b'
    org_matches = re.findall(org_pattern, normalized_text)

    # Filter out common sentence starters and short phrases
    exclude_patterns = [
        r'^(?:The|A|An|Der|Die|Das|Ein|Eine|This|That|These|Those|Sie|Er|Es|Wir|Ich|Du)$',
        r'^[A-Za-z]{1,2}$'  # Exclude 1-2 letter entities
    ]

    for match in org_matches:
        if all(not re.match(pat, match) for pat in exclude_patterns) and len(match) > 3:
            entities.append(match)

    # Look for single-word organization names (typically capitalized nouns not at beginning of sentence)
    # This requires more filtering to avoid common words
    single_word_pattern = r'(?<![.!?]\s)\b([A-Z][a-zA-ZäöüÄÖÜß]{3,})\b'
    single_matches = re.findall(single_word_pattern, normalized_text)

    # Common words to exclude as entities
    common_words = set([
        "The", "This", "That", "These", "Those", "They", "Their", "And", "But", "However",
        "Der", "Die", "Das", "Diese", "Dieser", "Dieses", "Jene", "Und", "Aber", "Jedoch"
    ])

    for match in single_matches:
        if match not in common_words and not any(match in e for e in entities):
            entities.append(match)

    # ETH-specific named entities to look for
    eth_specific = []
    if language == "de":
        eth_specific = [
            "ETH Zürich", "ETH-Zürich", "ETH", "Universität Zürich", "UZH",
            "Hönggerberg", "ETH-Karte", "RFID-Chip", "ASVZ", "Polyterrasse"
        ]
    else:
        eth_specific = [
            "ETH Zurich", "ETH", "University of Zurich", "UZH",
            "Hönggerberg", "ETH Card", "RFID Chip", "ASVZ", "Polyterrasse"
        ]

    for entity in eth_specific:
        if entity in normalized_text and not any(entity in e for e in entities):
            entities.append(entity)

    # Look for people names mentioned with titles
    titles = ["Prof", "Professor", "Dr", "Professorin", "Doktor"]
    for title in titles:
        name_pattern = f"{title}\\.?\\s+([A-Z][a-zäöüÄÖÜß]+(?:\\s+[A-Z][a-zäöüÄÖÜß]+){{1,2}})"
        prof_matches = re.findall(name_pattern, normalized_text)
        entities.extend(prof_matches)

    # Deduplicate entities (case-insensitive)
    unique_entities = []
    seen = set()
    for entity in entities:
        if entity.lower() not in seen:
            seen.add(entity.lower())
            unique_entities.append(entity)

    # Return up to 10 entities
    return unique_entities[:10]

# IMPROVED: Function 9: Extract Topics (Standardized to English)
def extract_topics(text, language):
    """
    Extract topics from text and standardize to English.
    """
    if not text:
        return ["University News"]

    # Pre-process text for better matching
    text_lower = text.lower()

    # Define comprehensive topic categories with weighted keyword sets
    # Format: (topic_name, [(keyword, weight), ...])
    topics_keywords = [
        ("Infrastructure", [
            ("karte", 3), ("card", 3), ("ausweis", 3), ("identification", 3),
            ("gebäude", 2), ("building", 2), ("raum", 1), ("room", 1),
            ("campus", 2), ("hönggerberg", 3), ("zentrum", 1), ("center", 1)
        ]),

        ("Technology & Innovation", [
            ("technologie", 3), ("technology", 3), ("innovation", 3),
            ("digital", 2), ("software", 2), ("computer", 2), ("app", 2),
            ("rfid", 3), ("chip", 2), ("elektronisch", 1), ("electronic", 1),
            ("entwicklung", 1), ("development", 1), ("programmier", 2), ("coding", 2)
        ]),

        ("Research", [
            ("forschung", 3), ("research", 3), ("wissenschaft", 3), ("science", 3),
            ("studie", 2), ("study", 2), ("experiment", 2), ("projekt", 1), ("project", 1),
            ("entdeckung", 2), ("discovery", 2), ("publikation", 2), ("publication", 2)
        ]),

        ("Education", [
            ("ausbildung", 3), ("education", 3), ("studium", 3), ("studies", 3),
            ("student", 3), ("studierend", 3), ("lehre", 3), ("teaching", 3),
            ("vorlesung", 2), ("lecture", 2), ("kurs", 2), ("course", 2),
            ("prüfung", 2), ("exam", 2), ("seminar", 2), ("unterricht", 2)
        ]),

        ("Finance", [
            ("finanzen", 3), ("finance", 3), ("kosten", 3), ("costs", 3),
            ("preis", 3), ("price", 3), ("erhöhung", 2), ("increase", 2),
            ("budget", 3), ("geld", 2), ("money", 2), ("zahlung", 1), ("payment", 1)
        ]),

        ("University Administration", [
            ("verwaltung", 3), ("administration", 3), ("leitung", 2), ("management", 2),
            ("präsident", 2), ("president", 2), ("rektor", 2), ("rector", 2),
            ("direktor", 2), ("director", 2), ("strategie", 2), ("strategy", 2)
        ]),

        ("Campus Life", [
            ("campus", 3), ("student", 2), ("mensa", 3), ("cafeteria", 3),
            ("essen", 2), ("food", 2), ("veranstaltung", 2), ("event", 2),
            ("freizeit", 2), ("leisure", 2), ("sport", 2), ("asvz", 3)
        ]),

        ("International", [
            ("international", 3), ("global", 3), ("weltweit", 2), ("worldwide", 2),
            ("ausland", 2), ("foreign", 2), ("kooperation", 2), ("cooperation", 2),
            ("austausch", 2), ("exchange", 2), ("partner", 2)
        ]),

        ("Sustainability", [
            ("nachhaltig", 3), ("sustainable", 3), ("umwelt", 3), ("environment", 3),
            ("klima", 3), ("climate", 3), ("grün", 2), ("green", 2),
            ("energie", 2), ("energy", 2), ("ressource", 2), ("resource", 2)
        ]),

        ("Weather & Environment", [
            ("wetter", 3), ("weather", 3), ("sturm", 3), ("storm", 3),
            ("umwelt", 2), ("environment", 2), ("klima", 2), ("climate", 2),
            ("regen", 2), ("rain", 2), ("wind", 2), ("temperatur", 2), ("temperature", 2)
        ]),

        ("Communication", [
            ("kommunikation", 3), ("communication", 3), ("mitteilung", 3), ("announcement", 3),
            ("information", 2), ("bericht", 2), ("report", 2), ("news", 3),
            ("presse", 2), ("press", 2), ("media", 2), ("medien", 2)
        ]),

        ("Catering & Food", [
            ("mensa", 3), ("cafeteria", 3), ("essen", 3), ("food", 3),
            ("verpflegung", 3), ("catering", 3), ("restaurant", 2),
            ("speise", 2), ("meal", 2), ("menü", 2), ("menu", 2)
        ]),

        ("Staff", [
            ("mitarbeiter", 3), ("staff", 3), ("personal", 3), ("employee", 3),
            ("anstellung", 2), ("employment", 2), ("arbeit", 2), ("work", 2),
            ("position", 2), ("stelle", 2), ("job", 2)
        ]),

        ("COVID-19", [
            ("covid", 3), ("corona", 3), ("pandemic", 3), ("pandemie", 3),
            ("lockdown", 3), ("virus", 2), ("impfung", 2), ("vaccination", 2),
            ("maske", 2), ("mask", 2), ("abstand", 2), ("distance", 2)
        ])
    ]

    # Calculate scores for each topic
    topic_scores = {}

    for topic_name, keywords in topics_keywords:
        score = 0
        for keyword, weight in keywords:
            # Count occurrences of the keyword
            count = text_lower.count(keyword)
            if count > 0:
                score += count * weight

        if score > 0:
            topic_scores[topic_name] = score

    # If no topics found, add a default
    if not topic_scores:
        return ["University News"]

    # Sort topics by score and return top 6
    sorted_topics = sorted(topic_scores.items(), key=lambda x: x[1], reverse=True)
    return [topic for topic, score in sorted_topics[:6]]

# IMPROVED: Function 10: Extract Keywords (Standardized to English)
def extract_keywords(text, language):
    """
    Extract keywords from text and standardize to English.
    """
    if not text:
        return ["ETH Zurich"]

    # Normalize and lowercase text
    text_normalized = text.lower()

    # Define keyword mapping from German to English
    de_to_en = {
        "eth zürich": "ETH Zurich",
        "eth-karte": "ETH Card",
        "karte": "Card",
        "ausweis": "Identification Card",
        "studierende": "Students",
        "studenten": "Students",
        "student": "Student",
        "mitarbeitende": "Staff",
        "mitarbeiter": "Staff",
        "personal": "Personnel",
        "design": "Design",
        "erscheinungsbild": "Visual Identity",
        "forschung": "Research",
        "wissenschaft": "Science",
        "preiserhöhung": "Price Increase",
        "kosten": "Costs",
        "mensa": "Cafeteria",
        "verpflegung": "Catering",
        "unwetter": "Storm",
        "sturm": "Storm",
        "wetter": "Weather",
        "nachhaltigkeit": "Sustainability",
        "umwelt": "Environment",
        "klima": "Climate",
        "gebäude": "Building",
        "campus": "Campus",
        "hönggerberg": "Hönggerberg",
        "zentrum": "Campus Center",
        "technologie": "Technology",
        "innovation": "Innovation",
        "digital": "Digital",
        "lehre": "Teaching",
        "bildung": "Education",
        "kommunikation": "Communication"
    }

    # Initialize keywords list
    keywords = []

    # First check for specific multi-word terms
    if language == "de":
        # German multi-word terms with their English translations
        multi_word_de = {
            "eth zürich": "ETH Zurich",
            "eth-karte": "ETH Card",
            "elektronische karte": "Electronic ID",
            "corporate design": "Corporate Design",
            "neue design": "New Design",
            "universität zürich": "University of Zurich",
            "hönggerberg campus": "Hönggerberg Campus"
        }

        for de_term, en_term in multi_word_de.items():
            if de_term in text_normalized and en_term not in keywords:
                keywords.append(en_term)
    else:
        # English multi-word terms
        multi_word_en = [
            "ETH Zurich", "ETH Card", "Electronic ID", "Corporate Design",
            "New Design", "University of Zurich", "Hönggerberg Campus"
        ]

        for term in multi_word_en:
            if term.lower() in text_normalized and term not in keywords:
                keywords.append(term)

    # Check for single words and translate if German
    # First tokenize by removing punctuation and splitting
    translator = str.maketrans('', '', string.punctuation)
    words = text_normalized.translate(translator).split()

    # Get word frequency
    word_freq = {}
    for word in words:
        if len(word) > 3:  # Skip very short words
            word_freq[word] = word_freq.get(word, 0) + 1

    # Get the most frequent words
    sorted_words = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)

    # Add top words, translating from German if needed
    for word, _ in sorted_words[:15]:  # Get top 15 words to ensure we have enough after filtering
        if language == "de" and word in de_to_en:
            en_word = de_to_en[word]
            if en_word not in keywords:
                keywords.append(en_word)
        elif language == "en" and len(word) > 3:
            # Capitalize the first letter for English words
            word_cap = word.capitalize()
            if word_cap not in keywords:
                keywords.append(word_cap)

    # Add specific ETH-related keywords if they're relevant
    eth_keywords = ["ETH Zurich", "University", "Research", "Science", "Campus", "Education"]
    for keyword in eth_keywords:
        if keyword not in keywords:
            keywords.append(keyword)

    # Return up to 7 keywords (deduped)
    unique_keywords = []
    seen = set()
    for kw in keywords:
        if kw.lower() not in seen:
            seen.add(kw.lower())
            unique_keywords.append(kw)

    return unique_keywords[:7]

# IMPROVED: Function 11: Generate Rich Metadata (Standardized to English)
def generate_rich_metadata(text, language):
    """
    Generate rich metadata for the article with standardized English fields.
    """
    if not text:
        return {
            "document_length_words": 0,
            "readability_score": "unknown",
            "sentiment": "neutral",
            "contains_compound_words": False,
            "contains_umlauts": False,
            "embedding_vector": "[...]",
            "audience_type": ["Students", "Staff"],
            "context_tags": ["ETH Internal"]
        }

    # Count words
    words = re.findall(r'\b\w+\b', text)
    word_count = len(words)

    # Determine readability (standardized to English)
    # More accurate approach based on sentence length and word length
    sentences = re.split(r'[.!?]+', text)
    sentence_count = len([s for s in sentences if s.strip()])

    avg_words_per_sentence = word_count / max(sentence_count, 1)
    long_words = len([w for w in words if len(w) > 6])
    long_word_percentage = (long_words / max(word_count, 1)) * 100

    if avg_words_per_sentence > 25 or long_word_percentage > 30:
        readability = "complex"
    elif avg_words_per_sentence > 15 or long_word_percentage > 20:
        readability = "moderately complex"
    else:
        readability = "easy to read"

    # Improved sentiment analysis with more keywords
    positive_patterns = [
        r'\b(?:gut|besser|positiv|erfolgreich|vorteil|nutzen|förder|erfreut|freude|verbessert)',
        r'\b(?:good|better|positive|successful|advantage|benefit|promote|pleased|improve|happy)'
    ]

    negative_patterns = [
        r'\b(?:schlecht|problem|negativ|schwierig|nachteil|kritisch|belastung|sorge|verschlechter)',
        r'\b(?:bad|problem|negative|difficult|disadvantage|critical|burden|worry|worsen)'
    ]

    positive_count = 0
    for pattern in positive_patterns:
        positive_count += len(re.findall(pattern, text.lower()))

    negative_count = 0
    for pattern in negative_patterns:
        negative_count += len(re.findall(pattern, text.lower()))

    # Calculate sentiment ratio
    total = positive_count + negative_count
    if total == 0:
        sentiment = "neutral"
    elif positive_count > negative_count * 2:
        sentiment = "positive"
    elif negative_count > positive_count * 2:
        sentiment = "negative"
    elif positive_count > negative_count:
        sentiment = "slightly positive"
    elif negative_count > positive_count:
        sentiment = "slightly negative"
    else:
        sentiment = "neutral"

    # Check for compound words and umlauts
    contains_compound_words = bool(re.search(r'\b\w{15,}\b', text))
    contains_umlauts = bool(re.search(r'[äöüÄÖÜß]', text))

    # IMPROVED: Determine audience type (standardized to English)
    audience_type = []

    # More specific patterns for different audience types
    audience_patterns = [
        ("Students", [r'\bstud(?:ent|ierend|ium|ies)', r'\bschule\b', r'\buniversity\b']),
        ("Staff", [r'\bmitarbeit|\bpersonal|\bstaff|\bemployee|\bangestellt']),
        ("Faculty", [r'\bprofessor|\bdozent|\bfaculty|\blectur|\bdocent']),
        ("Researchers", [r'\bforsch|\bresearch|\bwissenschaft|\bscience|\blabor|\blab\b']),
        ("Administration", [r'\bverwaltung|\badministration|\bleitung|\bmanagement']),
        ("General Public", [r'\böffentlich|\bpublic|\ballgemein|\bgeneral|\bcommunity|\bgesellschaft'])
    ]

    for audience, patterns in audience_patterns:
        for pattern in patterns:
            if re.search(pattern, text.lower()):
                audience_type.append(audience)
                break

    # Default audience if none detected
    if not audience_type:
        audience_type = ["Students", "Staff"]

    # IMPROVED: Context tags (standardized to English)
    # IMPROVED: Context tags (standardized to English)
    context_tags = []

    # More comprehensive context tagging system
    context_patterns = [
        ("Financial", [r'\bfinan|\bkosten|\bbudget|\bcost|\bprice|\bpreis|\bgeld|\bmoney']),
        ("Catering", [r'\bmensa|\bessen|\bverpfleg|\bfood|\bdining|\bcafeteria|\bmahlzeit|\bmeal']),
        ("COVID-19", [r'\bcorona|\bcovid|\bpandemie|\bpandemic|\blockdown|\bvirus']),
        ("Sustainability", [r'\bnachhaltig|\bsustainable|\benvironment|\bumwelt|\bklima|\bclimate']),
        ("Infrastructure", [r'\bgebäude|\bbuilding|\binfrastruktur|\bcampus|\braum|\bspace']),
        ("Technology", [r'\btechnologie|\btechnology|\bdigital|\bsoftware|\brfid|\bapp']),
        ("Research", [r'\bforschung|\bresearch|\bwissenschaft|\bscience|\bstudie|\bstudy']),
        ("Education", [r'\bausbildung|\beducation|\bstudium|\bstudies|\blehre|\bteaching']),
        ("Administrative", [r'\bverwaltung|\badministration|\bmanagement|\bleitung|\bpolicy']),
        ("Communications", [r'\bkommunikation|\bcommunication|\bmitteilung|\bannouncement']),
        ("Events", [r'\bveranstaltung|\bevent|\bkonferenz|\bconference|\bmeeting|\bseminar']),
        ("Weather", [r'\bwetter|\bweather|\bsturm|\bstorm|\bregen|\brain|\btemperatur']),
        ("International", [r'\binternational|\bglobal|\bweltweit|\bworldwide|\bausland']),
        ("Career", [r'\bkarriere|\bcareer|\bjob|\bstelle|\bposition|\bbewerbung|\bapplication'])
    ]

    # Check for context tags
    for tag, patterns in context_patterns:
        for pattern in patterns:
            if re.search(pattern, text.lower()):
                context_tags.append(tag)
                break

    # Check if ETH-related
    if re.search(r'\beth|\bethz|\beidgenössische|\bpoly', text.lower()):
        context_tags.append("ETH Internal")

    # Limit to most relevant tags (max 4)
    if len(context_tags) > 4:
        context_tags = context_tags[:4]
    elif not context_tags:
        context_tags = ["University News"]

    # Create rich metadata with all fields in English
    rich_metadata = {
        "document_length_words": word_count,
        "readability_score": readability,
        "sentiment": sentiment,
        "contains_compound_words": contains_compound_words,
        "contains_umlauts": contains_umlauts,
        "embedding_vector": "[...]",
        "audience_type": audience_type,
        "context_tags": context_tags
    }

    return rich_metadata

# Main processing function
def process_article(markdown_text, filename):
    """
    Process a single article through all cleaning steps.
    """
    try:
        # Step a: Basic cleaning
        cleaned_text = basic_text_cleaning(markdown_text)

        # Step b: Extract article structure
        article_structure = extract_article_structure(cleaned_text)

        # Step c: Extract main content (keep in original language)
        main_content = extract_main_content(article_structure)

        # Step d: Detect language
        language = detect_language(main_content or cleaned_text)

        # Step e: Extract title (keep in original language)
        title = extract_title(article_structure, filename)

        # Step f: Extract date (standardized format)
        date = extract_date(main_content or cleaned_text)

        # Step g: Extract source (keep in original language)
        source = extract_source(article_structure, main_content or cleaned_text, language)

        # Step h: Extract named entities (keep in original language)
        named_entities = extract_named_entities(main_content, language)

        # Step i: Extract topics (standardized to English)
        topics = extract_topics(main_content, language)

        # Step j: Extract keywords (standardized to English)
        keywords = extract_keywords(main_content, language)

        # Step k: Generate summary (keep in original language)
        summary = generate_summary(main_content)

        # Step l: Generate rich metadata (standardized to English)
        rich_metadata = generate_rich_metadata(main_content, language)

        # Create final structured document
        processed_article = {
            "language": language,
            "title": title,
            "date": date,
            "source": source,
            "main_content": main_content,
            "named_entities": named_entities,
            "topics": topics,
            "keywords": keywords,
            "summary": summary,
            "rich_metadata": rich_metadata
        }

        return processed_article

    except Exception as e:
        logger.error(f"Error processing {filename}: {str(e)}")
        # Return a minimal valid structure in case of failure
        return {
            "language": "unknown",
            "title": filename.replace('.md', '').replace('-', ' ').title(),
            "date": "",
            "source": "ETH Zurich",
            "main_content": "",
            "named_entities": [],
            "topics": ["University News"],
            "keywords": ["ETH Zurich"],
            "summary": "",
            "rich_metadata": {
                "document_length_words": 0,
                "readability_score": "unknown",
                "sentiment": "neutral",
                "contains_compound_words": False,
                "contains_umlauts": False,
                "embedding_vector": "[...]",
                "audience_type": ["Students", "Staff"],
                "context_tags": ["ETH Internal"]
            }
        }

# Function to generate a summary
def generate_summary(text, max_length=200):
    """
    Generate a summary of the text.
    Keep in original language.
    """
    if not text:
        return ""

    # Take the first few sentences, up to max_length
    sentences = re.split(r'(?<=[.!?])\s+', text)

    summary = ""
    for sentence in sentences:
        if len(summary) + len(sentence) <= max_length:
            summary += sentence + " "
        else:
            break

    summary = summary.strip()

    # Add ellipsis if truncated
    if len(summary) < len(text) and summary:
        summary += "..."

    return summary

# Batch processing function for handling large datasets efficiently
def process_articles_in_batches(parsed_articles, batch_size=100):
    """
    Process articles in batches to handle large datasets efficiently.
    """
    all_processed = {}
    articles = list(parsed_articles.items())
    total_batches = (len(articles) + batch_size - 1) // batch_size

    logger.info(f"Processing {len(articles)} articles in {total_batches} batches of {batch_size}...")

    for batch_num in range(total_batches):
        start_idx = batch_num * batch_size
        end_idx = min((batch_num + 1) * batch_size, len(articles))

        logger.info(f"Processing batch {batch_num+1}/{total_batches} (articles {start_idx+1}-{end_idx})...")

        batch = articles[start_idx:end_idx]
        batch_results = {}

        for filename, content in tqdm(batch):
            try:
                processed = process_article(content, filename)
                batch_results[filename] = processed
            except Exception as e:
                logger.error(f"Error processing {filename}: {str(e)}")

        all_processed.update(batch_results)

        # Save intermediate results every batch
        with open(f'processed_articles_batch_{batch_num+1}.json', 'w', encoding='utf-8') as f:
            json.dump(batch_results, f, ensure_ascii=False, indent=2)

        logger.info(f"Saved batch {batch_num+1} with {len(batch_results)} articles")

    # Combine all batches into final result
    with open('processed_articles.json', 'w', encoding='utf-8') as f:
        json.dump(all_processed, f, ensure_ascii=False, indent=2)

    logger.info(f"Processing complete. Saved {len(all_processed)} articles to JSON.")

    return all_processed

# Main execution function
def main():
    """
    Main execution function.
    """
    # Define folder path
    folder_path = 'news-qa-ethz1/notebooks/parsed_markdown'

    # Ensure the folder exists
    if not os.path.exists(folder_path):
        raise FileNotFoundError(f"Folder not found: {folder_path}")

    # List all files in the folder
    files = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
    logger.info(f"Found {len(files)} files.")

    # Read all files into a dictionary
    parsed_articles = {}

    for file in tqdm(files, desc="Reading files"):
        full_path = os.path.join(folder_path, file)
        try:
            with open(full_path, 'r', encoding='utf-8') as f:
                parsed_articles[file] = f.read()
        except UnicodeDecodeError:
            # Try with different encodings if utf-8 fails
            try:
                with open(full_path, 'r', encoding='latin-1') as f:
                    parsed_articles[file] = f.read()
            except Exception as e:
                logger.error(f"Could not read {file}: {str(e)}")

    # Process all articles in batches for efficiency
    processed_data = process_articles_in_batches(parsed_articles, batch_size=100)

    # Print some statistics
    languages = [article["language"] for article in processed_data.values()]
    logger.info(f"\nLanguage distribution:")
    for lang in set(languages):
        count = languages.count(lang)
        logger.info(f"  - {lang}: {count} articles ({count/len(languages)*100:.1f}%)")

    return processed_data

# Run the main function
if __name__ == "__main__":
    processed_data = main()

100%|██████████| 4/4 [00:02<00:00,  1.75it/s]


In [ ]:
# Print a sample of the processed data
if processed_data:
    sample_key = list(processed_data.keys())[0]
    print(f"\nSample processed article ('{sample_key}'):")
    sample_article = processed_data[sample_key]

    # Print a prettier version with truncated main_content
    pretty_sample = sample_article.copy()
    if pretty_sample["main_content"] and len(pretty_sample["main_content"]) > 100:
        pretty_sample["main_content"] = pretty_sample["main_content"][:100] + "..."

    print(json.dumps(pretty_sample, indent=2, ensure_ascii=False))


Sample processed article ('die-eth-karte-erhaelt-ein-neues-design.md'):
{
  "language": "de",
  "title": "In diesem Sommer erhalten Angehörige der ETH Zürich eine neue ETH-Karte. Die Karte erscheint dabei im neuen Design: Das 2008 eingeführte, grüne Erscheinungsbild wird ersetzt durch ein Blau aus dem Corporate Design der ETH Zürich.",
  "date": "2008-01-01",
  "source": "ETH Zürich, Hochschulkommunikation",
  "main_content": "In diesem Sommer erhalten Angehörige der ETH Zürich eine neue ETH-Karte. Die Karte erscheint dabei i...",
  "named_entities": [
    "Die Karte",
    "Corporate Design",
    "Der Austausch",
    "Mitte August",
    "Universität Zürich",
    "Sommer",
    "Angehörige",
    "Erscheinungsbild",
    "Blau",
    "Version"
  ],
  "topics": [
    "Infrastructure",
    "Staff",
    "Campus Life",
    "Technology & Innovation",
    "Education",
    "Research"
  ],
  "keywords": [
    "ETH Zurich",
    "ETH Card",
    "Corporate Design",
    "University of Zurich",
    "Ca

### version 6

In [ ]:
#!rm -rf /content/news-qa-ethz1/news-qa-ethz1

In [5]:
%cd /content/news-qa-ethz1


/content/news-qa-ethz1


In [ ]:
# Create the scripts directory
#!mkdir -p scripts

# Now create the processing script file
#!touch scripts/process_eth_news.py

In [6]:
!pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 13.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=6e8e6085fc01333036174c6087f521478b3a69b907ffc85af122a3bdd5a360fe
  Stored in directory: /root/.cache/pip/wheels/0a/f2/b2/e5ca405801e05eb7c8ed5b3b4bcf1fcabcd6272c167640072e
Successfully built langdetect


In [ ]:

#%%writefile scripts/process_eth_news.py

import os
import json
import re
import string
import unicodedata
import logging
from tqdm import tqdm
from langdetect import detect
from pathlib import Path


# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Function 1: Basic Text Cleaning
def basic_text_cleaning(text):
    """
    Perform basic text cleaning operations.
    """
    # Normalize Unicode characters
    text = unicodedata.normalize('NFKC', text)

    # Strip leading/trailing whitespace
    text = text.strip()

    return text

# Function 2: Language Detection
def detect_language(text):
    """
    Detect the language of the given text.
    """
    if not text:
        return "unknown"

    try:
        # Use only the first 1000 characters for faster detection
        sample = text[:1000]
        language = detect(sample)
        return language
    except:
        # Default to unknown if detection fails
        return "unknown"

# Function 3: Extract Article Structure
def extract_article_structure(markdown_text):
    """
    Extract article structure from markdown text.
    """
    # Initialize article structure
    article = {
        'original_filename': '',
        'sections': {},
    }

    # Extract original filename from the header line
    header_match = re.search(r'^# (.+?)$', markdown_text, re.MULTILINE)
    if header_match:
        article['original_filename'] = header_match.group(1).strip()

    # Extract sections using line-by-line parsing
    lines = markdown_text.split('\n')
    current_section = None
    section_content = []

    for line in lines:
        if line.startswith('## '):
            # Save previous section if exists
            if current_section is not None:
                article['sections'][current_section] = '\n'.join(section_content).strip()

            # Start new section
            current_section = line[3:].strip()  # Remove '## ' prefix
            section_content = []
        elif current_section is not None:
            section_content.append(line)

    # Save the last section
    if current_section is not None and section_content:
        article['sections'][current_section] = '\n'.join(section_content).strip()

    return article

# Function 4: Extract Title
def extract_title(article_dict, filename):
    """
    Extract title from the article structure or filename.
    Keep in original language.
    """
    # First try to extract from content sections
    title_sections = ["In brief", "Main article", "Title", "Headline", "Titel", "Überschrift", "Kurz gefasst"]

    for section_name in title_sections:
        if section_name in article_dict['sections']:
            content = article_dict['sections'][section_name]
            lines = content.split('\n')
            if lines:
                # Take first non-empty line as title
                for line in lines:
                    if line.strip():
                        return line.strip()

    # If no title found in sections, try original filename
    if article_dict['original_filename']:
        # Clean up the filename to create a title
        clean_filename = article_dict['original_filename'].replace('.html', '')
        # Replace hyphens with spaces and capitalize words
        words = clean_filename.split('-')
        title = ' '.join(word.capitalize() for word in words)
        return title

    # If no original filename, use the markdown filename
    clean_filename = filename.replace('.md', '')
    words = clean_filename.split('-')
    title = ' '.join(word.capitalize() for word in words)
    return title

# # Function 5: Extract Date
# def extract_date(text, filepath=None):
#     """
#     Extract date from text or fallback to file path (e.g., 2021/01/...).
#     Returns a standardized date in YYYY-MM-DD format.
#     """
#     # Same patterns as before
#     date_patterns = [
#         r'(\d{1,2})\.(\d{1,2})\.(\d{4})',  # DD.MM.YYYY
#         r'(\d{1,2})[/\.](\d{1,2})[/\.](\d{4})',  # DD/MM/YYYY
#         r'(\d{4})-(\d{1,2})-(\d{1,2})',  # YYYY-MM-DD
#         r'(\d{1,2})(?:st|nd|rd|th)? (?:of )?([A-Za-z]+)[,]? (\d{4})'  # e.g., 13th July 2021
#     ]

#     months = {
#         'january': '01', 'february': '02', 'march': '03', 'april': '04',
#         'may': '05', 'june': '06', 'july': '07', 'august': '08',
#         'september': '09', 'october': '10', 'november': '11', 'december': '12',
#         'januar': '01', 'februar': '02', 'märz': '03', 'april': '04',
#         'mai': '05', 'juni': '06', 'juli': '07', 'august': '08',
#         'september': '09', 'oktober': '10', 'november': '11', 'dezember': '12'
#     }

#     for pattern in date_patterns:
#         match = re.search(pattern, text, re.IGNORECASE)
#         if match:
#             if len(match.groups()) == 3:
#                 if pattern == r'(\d{4})-(\d{1,2})-(\d{1,2})':
#                     year, month, day = match.groups()
#                 elif pattern == r'(\d{1,2})(?:st|nd|rd|th)? (?:of )?([A-Za-z]+)[,]? (\d{4})':
#                     day, month_name, year = match.groups()
#                     month = months.get(month_name.lower(), '01')
#                 else:
#                     day, month, year = match.groups()
#                 return f"{year}-{month.zfill(2)}-{day.zfill(2)}"

#     # Fallback: try getting year/month from filepath
#     if filepath:
#         parts = Path(filepath).parts
#         year = None
#         month = None
#         for i, part in enumerate(parts):
#             if not year and re.fullmatch(r'20\d{2}', part):
#                 year = part
#                 # Try to get the next part as month
#                 if i + 1 < len(parts):
#                     maybe_month = parts[i + 1]
#                     if re.fullmatch(r'\d{1,2}', maybe_month):
#                         month = maybe_month.zfill(2)
#                 break  # stop after first valid year found
#         if year and month:
#             return f"{year}-{month}-01"

#     return ""

def extract_date(text, filepath=None):
    """
    Extract date from text or fallback to file path (e.g., 2021/01/...).
    Returns a standardized date in YYYY-MM-DD format.
    """
    # Original date patterns
    date_patterns = [
        r'(\d{1,2})\.(\d{1,2})\.(\d{4})',  # DD.MM.YYYY
        r'(\d{1,2})[/\.](\d{1,2})[/\.](\d{4})',  # DD/MM/YYYY
        r'(\d{4})-(\d{1,2})-(\d{1,2})',  # YYYY-MM-DD
        r'(\d{1,2})(?:st|nd|rd|th)? (?:of )?([A-Za-z]+)[,]? (\d{4})'  # e.g., 13th July 2021
    ]

    # Add academic publication date patterns
    academic_patterns = [
        r'published (?:online )?(?:on )?(?:the )?(\d{1,2})(?:st|nd|rd|th)? ([A-Za-z]+)(?:[,]? (\d{4}))?',  # published online 24th June 2021
        r'published (?:online )?(?:on )?(?:the )?([A-Za-z]+) (\d{1,2})(?:st|nd|rd|th)?(?:[,]? (\d{4}))?',  # published online June 24th 2021
        r'published:? (\d{1,2})\.(\d{1,2})\.(\d{4})',  # published: 24.06.2021
        r'published:? (\d{1,2})[/\-](\d{1,2})[/\-](\d{4})',  # published: 24/06/2021 or 24-06-2021
        r'published in ([A-Za-z]+) (\d{4})',  # published in June 2021
        r'(?:date|pub\. date|publication date)[:\s]+(\d{1,2})\.(\d{1,2})\.(\d{4})',  # publication date: 24.06.2021
        r'(?:date|pub\. date|publication date)[:\s]+(\d{1,2})[/\-](\d{1,2})[/\-](\d{4})'  # publication date: 24/06/2021
    ]

    # Add partial date patterns (month/year only)
    partial_date_patterns = [
        r'(\d{2})\.(\d{2})',  # MM.YY (like 06.24 in your example)
        r'(\d{2})[/\-](\d{4})',  # MM/YYYY or MM-YYYY
        r'(\d{2})[/\-](\d{2})',  # MM/YY or MM-YY
    ]

    months = {
        'january': '01', 'february': '02', 'march': '03', 'april': '04',
        'may': '05', 'june': '06', 'july': '07', 'august': '08',
        'september': '09', 'october': '10', 'november': '11', 'december': '12',
        'januar': '01', 'februar': '02', 'märz': '03', 'april': '04',
        'mai': '05', 'juni': '06', 'juli': '07', 'august': '08',
        'september': '09', 'oktober': '10', 'november': '11', 'dezember': '12',
        'jan': '01', 'feb': '02', 'mar': '03', 'apr': '04',
        'may': '05', 'jun': '06', 'jul': '07', 'aug': '08',
        'sep': '09', 'oct': '10', 'nov': '11', 'dec': '12'
    }

    # Extract year from text explicitly to help with partial dates
    year_pattern = r'\b(20\d{2})\b'
    year_match = re.search(year_pattern, text)
    year_from_text = year_match.group(1) if year_match else None

    # Extract month from folder path if available
    path_month = None
    path_year = None
    if filepath:
        parts = Path(filepath).parts
        for i, part in enumerate(parts):
            if re.fullmatch(r'20\d{2}', part):
                path_year = part
                # Try to get the next part as month
                if i + 1 < len(parts):
                    maybe_month = parts[i + 1]
                    if re.fullmatch(r'\d{1,2}', maybe_month):
                        path_month = maybe_month.zfill(2)
                break  # stop after first valid year found

    # First try regular date patterns
    for pattern in date_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            if len(match.groups()) == 3:
                if pattern == r'(\d{4})-(\d{1,2})-(\d{1,2})':
                    year, month, day = match.groups()
                elif pattern == r'(\d{1,2})(?:st|nd|rd|th)? (?:of )?([A-Za-z]+)[,]? (\d{4})':
                    day, month_name, year = match.groups()
                    month = months.get(month_name.lower(), '01')
                else:
                    day, month, year = match.groups()
                return f"{year}-{month.zfill(2)}-{day.zfill(2)}"

    # Then try academic publication date patterns
    for pattern in academic_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            groups = match.groups()

            if len(groups) == 3:
                # Patterns with day, month, year
                if re.match(r'\d{1,2}', groups[0]) and re.match(r'[A-Za-z]+', groups[1]):
                    # Format: day, month_name, year
                    day, month_name, year = groups
                    month = months.get(month_name.lower(), '01')
                elif re.match(r'[A-Za-z]+', groups[0]) and re.match(r'\d{1,2}', groups[1]):
                    # Format: month_name, day, year
                    month_name, day, year = groups
                    month = months.get(month_name.lower(), '01')
                else:
                    # Format: day, month, year (all numeric)
                    day, month, year = groups

                # Handle missing year
                if not year and year_from_text:
                    year = year_from_text

                if year and day and (re.match(r'\d+', day) and len(year) == 4):
                    return f"{year}-{month.zfill(2)}-{day.zfill(2)}"

            elif len(groups) == 2:
                # Patterns with month, year only
                if re.match(r'[A-Za-z]+', groups[0]) and re.match(r'\d{4}', groups[1]):
                    month_name, year = groups
                    month = months.get(month_name.lower(), '01')
                    return f"{year}-{month.zfill(2)}-01"  # Default to 1st of month

    # Try partial date patterns (MM.YY format like in the example)
    for pattern in partial_date_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            groups = match.groups()

            if len(groups) == 2:
                # Check if both are numeric
                if re.match(r'\d+', groups[0]) and re.match(r'\d+', groups[1]):
                    # First value is likely month if it's between 1-12
                    month_val = int(groups[0])
                    if 1 <= month_val <= 12:
                        month = groups[0].zfill(2)
                        year_part = groups[1]

                        # Handle 2-digit year
                        if len(year_part) == 2:
                            # Assume 20xx for year values
                            year = f"20{year_part}"
                        else:
                            year = year_part

                        # Use year from text if found and the matched year is short
                        if len(year) < 4 and year_from_text:
                            year = year_from_text

                        if len(year) == 4:  # Only proceed if we have a valid year
                            return f"{year}-{month}-01"  # Default to 1st of month

    # If we've found a year but no specific date, use folder path for month information
    if year_from_text and path_month:
        return f"{year_from_text}-{path_month}-01"
    elif year_from_text:
        # If only year is found without month, check folder path for month
        return f"{year_from_text}-01-01"

    # Fallback: use folder path completely
    if path_year and path_month:
        return f"{path_year}-{path_month}-01"
    elif path_year:
        return f"{path_year}-01-01"

    return ""

# Function 6: Extract Source
def extract_source(article_dict, text, language):
    """
    Extract the source of the article.
    Keep in original language.
    """
    # Check in Reference section
    reference_sections = ["Reference", "Referenz", "Quelle", "Source"]
    for section in reference_sections:
        if section in article_dict['sections']:
            source_text = article_dict['sections'][section]
            if source_text.strip():
                return source_text.strip()

    # Look for specific source patterns in the text
    source_patterns = [
        r'(?:Source|Quelle):\s*([^\.]+)',
        r'(?:By|Von|Author|Autor):\s*([^\.]+)',
        r'(?:Copyright|©)\s*([^\.]+)',
        r'(?:Published by|Veröffentlicht von):\s*([^\.]+)'
    ]

    for pattern in source_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1).strip()

    # Check for ETH departments or units
    eth_units = [
        r'(ETH[\s-]Zürich[^\.;,]*(?:Kommunikation|Communication|Department|Departement|Abteilung)[^\.;,]*)',
        r'(ETH[\s-]Zurich[^\.;,]*(?:Communication|Department|Unit|Division)[^\.;,]*)',
        r'((?:Hochschulkommunikation|University Communication)[^\.;,]*ETH[^\.;,]*)'
    ]

    for pattern in eth_units:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1).strip()

    # Extract potential domains from the filename
    if article_dict['original_filename']:
        domain_match = re.search(r'(?:www\.)?([a-z0-9-]+)\.(?:com|org|edu|ch|de)', article_dict['original_filename'], re.IGNORECASE)
        if domain_match:
            domain = domain_match.group(1).lower()
            if 'ethz' in domain:
                return "ETH Zürich" if language == "de" else "ETH Zurich"
            elif 'uzh' in domain:
                return "Universität Zürich" if language == "de" else "University of Zurich"
            else:
                # Format domain as a source
                domain = domain.replace('-', ' ').title()
                return domain

    # Default source based on language and content clues
    if "ETH" in text:
        return "ETH Zürich, Hochschulkommunikation" if language == "de" else "ETH Zurich, University Communications"

    # Generic fallback
    return "Unbekannte Quelle" if language == "de" else "Unknown source"

# Function 7: Extract Main Content
def extract_main_content(article_dict):
    """
    Extract the main content of the article.
    Keep in original language.
    """
    # Skip these sections
    skip_sections = ["Reference", "Referenz", "Quelle", "Source"]

    # If there's only one section and it's not in skip_sections, return it
    if len(article_dict['sections']) == 1:
        section_name = list(article_dict['sections'].keys())[0]
        if section_name not in skip_sections:
            return article_dict['sections'][section_name]

    # Combine all relevant sections
    content_parts = []

    for section_name, content in article_dict['sections'].items():
        if section_name not in skip_sections and content.strip():
            # Add section name as header if multiple sections
            if len(article_dict['sections']) > 1:
                content_parts.append(f"{section_name}: {content}")
            else:
                content_parts.append(content)

    # Join with double newlines to separate sections
    return "\n\n".join(content_parts)

# Function 8: Extract Named Entities
def extract_named_entities(text, language):
    """
    Extract named entities from text.
    Keep in original language.
    """
    if not text:
        return []

    # Normalize text for better extraction
    normalized_text = text.replace('\n', ' ').replace('  ', ' ')

    entities = []

    # First look for multi-word capitalized phrases (potential organizations and people)
    # This regex looks for 2-4 capitalized words in sequence
    org_pattern = r'\b([A-Z][a-zäöüÄÖÜß]+(?:[ \-][A-Z][a-zäöüÄÖÜß]+){1,3})\b'
    org_matches = re.findall(org_pattern, normalized_text)

    # Filter out common sentence starters and short phrases
    exclude_patterns = [
        r'^(?:The|A|An|Der|Die|Das|Ein|Eine|This|That|These|Those|Sie|Er|Es|Wir|Ich|Du)$',
        r'^[A-Za-z]{1,2}$'  # Exclude 1-2 letter entities
    ]

    for match in org_matches:
        if all(not re.match(pat, match) for pat in exclude_patterns) and len(match) > 3:
            entities.append(match)

    # Look for single-word organization names (typically capitalized nouns not at beginning of sentence)
    # This requires more filtering to avoid common words
    single_word_pattern = r'(?<![.!?]\s)\b([A-Z][a-zA-ZäöüÄÖÜß]{3,})\b'
    single_matches = re.findall(single_word_pattern, normalized_text)

    # Common words to exclude as entities
    common_words = set([
        "The", "This", "That", "These", "Those", "They", "Their", "And", "But", "However",
        "Der", "Die", "Das", "Diese", "Dieser", "Dieses", "Jene", "Und", "Aber", "Jedoch"
    ])

    for match in single_matches:
        if match not in common_words and not any(match in e for e in entities):
            entities.append(match)

    # ETH-specific named entities to look for
    eth_specific = []
    if language == "de":
        eth_specific = [
            "ETH Zürich", "ETH-Zürich", "ETH", "Universität Zürich", "UZH",
            "Hönggerberg", "ETH-Karte", "RFID-Chip", "ASVZ", "Polyterrasse"
        ]
    else:
        eth_specific = [
            "ETH Zurich", "ETH", "University of Zurich", "UZH",
            "Hönggerberg", "ETH Card", "RFID Chip", "ASVZ", "Polyterrasse"
        ]

    for entity in eth_specific:
        if entity in normalized_text and not any(entity in e for e in entities):
            entities.append(entity)

    # Look for people names mentioned with titles
    titles = ["Prof", "Professor", "Dr", "Professorin", "Doktor"]
    for title in titles:
        name_pattern = f"{title}\\.?\\s+([A-Z][a-zäöüÄÖÜß]+(?:\\s+[A-Z][a-zäöüÄÖÜß]+){{1,2}})"
        prof_matches = re.findall(name_pattern, normalized_text)
        entities.extend(prof_matches)

    # Deduplicate entities (case-insensitive)
    unique_entities = []
    seen = set()
    for entity in entities:
        if entity.lower() not in seen:
            seen.add(entity.lower())
            unique_entities.append(entity)

    # Return up to 10 entities
    return unique_entities[:10]

# Function 9: Extract Topics (Standardized to English)
def extract_topics(text, language):
    """
    Extract topics from text and standardize to English.
    """
    if not text:
        return ["University News"]

    # Pre-process text for better matching
    text_lower = text.lower()

    # Define comprehensive topic categories with weighted keyword sets
    # Format: (topic_name, [(keyword, weight), ...])
    topics_keywords = [
        ("Infrastructure", [
            ("karte", 3), ("card", 3), ("ausweis", 3), ("identification", 3),
            ("gebäude", 2), ("building", 2), ("raum", 1), ("room", 1),
            ("campus", 2), ("hönggerberg", 3), ("zentrum", 1), ("center", 1)
        ]),

        ("Technology & Innovation", [
            ("technologie", 3), ("technology", 3), ("innovation", 3),
            ("digital", 2), ("software", 2), ("computer", 2), ("app", 2),
            ("rfid", 3), ("chip", 2), ("elektronisch", 1), ("electronic", 1),
            ("entwicklung", 1), ("development", 1), ("programmier", 2), ("coding", 2)
        ]),

        ("Research", [
            ("forschung", 3), ("research", 3), ("wissenschaft", 3), ("science", 3),
            ("studie", 2), ("study", 2), ("experiment", 2), ("projekt", 1), ("project", 1),
            ("entdeckung", 2), ("discovery", 2), ("publikation", 2), ("publication", 2)
        ]),

        ("Education", [
            ("ausbildung", 3), ("education", 3), ("studium", 3), ("studies", 3),
            ("student", 3), ("studierend", 3), ("lehre", 3), ("teaching", 3),
            ("vorlesung", 2), ("lecture", 2), ("kurs", 2), ("course", 2),
            ("prüfung", 2), ("exam", 2), ("seminar", 2), ("unterricht", 2)
        ]),

        ("Finance", [
            ("finanzen", 3), ("finance", 3), ("kosten", 3), ("costs", 3),
            ("preis", 3), ("price", 3), ("erhöhung", 2), ("increase", 2),
            ("budget", 3), ("geld", 2), ("money", 2), ("zahlung", 1), ("payment", 1)
        ]),

        ("University Administration", [
            ("verwaltung", 3), ("administration", 3), ("leitung", 2), ("management", 2),
            ("präsident", 2), ("president", 2), ("rektor", 2), ("rector", 2),
            ("direktor", 2), ("director", 2), ("strategie", 2), ("strategy", 2)
        ]),

        ("Campus Life", [
            ("campus", 3), ("student", 2), ("mensa", 3), ("cafeteria", 3),
            ("essen", 2), ("food", 2), ("veranstaltung", 2), ("event", 2),
            ("freizeit", 2), ("leisure", 2), ("sport", 2), ("asvz", 3)
        ]),

        ("International", [
            ("international", 3), ("global", 3), ("weltweit", 2), ("worldwide", 2),
            ("ausland", 2), ("foreign", 2), ("kooperation", 2), ("cooperation", 2),
            ("austausch", 2), ("exchange", 2), ("partner", 2)
        ]),

        ("Sustainability", [
            ("nachhaltig", 3), ("sustainable", 3), ("umwelt", 3), ("environment", 3),
            ("klima", 3), ("climate", 3), ("grün", 2), ("green", 2),
            ("energie", 2), ("energy", 2), ("ressource", 2), ("resource", 2)
        ]),

        ("Weather & Environment", [
            ("wetter", 3), ("weather", 3), ("sturm", 3), ("storm", 3),
            ("umwelt", 2), ("environment", 2), ("klima", 2), ("climate", 2),
            ("regen", 2), ("rain", 2), ("wind", 2), ("temperatur", 2), ("temperature", 2)
        ]),

        ("Communication", [
            ("kommunikation", 3), ("communication", 3), ("mitteilung", 3), ("announcement", 3),
            ("information", 2), ("bericht", 2), ("report", 2), ("news", 3),
            ("presse", 2), ("press", 2), ("media", 2), ("medien", 2)
        ]),

        ("Catering & Food", [
            ("mensa", 3), ("cafeteria", 3), ("essen", 3), ("food", 3),
            ("verpflegung", 3), ("catering", 3), ("restaurant", 2),
            ("speise", 2), ("meal", 2), ("menü", 2), ("menu", 2)
        ]),

        ("Staff", [
            ("mitarbeiter", 3), ("staff", 3), ("personal", 3), ("employee", 3),
            ("anstellung", 2), ("employment", 2), ("arbeit", 2), ("work", 2),
            ("position", 2), ("stelle", 2), ("job", 2)
        ]),

        ("COVID-19", [
            ("covid", 3), ("corona", 3), ("pandemic", 3), ("pandemie", 3),
            ("lockdown", 3), ("virus", 2), ("impfung", 2), ("vaccination", 2),
            ("maske", 2), ("mask", 2), ("abstand", 2), ("distance", 2)
        ])
    ]

    # Calculate scores for each topic
    topic_scores = {}

    for topic_name, keywords in topics_keywords:
        score = 0
        for keyword, weight in keywords:
            # Count occurrences of the keyword
            count = text_lower.count(keyword)
            if count > 0:
                score += count * weight

        if score > 0:
            topic_scores[topic_name] = score

    # If no topics found, add a default
    if not topic_scores:
        return ["University News"]

    # Sort topics by score and return top 6
    sorted_topics = sorted(topic_scores.items(), key=lambda x: x[1], reverse=True)
    return [topic for topic, score in sorted_topics[:6]]

# Function 10: Extract Keywords (Standardized to English)
def extract_keywords(text, language):
    """
    Extract keywords from text and standardize to English.
    """
    if not text:
        return ["ETH Zurich"]

    # Normalize and lowercase text
    text_normalized = text.lower()

    # Define keyword mapping from German to English
    de_to_en = {
        "eth zürich": "ETH Zurich",
        "eth-karte": "ETH Card",
        "karte": "Card",
        "ausweis": "Identification Card",
        "studierende": "Students",
        "studenten": "Students",
        "student": "Student",
        "mitarbeitende": "Staff",
        "mitarbeiter": "Staff",
        "personal": "Personnel",
        "design": "Design",
        "erscheinungsbild": "Visual Identity",
        "forschung": "Research",
        "wissenschaft": "Science",
        "preiserhöhung": "Price Increase",
        "kosten": "Costs",
        "mensa": "Cafeteria",
        "verpflegung": "Catering",
        "unwetter": "Storm",
        "sturm": "Storm",
        "wetter": "Weather",
        "nachhaltigkeit": "Sustainability",
        "umwelt": "Environment",
        "klima": "Climate",
        "gebäude": "Building",
        "campus": "Campus",
        "hönggerberg": "Hönggerberg",
        "zentrum": "Campus Center",
        "technologie": "Technology",
        "innovation": "Innovation",
        "digital": "Digital",
        "lehre": "Teaching",
        "bildung": "Education",
        "kommunikation": "Communication"
    }

    # Initialize keywords list
    keywords = []

    # First check for specific multi-word terms
    if language == "de":
        # German multi-word terms with their English translations
        multi_word_de = {
            "eth zürich": "ETH Zurich",
            "eth-karte": "ETH Card",
            "elektronische karte": "Electronic ID",
            "corporate design": "Corporate Design",
            "neue design": "New Design",
            "universität zürich": "University of Zurich",
            "hönggerberg campus": "Hönggerberg Campus"
        }

        for de_term, en_term in multi_word_de.items():
            if de_term in text_normalized and en_term not in keywords:
                keywords.append(en_term)
    else:
        # English multi-word terms
        multi_word_en = [
            "ETH Zurich", "ETH Card", "Electronic ID", "Corporate Design",
            "New Design", "University of Zurich", "Hönggerberg Campus"
        ]

        for term in multi_word_en:
            if term.lower() in text_normalized and term not in keywords:
                keywords.append(term)

    # Check for single words and translate if German
    # First tokenize by removing punctuation and splitting
    translator = str.maketrans('', '', string.punctuation)
    words = text_normalized.translate(translator).split()

    # Get word frequency
    word_freq = {}
    for word in words:
        if len(word) > 3:  # Skip very short words
            word_freq[word] = word_freq.get(word, 0) + 1

    # Get the most frequent words
    sorted_words = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)

    # Add top words, translating from German if needed
    for word, _ in sorted_words[:15]:  # Get top 15 words to ensure we have enough after filtering
        if language == "de" and word in de_to_en:
            en_word = de_to_en[word]
            if en_word not in keywords:
                keywords.append(en_word)
        elif language == "en" and len(word) > 3:
            # Capitalize the first letter for English words
            word_cap = word.capitalize()
            if word_cap not in keywords:
                keywords.append(word_cap)

    # Add specific ETH-related keywords if they're relevant
    eth_keywords = ["ETH Zurich", "University", "Research", "Science", "Campus", "Education"]
    for keyword in eth_keywords:
        if keyword not in keywords:
            keywords.append(keyword)

    # Return up to 7 keywords (deduped)
    unique_keywords = []
    seen = set()
    for kw in keywords:
        if kw.lower() not in seen:
            seen.add(kw.lower())
            unique_keywords.append(kw)

    return unique_keywords[:7]

# Function to generate a summary
def generate_summary(text, max_length=200):
    """
    Generate a summary of the text.
    Keep in original language.
    """
    if not text:
        return ""

    # Take the first few sentences, up to max_length
    sentences = re.split(r'(?<=[.!?])\s+', text)

    summary = ""
    for sentence in sentences:
        if len(summary) + len(sentence) <= max_length:
            summary += sentence + " "
        else:
            break

    summary = summary.strip()

    # Add ellipsis if truncated
    if len(summary) < len(text) and summary:
        summary += "..."

    return summary

# Function 11: Generate Rich Metadata (Standardized to English)
def generate_rich_metadata(text, language):
    """
    Generate rich metadata for the article with standardized English fields.
    """
    if not text:
        return {
            "document_length_words": 0,
            "readability_score": "unknown",
            "sentiment": "neutral",
            "contains_compound_words": False,
            "contains_umlauts": False,
            "embedding_vector": "[...]",
            "audience_type": ["Students", "Staff"],
            "context_tags": ["ETH Internal"]
        }

    # Count words
    words = re.findall(r'\b\w+\b', text)
    word_count = len(words)

    # Determine readability (standardized to English)
    # More accurate approach based on sentence length and word length
    sentences = re.split(r'[.!?]+', text)
    sentence_count = len([s for s in sentences if s.strip()])

    avg_words_per_sentence = word_count / max(sentence_count, 1)
    long_words = len([w for w in words if len(w) > 6])
    long_word_percentage = (long_words / max(word_count, 1)) * 100

    if avg_words_per_sentence > 25 or long_word_percentage > 30:
        readability = "complex"
    elif avg_words_per_sentence > 15 or long_word_percentage > 20:
        readability = "moderately complex"
    else:
        readability = "easy to read"

    # Improved sentiment analysis with more keywords
    positive_patterns = [
        r'\b(?:gut|besser|positiv|erfolgreich|vorteil|nutzen|förder|erfreut|freude|verbessert)',
        r'\b(?:good|better|positive|successful|advantage|benefit|promote|pleased|improve|happy)'
    ]

    negative_patterns = [
        r'\b(?:schlecht|problem|negativ|schwierig|nachteil|kritisch|belastung|sorge|verschlechter)',
        r'\b(?:bad|problem|negative|difficult|disadvantage|critical|burden|worry|worsen)'
    ]

    positive_count = 0
    for pattern in positive_patterns:
        positive_count += len(re.findall(pattern, text.lower()))

    negative_count = 0
    for pattern in negative_patterns:
        negative_count += len(re.findall(pattern, text.lower()))

    # Calculate sentiment ratio
    total = positive_count + negative_count
    if total == 0:
        sentiment = "neutral"
    elif positive_count > negative_count * 2:
        sentiment = "positive"
    elif negative_count > positive_count * 2:
        sentiment = "negative"
    elif positive_count > negative_count:
        sentiment = "slightly positive"
    elif negative_count > positive_count:
        sentiment = "slightly negative"
    else:
        sentiment = "neutral"

    # Check for compound words and umlauts
    contains_compound_words = bool(re.search(r'\b\w{15,}\b', text))
    contains_umlauts = bool(re.search(r'[äöüÄÖÜß]', text))

    # Determine audience type (standardized to English)
    audience_type = []

    # More specific patterns for different audience types
    audience_patterns = [
        ("Students", [r'\bstud(?:ent|ierend|ium|ies)', r'\bschule\b', r'\buniversity\b']),
        ("Staff", [r'\bmitarbeit|\bpersonal|\bstaff|\bemployee|\bangestellt']),
        ("Faculty", [r'\bprofessor|\bdozent|\bfaculty|\blectur|\bdocent']),
        ("Researchers", [r'\bforsch|\bresearch|\bwissenschaft|\bscience|\blabor|\blab\b']),
        ("Administration", [r'\bverwaltung|\badministration|\bleitung|\bmanagement']),
        ("General Public", [r'\böffentlich|\bpublic|\ballgemein|\bgeneral|\bcommunity|\bgesellschaft'])
    ]

    for audience, patterns in audience_patterns:
        for pattern in patterns:
            if re.search(pattern, text.lower()):
                audience_type.append(audience)
                break

    # Default audience if none detected
    if not audience_type:
        audience_type = ["Students", "Staff"]

    # Context tags (standardized to English)
    context_tags = []

    # More comprehensive context tagging system
    context_patterns = [
        ("Financial", [r'\bfinan|\bkosten|\bbudget|\bcost|\bprice|\bpreis|\bgeld|\bmoney']),
        ("Catering", [r'\bmensa|\bessen|\bverpfleg|\bfood|\bdining|\bcafeteria|\bmahlzeit|\bmeal']),
        ("COVID-19", [r'\bcorona|\bcovid|\bpandemie|\bpandemic|\blockdown|\bvirus']),
        ("Sustainability", [r'\bnachhaltig|\bsustainable|\benvironment|\bumwelt|\bklima|\bclimate']),
        ("Infrastructure", [r'\bgebäude|\bbuilding|\binfrastruktur|\bcampus|\braum|\bspace']),
        ("Technology", [r'\btechnologie|\btechnology|\bdigital|\bsoftware|\brfid|\bapp']),
        ("Research", [r'\bforschung|\bresearch|\bwissenschaft|\bscience|\bstudie|\bstudy']),
        ("Education", [r'\bausbildung|\beducation|\bstudium|\bstudies|\blehre|\bteaching']),
        ("Administrative", [r'\bverwaltung|\badministration|\bmanagement|\bleitung|\bpolicy']),
        ("Communications", [r'\bkommunikation|\bcommunication|\bmitteilung|\bannouncement']),
        ("Events", [r'\bveranstaltung|\bevent|\bkonferenz|\bconference|\bmeeting|\bseminar']),
        ("Weather", [r'\bwetter|\bweather|\bsturm|\bstorm|\bregen|\brain|\btemperatur']),
        ("International", [r'\binternational|\bglobal|\bweltweit|\bworldwide|\bausland']),
        ("Career", [r'\bkarriere|\bcareer|\bjob|\bstelle|\bposition|\bbewerbung|\bapplication'])
    ]

    # Check for context tags
    for tag, patterns in context_patterns:
        for pattern in patterns:
            if re.search(pattern, text.lower()):
                context_tags.append(tag)
                break

    # Check if ETH-related
    if re.search(r'\beth|\bethz|\beidgenössische|\bpoly', text.lower()):
        context_tags.append("ETH Internal")

    # Limit to most relevant tags (max 4)
    if len(context_tags) > 4:
        context_tags = context_tags[:4]
    elif not context_tags:
        context_tags = ["University News"]

    # Create rich metadata with all fields in English
    rich_metadata = {
        "document_length_words": word_count,
        "readability_score": readability,
        "sentiment": sentiment,
        "contains_compound_words": contains_compound_words,
        "contains_umlauts": contains_umlauts,
        "embedding_vector": "[...]",
        "audience_type": audience_type,
        "context_tags": context_tags
    }

    return rich_metadata

# Main article processing function
def process_article(markdown_text, filename, filepath):
    """
    Process a single article through all cleaning steps.
    Args:
        markdown_text (str): The content of the .md file
        filename (str): The name of the file (e.g., "article.md")
        filepath (Path or str): Full path to the .md file, used to extract date if needed
    Returns:
        dict: Structured metadata and article info
    """
    try:
        # Step a: Basic cleaning
        cleaned_text = basic_text_cleaning(markdown_text)

        # Step b: Extract article structure
        article_structure = extract_article_structure(cleaned_text)

        # Step c: Extract main content (keep in original language)
        main_content = extract_main_content(article_structure)

        # Step d: Detect language
        language = detect_language(main_content or cleaned_text)

        # Step e: Extract title (keep in original language)
        title = extract_title(article_structure, filename)

        # Step f: Extract date (standardized format, fallback to folder structure)
        date = extract_date(main_content or cleaned_text, filepath=filepath)

        # Step g: Extract source (keep in original language)
        source = extract_source(article_structure, main_content or cleaned_text, language)

        # Step h: Extract named entities (keep in original language)
        named_entities = extract_named_entities(main_content, language)

        # Step i: Extract topics (standardized to English)
        topics = extract_topics(main_content, language)

        # Step j: Extract keywords (standardized to English)
        keywords = extract_keywords(main_content, language)

        # Step k: Generate summary (keep in original language)
        summary = generate_summary(main_content)

        # Step l: Generate rich metadata (standardized to English)
        rich_metadata = generate_rich_metadata(main_content, language)

        # Step m: Final structured document
        processed_article = {
            "language": language,
            "title": title,
            "date": date,
            "source": source,
            "main_content": main_content,
            "named_entities": named_entities,
            "topics": topics,
            "keywords": keywords,
            "summary": summary,
            "rich_metadata": rich_metadata
        }

        return processed_article

    except Exception as e:
        logger.error(f"Error processing {filename}: {str(e)}")

        # Return a minimal valid structure in case of failure
        return {
            "language": "unknown",
            "title": filename.replace('.md', '').replace('-', ' ').title(),
            "date": "",
            "source": "ETH Zurich",
            "main_content": "",
            "named_entities": [],
            "topics": ["University News"],
            "keywords": ["ETH Zurich"],
            "summary": "",
            "rich_metadata": {
                "document_length_words": 0,
                "readability_score": "unknown",
                "sentiment": "neutral",
                "contains_compound_words": False,
                "contains_umlauts": False,
                "embedding_vector": "[...]",
                "audience_type": ["Students", "Staff"],
                "context_tags": ["ETH Internal"]
            }
        }

In [ ]:
from pathlib import Path
import os
import json
import re
import string
import unicodedata
import logging
from tqdm import tqdm
from langdetect import detect
from pathlib import Path

# Step 1: Define base folder
hknews_dir = Path("/content/news-qa-ethz1/HKNews")

# Step 2: Find all .md files recursively
md_files = list(hknews_dir.rglob("*.md"))
print(f"✅ Found {len(md_files)} markdown files.")

# Step 3: Process and save JSON in the same directory
for md_path in tqdm(md_files, desc="Processing articles"):
    try:
        with open(md_path, 'r', encoding='utf-8') as f:
            markdown_text = f.read()

        result = process_article(markdown_text, filename=os.path.basename(md_path), filepath=md_path)

        # Output path: same folder, replace .md with .json
        json_path = md_path.with_suffix(".json")
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

    except Exception as e:
        logger.error(f"❌ Failed to process {md_path}: {e}")



✅ Found 4390 markdown files.


Processing articles: 100%|██████████| 4390/4390 [01:05<00:00, 66.85it/s] 


In [ ]:
# Navigate to your repo (if not already there)
#%cd /content/news-qa-ethz1

# Track all newly created JSON files
!git add HKNews/**/*.json

# Commit the changes with a clear message
!git commit -m "🔄 Add processed article metadata as JSON files"

# Push the changes to your GitHub fork
!git push origin main

Auto packing the repository in background for optimum performance.
See "git help gc" for manual housekeeping.
[main 3c2b2f92] 🔄 Add processed article metadata as JSON files
 3243 files changed, 3261 insertions(+), 3263 deletions(-)
To https://github.com/ddannia/news-qa-ethz1.git
 ! [rejected]          main -> main (fetch first)
error: failed to push some refs to 'https://github.com/ddannia/news-qa-ethz1.git'
hint: Updates were rejected because the remote contains work that you do
hint: not have locally. This is usually caused by another repository pushing
hint: to the same ref. You may want to first integrate the remote changes
hint: (e.g., 'git pull ...') before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.


In [ ]:
!git push origin main

Enumerating objects: 6532, done.
Counting objects: 100% (6525/6525), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3672/3672), done.
Writing objects: 100% (3765/3765), 1.73 MiB | 6.18 MiB/s, done.
Total 3765 (delta 2783), reused 1070 (delta 92), pack-reused 0
remote: Resolving deltas: 100% (2783/2783), completed with 2689 local objects.
remote: This repository moved. Please use the new location:
remote:   https://github.com/Ddannia/news-qa-ethz1.git
To https://github.com/ddannia/news-qa-ethz1.git
   21c70d42..af202ebe  main -> main


## Version 7 - refined


In [14]:
import os
import json
import re
import string
import unicodedata
import logging
from tqdm import tqdm
from langdetect import detect
from pathlib import Path


logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def basic_text_cleaning(text):
    text = unicodedata.normalize('NFKC', text)
    text = text.strip()
    return text

def detect_language(text):
    if not text:
        return "unknown"

    try:
        sample = text[:1000]
        language = detect(sample)
        return language
    except:
        return "unknown"

def extract_article_structure(markdown_text):
    article = {
        'original_filename': '',
        'sections': {},
    }

    header_match = re.search(r'^# (.+?)$', markdown_text, re.MULTILINE)
    if header_match:
        article['original_filename'] = header_match.group(1).strip()

    lines = markdown_text.split('\n')
    current_section = None
    section_content = []

    for line in lines:
        if line.startswith('## '):
            if current_section is not None:
                article['sections'][current_section] = '\n'.join(section_content).strip()

            current_section = line[3:].strip()
            section_content = []
        elif current_section is not None:
            section_content.append(line)

    if current_section is not None and section_content:
        article['sections'][current_section] = '\n'.join(section_content).strip()

    return article

def extract_title(article_dict, filename):
    title_sections = ["In brief", "Main article", "Title", "Headline", "Titel", "Überschrift", "Kurz gefasst"]

    for section_name in title_sections:
        if section_name in article_dict['sections']:
            content = article_dict['sections'][section_name]
            lines = content.split('\n')
            if lines:
                for line in lines:
                    if line.strip():
                        return line.strip()

    if article_dict['original_filename']:
        clean_filename = article_dict['original_filename'].replace('.html', '')
        words = clean_filename.split('-')
        title = ' '.join(word.capitalize() for word in words)
        return title

    clean_filename = filename.replace('.md', '')
    words = clean_filename.split('-')
    title = ' '.join(word.capitalize() for word in words)
    return title

def extract_date(text, filepath=None):
    date_patterns = [
        r'(\d{1,2})\.(\d{1,2})\.(\d{4})',  # DD.MM.YYYY
        r'(\d{1,2})[/\.](\d{1,2})[/\.](\d{4})',  # DD/MM/YYYY
        r'(\d{4})-(\d{1,2})-(\d{1,2})',  # YYYY-MM-DD
        r'(\d{1,2})(?:st|nd|rd|th)? (?:of )?([A-Za-z]+)[,]? (\d{4})'  # e.g., 13th July 2021
    ]

    academic_patterns = [
        r'published (?:online )?(?:on )?(?:the )?(\d{1,2})(?:st|nd|rd|th)? ([A-Za-z]+)(?:[,]? (\d{4}))?',  # published online 24th June 2021
        r'published (?:online )?(?:on )?(?:the )?([A-Za-z]+) (\d{1,2})(?:st|nd|rd|th)?(?:[,]? (\d{4}))?',  # published online June 24th 2021
        r'published:? (\d{1,2})\.(\d{1,2})\.(\d{4})',  # published: 24.06.2021
        r'published:? (\d{1,2})[/\-](\d{1,2})[/\-](\d{4})',  # published: 24/06/2021 or 24-06-2021
        r'published in ([A-Za-z]+) (\d{4})',  # published in June 2021
        r'(?:date|pub\. date|publication date)[:\s]+(\d{1,2})\.(\d{1,2})\.(\d{4})',  # publication date: 24.06.2021
        r'(?:date|pub\. date|publication date)[:\s]+(\d{1,2})[/\-](\d{1,2})[/\-](\d{4})'  # publication date: 24/06/2021
    ]

    partial_date_patterns = [
        r'(\d{2})\.(\d{2})',  # MM.YY (like 06.24 in your example)
        r'(\d{2})[/\-](\d{4})',  # MM/YYYY or MM-YYYY
        r'(\d{2})[/\-](\d{2})',  # MM/YY or MM-YY
    ]

    months = {
        'january': '01', 'february': '02', 'march': '03', 'april': '04',
        'may': '05', 'june': '06', 'july': '07', 'august': '08',
        'september': '09', 'october': '10', 'november': '11', 'december': '12',
        'januar': '01', 'februar': '02', 'märz': '03', 'april': '04',
        'mai': '05', 'juni': '06', 'juli': '07', 'august': '08',
        'september': '09', 'oktober': '10', 'november': '11', 'dezember': '12',
        'jan': '01', 'feb': '02', 'mar': '03', 'apr': '04',
        'may': '05', 'jun': '06', 'jul': '07', 'aug': '08',
        'sep': '09', 'oct': '10', 'nov': '11', 'dec': '12'
    }

    year_pattern = r'\b(20\d{2})\b'
    year_match = re.search(year_pattern, text)
    year_from_text = year_match.group(1) if year_match else None

    path_month = None
    path_year = None
    if filepath:
        parts = Path(filepath).parts
        for i, part in enumerate(parts):
            if re.fullmatch(r'20\d{2}', part):
                path_year = part
                if i + 1 < len(parts):
                    maybe_month = parts[i + 1]
                    if re.fullmatch(r'\d{1,2}', maybe_month):
                        path_month = maybe_month.zfill(2)
                break

    for pattern in date_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            if len(match.groups()) == 3:
                if pattern == r'(\d{4})-(\d{1,2})-(\d{1,2})':
                    year, month, day = match.groups()
                elif pattern == r'(\d{1,2})(?:st|nd|rd|th)? (?:of )?([A-Za-z]+)[,]? (\d{4})':
                    day, month_name, year = match.groups()
                    month = months.get(month_name.lower(), '01')
                else:
                    day, month, year = match.groups()
                return f"{year}-{month.zfill(2)}-{day.zfill(2)}"

    for pattern in academic_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            groups = match.groups()

            if len(groups) == 3:
                if re.match(r'\d{1,2}', groups[0]) and re.match(r'[A-Za-z]+', groups[1]):
                    day, month_name, year = groups
                    month = months.get(month_name.lower(), '01')
                elif re.match(r'[A-Za-z]+', groups[0]) and re.match(r'\d{1,2}', groups[1]):
                    month_name, day, year = groups
                    month = months.get(month_name.lower(), '01')
                else:
                    day, month, year = groups

                if not year and year_from_text:
                    year = year_from_text

                if year and day and (re.match(r'\d+', day) and len(year) == 4):
                    return f"{year}-{month.zfill(2)}-{day.zfill(2)}"

            elif len(groups) == 2:
                if re.match(r'[A-Za-z]+', groups[0]) and re.match(r'\d{4}', groups[1]):
                    month_name, year = groups
                    month = months.get(month_name.lower(), '01')
                    return f"{year}-{month.zfill(2)}-01"

    for pattern in partial_date_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            groups = match.groups()

            if len(groups) == 2:
                if re.match(r'\d+', groups[0]) and re.match(r'\d+', groups[1]):
                    month_val = int(groups[0])
                    if 1 <= month_val <= 12:
                        month = groups[0].zfill(2)
                        year_part = groups[1]

                        if len(year_part) == 2:
                            year = f"20{year_part}"
                        else:
                            year = year_part

                        if len(year) < 4 and year_from_text:
                            year = year_from_text

                        if len(year) == 4:
                            return f"{year}-{month}-01"

    if year_from_text and path_month:
        return f"{year_from_text}-{path_month}-01"
    elif year_from_text:
        return f"{year_from_text}-01-01"

    if path_year and path_month:
        return f"{path_year}-{path_month}-01"
    elif path_year:
        return f"{path_year}-01-01"

    return ""

def extract_source(article_dict, text, language):
    reference_sections = ["Reference", "Referenz", "Quelle", "Source"]
    for section in reference_sections:
        if section in article_dict['sections']:
            source_text = article_dict['sections'][section]
            if source_text.strip():
                return source_text.strip()

    source_patterns = [
        r'(?:Source|Quelle):\s*([^\.]+)',
        r'(?:By|Von|Author|Autor):\s*([^\.]+)',
        r'(?:Copyright|©)\s*([^\.]+)',
        r'(?:Published by|Veröffentlicht von):\s*([^\.]+)'
    ]

    for pattern in source_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1).strip()

    eth_units = [
        r'(ETH[\s-]Zürich[^\.;,]*(?:Kommunikation|Communication|Department|Departement|Abteilung)[^\.;,]*)',
        r'(ETH[\s-]Zurich[^\.;,]*(?:Communication|Department|Unit|Division)[^\.;,]*)',
        r'((?:Hochschulkommunikation|University Communication)[^\.;,]*ETH[^\.;,]*)'
    ]

    for pattern in eth_units:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1).strip()

    if article_dict['original_filename']:
        domain_match = re.search(r'(?:www\.)?([a-z0-9-]+)\.(?:com|org|edu|ch|de)', article_dict['original_filename'], re.IGNORECASE)
        if domain_match:
            domain = domain_match.group(1).lower()
            if 'ethz' in domain:
                return "ETH Zürich" if language == "de" else "ETH Zurich"
            elif 'uzh' in domain:
                return "Universität Zürich" if language == "de" else "University of Zurich"
            else:
                domain = domain.replace('-', ' ').title()
                return domain

    if "ETH" in text:
        return "ETH Zürich, Hochschulkommunikation" if language == "de" else "ETH Zurich, University Communications"

    return "Unbekannte Quelle" if language == "de" else "Unknown source"

def extract_main_content(article_dict):
    skip_sections = ["Reference", "Referenz", "Quelle", "Source"]

    if len(article_dict['sections']) == 1:
        section_name = list(article_dict['sections'].keys())[0]
        if section_name not in skip_sections:
            return article_dict['sections'][section_name]

    content_parts = []

    for section_name, content in article_dict['sections'].items():
        if section_name not in skip_sections and content.strip():
            if len(article_dict['sections']) > 1:
                content_parts.append(f"{section_name}: {content}")
            else:
                content_parts.append(content)

    return "\n\n".join(content_parts)

def extract_named_entities(text, language):
    if not text:
        return []

    normalized_text = text.replace('\n', ' ').replace('  ', ' ')

    # Common words to exclude
    stop_words = set([
        "The", "This", "That", "These", "Those", "They", "Their", "And", "But", "However", "From", "Both", "When", "Then", "In", "On", "At",
        "Der", "Die", "Das", "Diese", "Dieser", "Dieses", "Jene", "Und", "Aber", "Jedoch", "Von", "Beide", "Wenn", "Dann", "In", "Auf", "Bei"
    ])

    # Prepositions and articles to exclude
    prepositions = set([
        "in", "on", "at", "by", "for", "with", "from", "to", "of", "over", "under", "between", "among", "through", "during", "after", "before",
        "in", "auf", "an", "bei", "für", "mit", "von", "zu", "über", "unter", "zwischen", "durch", "während", "nach", "vor"
    ])

    articles = set(["a", "an", "the", "ein", "eine", "der", "die", "das"])

    # Look for organization names (2-4 capitalized words in sequence)
    org_pattern = r'\b([A-Z][a-zäöüÄÖÜß]+(?:[ \-][A-Z][a-zäöüÄÖÜß]+){1,3})\b'
    org_matches = re.findall(org_pattern, normalized_text)

    entities = []

    for match in org_matches:
        # Skip if it's just a common stop word
        if match in stop_words:
            continue

        # Skip if it starts with a prepositional phrase
        words = match.split()
        if words[0].lower() in prepositions or words[0].lower() in articles:
            continue

        # Skip if it contains any month names
        month_names = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
        if any(month in match for month in month_names):
            continue

        # Skip phrases starting with numbers
        if re.match(r'\d+', match):
            continue

        # Check if entity is suitable (at least 2 characters, not just initials)
        if len(match) > 3 and not re.match(r'^[A-Z]\. [A-Z]\.', match):
            entities.append(match)

    # ETH-specific named entities
    eth_specific = []
    if language == "de":
        eth_specific = [
            "ETH Zürich", "ETH-Zürich", "ETH", "Universität Zürich", "UZH",
            "Hönggerberg", "ASVZ", "Polyterrasse"
        ]
    else:
        eth_specific = [
            "ETH Zurich", "ETH", "University of Zurich", "UZH",
            "Hönggerberg", "ASVZ", "Polyterrasse"
        ]

    for entity in eth_specific:
        if entity in normalized_text and entity not in entities:
            entities.append(entity)

    # Look for people names with titles
    titles = ["Prof", "Professor", "Dr", "Professorin", "Doktor"]
    for title in titles:
        name_pattern = f"{title}\\.?\\s+([A-Z][a-zäöüÄÖÜß]+(?:\\s+[A-Z][a-zäöüÄÖÜß]+){{1,2}})"
        prof_matches = re.findall(name_pattern, normalized_text)
        for match in prof_matches:
            if match not in entities and not any(word.lower() in prepositions for word in match.split()):
                entities.append(match)

    # Deduplicate entities
    unique_entities = []
    seen = set()
    for entity in entities:
        if entity.lower() not in seen:
            seen.add(entity.lower())
            unique_entities.append(entity)

    return unique_entities[:10]

def extract_topics(text, language):
    if not text:
        return ["University News"]

    text_lower = text.lower()

    topics_keywords = [
        ("Infrastructure", [
            ("karte", 3), ("card", 3), ("ausweis", 3), ("identification", 3),
            ("gebäude", 2), ("building", 2), ("raum", 1), ("room", 1),
            ("campus", 2), ("hönggerberg", 3), ("zentrum", 1), ("center", 1)
        ]),

        ("Technology & Innovation", [
            ("technologie", 3), ("technology", 3), ("innovation", 3),
            ("digital", 2), ("software", 2), ("computer", 2), ("app", 2),
            ("rfid", 3), ("chip", 2), ("elektronisch", 1), ("electronic", 1),
            ("entwicklung", 1), ("development", 1), ("programmier", 2), ("coding", 2)
        ]),

        ("Research", [
            ("forschung", 3), ("research", 3), ("wissenschaft", 3), ("science", 3),
            ("studie", 2), ("study", 2), ("experiment", 2), ("projekt", 1), ("project", 1),
            ("entdeckung", 2), ("discovery", 2), ("publikation", 2), ("publication", 2)
        ]),

        ("Education", [
            ("ausbildung", 3), ("education", 3), ("studium", 3), ("studies", 3),
            ("student", 3), ("studierend", 3), ("lehre", 3), ("teaching", 3),
            ("vorlesung", 2), ("lecture", 2), ("kurs", 2), ("course", 2),
            ("prüfung", 2), ("exam", 2), ("seminar", 2), ("unterricht", 2)
        ]),

        ("Finance", [
            ("finanzen", 3), ("finance", 3), ("kosten", 3), ("costs", 3),
            ("preis", 3), ("price", 3), ("erhöhung", 2), ("increase", 2),
            ("budget", 3), ("geld", 2), ("money", 2), ("zahlung", 1), ("payment", 1)
        ]),

        ("University Administration", [
            ("verwaltung", 3), ("administration", 3), ("leitung", 2), ("management", 2),
            ("präsident", 2), ("president", 2), ("rektor", 2), ("rector", 2),
            ("direktor", 2), ("director", 2), ("strategie", 2), ("strategy", 2)
        ]),

        ("Campus Life", [
            ("campus", 3), ("student", 2), ("mensa", 3), ("cafeteria", 3),
            ("essen", 2), ("food", 2), ("veranstaltung", 2), ("event", 2),
            ("freizeit", 2), ("leisure", 2), ("sport", 2), ("asvz", 3)
        ]),

        ("International", [
            ("international", 3), ("global", 3), ("weltweit", 2), ("worldwide", 2),
            ("ausland", 2), ("foreign", 2), ("kooperation", 2), ("cooperation", 2),
            ("austausch", 2), ("exchange", 2), ("partner", 2)
        ]),

        ("Sustainability", [
            ("nachhaltig", 3), ("sustainable", 3), ("umwelt", 3), ("environment", 3),
            ("klima", 3), ("climate", 3), ("grün", 2), ("green", 2),
            ("energie", 2), ("energy", 2), ("ressource", 2), ("resource", 2)
        ]),

        ("Weather & Environment", [
            ("wetter", 3), ("weather", 3), ("sturm", 3), ("storm", 3),
            ("umwelt", 2), ("environment", 2), ("klima", 2), ("climate", 2),
            ("regen", 2), ("rain", 2), ("wind", 2), ("temperatur", 2), ("temperature", 2)
        ]),

        ("Communication", [
            ("kommunikation", 3), ("communication", 3), ("mitteilung", 3), ("announcement", 3),
            ("information", 2), ("bericht", 2), ("report", 2), ("news", 3),
            ("presse", 2), ("press", 2), ("media", 2), ("medien", 2)
        ]),

        ("Catering & Food", [
            ("mensa", 3), ("cafeteria", 3), ("essen", 3), ("food", 3),
            ("verpflegung", 3), ("catering", 3), ("restaurant", 2),
            ("speise", 2), ("meal", 2), ("menü", 2), ("menu", 2)
        ]),

        ("Staff", [
            ("mitarbeiter", 3), ("staff", 3), ("personal", 3), ("employee", 3),
            ("anstellung", 2), ("employment", 2), ("arbeit", 2), ("work", 2),
            ("position", 2), ("stelle", 2), ("job", 2)
        ]),

        ("COVID-19", [
            ("covid", 3), ("corona", 3), ("pandemic", 3), ("pandemie", 3),
            ("lockdown", 3), ("virus", 2), ("impfung", 2), ("vaccination", 2),
            ("maske", 2), ("mask", 2), ("abstand", 2), ("distance", 2)
        ])
    ]

    topic_scores = {}

    for topic_name, keywords in topics_keywords:
        score = 0
        for keyword, weight in keywords:
            count = text_lower.count(keyword)
            if count > 0:
                score += count * weight

        if score > 0:
            topic_scores[topic_name] = score

    if not topic_scores:
        return ["University News"]

    sorted_topics = sorted(topic_scores.items(), key=lambda x: x[1], reverse=True)
    return [topic for topic, score in sorted_topics[:6]]

def extract_keywords(text, language):
    if not text:
        return ["ETH Zurich"]

    text_normalized = text.lower()

    # Expanded stop words list to filter out unsuitable keywords
    stop_words = set([
        "the", "and", "or", "but", "because", "as", "if", "when", "than", "but", "not",
        "during", "at", "by", "for", "with", "about", "against", "between", "into",
        "through", "from", "to", "in", "on", "that", "this", "these", "those", "also",
        "der", "die", "das", "und", "oder", "aber", "weil", "als", "wenn", "als", "aber", "nicht",
        "während", "bei", "für", "mit", "über", "gegen", "zwischen", "durch",
        "von", "zu", "in", "auf", "dass", "diese", "auch", "dann", "mehr", "andere"
    ])

    de_to_en = {
        "eth zürich": "ETH Zurich",
        "eth-karte": "ETH Card",
        "karte": "Card",
        "ausweis": "Identification Card",
        "studierende": "Students",
        "studenten": "Students",
        "student": "Student",
        "mitarbeitende": "Staff",
        "mitarbeiter": "Staff",
        "personal": "Personnel",
        "design": "Design",
        "erscheinungsbild": "Visual Identity",
        "forschung": "Research",
        "wissenschaft": "Science",
        "preiserhöhung": "Price Increase",
        "kosten": "Costs",
        "mensa": "Cafeteria",
        "verpflegung": "Catering",
        "unwetter": "Storm",
        "sturm": "Storm",
        "wetter": "Weather",
        "nachhaltigkeit": "Sustainability",
        "umwelt": "Environment",
        "klima": "Climate",
        "gebäude": "Building",
        "campus": "Campus",
        "hönggerberg": "Hönggerberg",
        "zentrum": "Campus Center",
        "technologie": "Technology",
        "innovation": "Innovation",
        "digital": "Digital",
        "lehre": "Teaching",
        "bildung": "Education",
        "kommunikation": "Communication"
    }

    keywords = []

    if language == "de":
        multi_word_de = {
            "eth zürich": "ETH Zurich",
            "eth-karte": "ETH Card",
            "elektronische karte": "Electronic ID",
            "corporate design": "Corporate Design",
            "neue design": "New Design",
            "universität zürich": "University of Zurich",
            "hönggerberg campus": "Hönggerberg Campus"
        }

        for de_term, en_term in multi_word_de.items():
            if de_term in text_normalized and en_term not in keywords:
                keywords.append(en_term)
    else:
        multi_word_en = [
            "ETH Zurich", "ETH Card", "Electronic ID", "Corporate Design",
            "New Design", "University of Zurich", "Hönggerberg Campus"
        ]

        for term in multi_word_en:
            if term.lower() in text_normalized and term not in keywords:
                keywords.append(term)

    translator = str.maketrans('', '', string.punctuation)
    words = text_normalized.translate(translator).split()

    # Filter out stopwords
    filtered_words = [word for word in words if word not in stop_words and len(word) > 3]

    word_freq = {}
    for word in filtered_words:
        word_freq[word] = word_freq.get(word, 0) + 1

    sorted_words = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)

    for word, _ in sorted_words[:20]:
        if language == "de" and word in de_to_en:
            en_word = de_to_en[word]
            if en_word not in keywords:
                keywords.append(en_word)
        elif language == "en" and len(word) > 3 and not any(char.isdigit() for char in word):
            # Capitalize English words and skip numeric words
            word_cap = word.capitalize()
            if word_cap not in keywords:
                keywords.append(word_cap)

    eth_keywords = ["ETH Zurich", "University", "Research", "Science", "Campus", "Education"]
    for keyword in eth_keywords:
        if keyword not in keywords:
            keywords.append(keyword)

    unique_keywords = []
    seen = set()
    for kw in keywords:
        if kw.lower() not in seen and not any(kw.lower() == sw for sw in stop_words):
            seen.add(kw.lower())
            unique_keywords.append(kw)

    return unique_keywords[:7]

def generate_summary(text, max_length=250):
    if not text:
        return ""

    # Split by sentences, but preserve sentence boundaries
    sentence_pattern = r'(?<=[.!?])\s+'
    sentences = re.split(sentence_pattern, text)

    # Ensure we get at least 2-3 sentences for a meaningful summary when possible
    min_sentences = min(3, len(sentences))

    summary = ""
    sentence_count = 0

    for sentence in sentences:
        if len(summary) + len(sentence) <= max_length:
            summary += sentence + " "
            sentence_count += 1

            # Make sure we have at least minimum number of sentences
            if sentence_count >= min_sentences and len(summary) > max_length * 0.6:
                break
        else:
            # If adding this sentence would exceed max length, only add if we haven't met minimum
            if sentence_count < min_sentences:
                summary += sentence + " "
                break
            else:
                break

    summary = summary.strip()

    if len(summary) < len(text) and summary:
        summary += "..."

    return summary

def extract_document_type(text, language):
    """
    Determine the document type based on content analysis.
    """
    text_lower = text.lower()

    # Document type patterns
    type_patterns = {
        "News Article": [
            r'news', r'artikel', r'bericht', r'mitteilung', r'press release',
            r'medienmitteilung', r'ankündigung', r'announcement'
        ],
        "Research Publication": [
            r'study', r'studie', r'research', r'forschung', r'publication',
            r'publikation', r'journal', r'paper', r'doi', r'published in', r'veröffentlicht in'
        ],
        "Event Announcement": [
            r'event', r'veranstaltung', r'seminar', r'workshop', r'conference',
            r'konferenz', r'invitation', r'einladung', r'upcoming', r'registration'
        ],
        "Policy Update": [
            r'policy', r'richtlinie', r'regulation', r'regulierung', r'guideline',
            r'leitfaden', r'rule', r'regel', r'procedure', r'prozedur'
        ],
        "Interview": [
            r'interview', r'conversation', r'gespräch', r'qa', r'q&a',
            r'fragen und antworten', r'discusses', r'diskutiert'
        ],
        "Campus Notice": [
            r'notice', r'hinweis', r'reminder', r'erinnerung', r'announcement',
            r'ankündigung', r'attention', r'achtung', r'important information'
        ]
    }

    type_scores = {}
    for doc_type, patterns in type_patterns.items():
        score = 0
        for pattern in patterns:
            matches = re.findall(pattern, text_lower)
            score += len(matches)
        if score > 0:
            type_scores[doc_type] = score

    if not type_scores:
        return "News Article"  # Default

    # Return the document type with the highest score
    return max(type_scores.items(), key=lambda x: x[1])[0]

def extract_citation_info(text):
    """
    Extract citation information from text.
    """
    if not text:
        return {}

    # Extract DOI references
    doi_pattern = r'(?:DOI|doi):\s*([0-9.]+\/[^\s,;]+)'
    doi_match = re.search(doi_pattern, text)
    doi = doi_match.group(1) if doi_match else ""

    # Extract journal name
    journal_pattern = r'(?:in|In)\s+([^,.:;]+(?:Journal|Proceedings|Transactions|Review|Science|Nature)[^,.:;]*)'
    journal_match = re.search(journal_pattern, text)
    journal = journal_match.group(1).strip() if journal_match else ""

    # Extract potential authors
    author_pattern = r'([A-Z][a-zäöüÄÖÜß]+(?:\s+[A-Z][a-zäöüÄÖÜß]+)*(?:,\s*(?:et al\.?|and|und|&)\s*[A-Z][a-zäöüÄÖÜß]+(?:\s+[A-Z][a-zäöüÄÖÜß]+)*)*)'
    author_matches = re.findall(author_pattern, text)
    authors = [a.strip() for a in author_matches[:5] if len(a) > 5 and ":" not in a and "." not in a]

    citation_info = {
        "doi": doi,
        "journal": journal,
        "authors": authors
    }

    return citation_info

def extract_temporal_references(text):
    """
    Extract temporal references and event dates from text.
    """
    if not text:
        return {}

    # Match date patterns for events
    event_date_patterns = [
        r'(?:on|am)\s+(\d{1,2}(?:st|nd|rd|th)?\s+(?:of\s+)?[A-Za-z]+(?:\s+\d{4})?)',  # on 15th of June 2021
        r'(\d{1,2}(?:st|nd|rd|th)?\s+(?:of\s+)?[A-Za-z]+(?:\s+\d{4})?)\s+(?:at|from)',  # 15th June 2021 at
        r'([A-Za-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:\s+\d{4})?)\s+(?:at|from)'  # June 15th 2021 at
    ]

    event_dates = []
    for pattern in event_date_patterns:
        matches = re.findall(pattern, text)
        event_dates.extend([m.strip() for m in matches if m.strip()])

    # Extract time references
    time_patterns = [
        r'(\d{1,2}(?::\d{2})?\s*(?:am|pm|AM|PM))', # 10:30 am
        r'(\d{1,2}(?::\d{2})?\s*(?:to|-|–)\s*\d{1,2}(?::\d{2})?\s*(?:am|pm|AM|PM)?)',  # 10:30-11:45 am
        r'(\d{1,2}[:.]\d{2}\s*(?:Uhr|h))'  # 10.30 Uhr or 10:30 h
    ]

    time_references = []
    for pattern in time_patterns:
        matches = re.findall(pattern, text)
        time_references.extend([m.strip() for m in matches if m.strip()])

    # Extract event names
    event_patterns = [
        r'(?:event|veranstaltung|conference|konferenz|workshop|seminar|symposium|meeting):\s*([^.!?]+)',
        r'(?:titled|entitled|genannt):\s*"([^"]+)"',
        r'(?:event|veranstaltung|conference|konferenz|workshop|seminar|symposium|meeting)\s+"([^"]+)"'
    ]

    events = []
    for pattern in event_patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        events.extend([m.strip() for m in matches if m.strip()])

    temporal_references = {
        "event_dates": event_dates[:3],
        "time_references": time_references[:3],
        "event_names": events[:2]
    }

    return temporal_references

def extract_semantic_entities(text):
    """
    Extract semantic entities like locations, departments, and contact info.
    """
    if not text:
        return {}

    # Extract locations
    location_pattern = r'(?:in|at|to|from)\s+([A-Z][a-zäöüÄÖÜß]+(?:[ \-][A-Z][a-zäöüÄÖÜß]+){0,2})'
    location_matches = re.findall(location_pattern, text)

    # Filter locations (exclude common words and short terms)
    common_words = set(["The", "This", "That", "Their", "They", "From", "ETH", "Both"])
    locations = [loc for loc in location_matches if loc not in common_words and len(loc) > 3]

    # Extract ETH departments and units
    department_patterns = [
        r'(?:Department|Institut|Departement|Abteilung) (?:of|für|für) ([A-Za-z][a-zäöüÄÖÜß]+(?:[ \-][A-Za-z][a-zäöüÄÖÜß]+){0,4})',
        r'([A-Za-z][a-zäöüÄÖÜß]+(?:[ \-][A-Za-z][a-zäöüÄÖÜß]+){0,4}) (?:Department|Institute|Departement|Institut)'
    ]

    departments = []
    for pattern in department_patterns:
        matches = re.findall(pattern, text)
        departments.extend([d.strip() for d in matches if len(d) > 3])

    # Extract email addresses and websites
    email_pattern = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b'
    emails = re.findall(email_pattern, text)

    website_pattern = r'(?:https?://)?(?:www\.)?([a-zA-Z0-9-]+\.[a-zA-Z0-9.-]+(?:\.[a-zA-Z]{2,}))(?:/[^\s]*)?'
    websites = re.findall(website_pattern, text)

    # Extract phone numbers
    phone_pattern = r'(?:\+\d{1,3}[ -]?)?(?:\(\d{1,4}\)|\d{1,4})[ -]?\d{1,4}[ -]?\d{1,4}[ -]?\d{1,4}'
    phones = re.findall(phone_pattern, text)

    semantic_entities = {
        "locations": list(set(locations))[:5],
        "departments": list(set(departments))[:3],
        "contact_info": {
            "emails": emails[:2],
            "websites": websites[:2],
            "phones": phones[:2]
        }
    }

    return semantic_entities

def analyze_content_structure(text):
    """
    Analyze the content structure to identify sections, paragraphs, and formatting.
    """
    if not text:
        return {}

    # Count paragraphs
    paragraphs = [p for p in text.split('\n\n') if p.strip()]
    paragraph_count = len(paragraphs)

    # Count bullet points and numbered lists
    bullet_pattern = r'^\s*[\*\-•]\s+'
    bullet_points = sum(1 for line in text.split('\n') if re.match(bullet_pattern, line))

    numbered_pattern = r'^\s*\d+\.\s+'
    numbered_items = sum(1 for line in text.split('\n') if re.match(numbered_pattern, line))

    # Detect if there are headings
    heading_pattern = r'^#+\s+'
    headings = sum(1 for line in text.split('\n') if re.match(heading_pattern, line))


    # Check for links
    link_pattern = r'\[([^\]]+)\]\(([^)]+)\)'
    links = len(re.findall(link_pattern, text))

    # Structure analysis
    structure_analysis = {
        "paragraph_count": paragraph_count,
        "bullet_points": bullet_points,
        "numbered_items": numbered_items,
        "headings": headings,
        "links": links
    }

    return structure_analysis

def generate_rich_metadata(text, language):
    """
    Generate rich metadata for the article with standardized English fields.
    """
    if not text:
        return {
            "document_length_words": 0,
            "readability_score": "unknown",
            "sentiment": "neutral",
            "audience_type": ["Students", "Staff"],
            "context_tags": ["ETH Internal"]
        }

    # Count words
    words = re.findall(r'\b\w+\b', text)
    word_count = len(words)

    # Determine readability (standardized to English)
    sentences = re.split(r'[.!?]+', text)
    sentence_count = len([s for s in sentences if s.strip()])

    avg_words_per_sentence = word_count / max(sentence_count, 1)
    long_words = len([w for w in words if len(w) > 6])
    long_word_percentage = (long_words / max(word_count, 1)) * 100

    if avg_words_per_sentence > 25 or long_word_percentage > 30:
        readability = "complex"
    elif avg_words_per_sentence > 15 or long_word_percentage > 20:
        readability = "moderately complex"
    else:
        readability = "easy to read"

    # Improved sentiment analysis with more keywords
    positive_patterns = [
        r'\b(?:gut|besser|positiv|erfolgreich|vorteil|nutzen|förder|erfreut|freude|verbessert)',
        r'\b(?:good|better|positive|successful|advantage|benefit|promote|pleased|improve|happy)'
    ]

    negative_patterns = [
        r'\b(?:schlecht|problem|negativ|schwierig|nachteil|kritisch|belastung|sorge|verschlechter)',
        r'\b(?:bad|problem|negative|difficult|disadvantage|critical|burden|worry|worsen)'
    ]

    positive_count = 0
    for pattern in positive_patterns:
        positive_count += len(re.findall(pattern, text.lower()))

    negative_count = 0
    for pattern in negative_patterns:
        negative_count += len(re.findall(pattern, text.lower()))

    # Calculate sentiment ratio
    total = positive_count + negative_count
    if total == 0:
        sentiment = "neutral"
    elif positive_count > negative_count * 2:
        sentiment = "positive"
    elif negative_count > positive_count * 2:
        sentiment = "negative"
    elif positive_count > negative_count:
        sentiment = "slightly positive"
    elif negative_count > positive_count:
        sentiment = "slightly negative"
    else:
        sentiment = "neutral"

    # Determine audience type (standardized to English)
    audience_type = []

    # More specific patterns for different audience types
    audience_patterns = [
        ("Students", [r'\bstud(?:ent|ierend|ium|ies)', r'\bschule\b', r'\buniversity\b']),
        ("Staff", [r'\bmitarbeit|\bpersonal|\bstaff|\bemployee|\bangestellt']),
        ("Faculty", [r'\bprofessor|\bdozent|\bfaculty|\blectur|\bdocent']),
        ("Researchers", [r'\bforsch|\bresearch|\bwissenschaft|\bscience|\blabor|\blab\b']),
        ("Administration", [r'\bverwaltung|\badministration|\bleitung|\bmanagement']),
        ("General Public", [r'\böffentlich|\bpublic|\ballgemein|\bgeneral|\bcommunity|\bgesellschaft'])
    ]

    for audience, patterns in audience_patterns:
        for pattern in patterns:
            if re.search(pattern, text.lower()):
                audience_type.append(audience)
                break

    # Default audience if none detected
    if not audience_type:
        audience_type = ["Students", "Staff"]

    # Context tags (standardized to English)
    context_tags = []

    # More comprehensive context tagging system
    context_patterns = [
        ("Financial", [r'\bfinan|\bkosten|\bbudget|\bcost|\bprice|\bpreis|\bgeld|\bmoney']),
        ("Catering", [r'\bmensa|\bessen|\bverpfleg|\bfood|\bdining|\bcafeteria|\bmahlzeit|\bmeal']),
        ("COVID-19", [r'\bcorona|\bcovid|\bpandemie|\bpandemic|\blockdown|\bvirus']),
        ("Sustainability", [r'\bnachhaltig|\bsustainable|\benvironment|\bumwelt|\bklima|\bclimate']),
        ("Infrastructure", [r'\bgebäude|\bbuilding|\binfrastruktur|\bcampus|\braum|\bspace']),
        ("Technology", [r'\btechnologie|\btechnology|\bdigital|\bsoftware|\brfid|\bapp']),
        ("Research", [r'\bforschung|\bresearch|\bwissenschaft|\bscience|\bstudie|\bstudy']),
        ("Education", [r'\bausbildung|\beducation|\bstudium|\bstudies|\blehre|\bteaching']),
        ("Administrative", [r'\bverwaltung|\badministration|\bmanagement|\bleitung|\bpolicy']),
        ("Communications", [r'\bkommunikation|\bcommunication|\bmitteilung|\bannouncement']),
        ("Events", [r'\bveranstaltung|\bevent|\bkonferenz|\bconference|\bmeeting|\bseminar']),
        ("Weather", [r'\bwetter|\bweather|\bsturm|\bstorm|\bregen|\brain|\btemperatur']),
        ("International", [r'\binternational|\bglobal|\bweltweit|\bworldwide|\bausland']),
        ("Career", [r'\bkarriere|\bcareer|\bjob|\bstelle|\bposition|\bbewerbung|\bapplication'])
    ]

    # Check for context tags
    for tag, patterns in context_patterns:
        for pattern in patterns:
            if re.search(pattern, text.lower()):
                context_tags.append(tag)
                break

    # Check if ETH-related
    if re.search(r'\beth|\bethz|\beidgenössische|\bpoly', text.lower()):
        context_tags.append("ETH Internal")

    # Limit to most relevant tags (max 4)
    if len(context_tags) > 4:
        context_tags = context_tags[:4]
    elif not context_tags:
        context_tags = ["University News"]

    # Extract document type
    document_type = extract_document_type(text, language)

    # Extract citation information if it appears to be a research publication
    citation_info = {}
    if document_type == "Research Publication" or "Research" in context_tags:
        citation_info = extract_citation_info(text)

    # Extract temporal references for events
    temporal_info = {}
    if document_type == "Event Announcement" or "Events" in context_tags:
        temporal_info = extract_temporal_references(text)

    # Extract semantic entities for better context
    semantic_entities = extract_semantic_entities(text)

    # Generate text complexity metrics
    word_lengths = [len(w) for w in words if w]
    avg_word_length = sum(word_lengths) / max(len(word_lengths), 1)

    # Text complexity metrics
    complexity_metrics = {
        "avg_words_per_sentence": round(avg_words_per_sentence, 2),
        "avg_word_length": round(avg_word_length, 2),
        "long_word_percentage": round(long_word_percentage, 2)
    }

    # Create rich metadata with all fields in English
    rich_metadata = {
        "document_length_words": word_count,
        "readability_score": readability,
        "sentiment": sentiment,
        "document_type": document_type,
        "audience_type": audience_type,
        "context_tags": context_tags,
        "complexity_metrics": complexity_metrics
    }

    # Add additional metadata based on document type
    if citation_info:
        rich_metadata["citation_info"] = citation_info

    if temporal_info and any(temporal_info.values()):
        rich_metadata["temporal_info"] = temporal_info

    if semantic_entities and any(semantic_entities.values()):
        rich_metadata["semantic_entities"] = semantic_entities

    return rich_metadata

def load_markdown_files(input_dir):
    """
    Load all markdown files from the input directory.
    Args:
        input_dir (str): Path to the directory containing .md files
    Returns:
        list: List of tuples (file_path, file_name, file_content)
    """
    markdown_files = []

    try:
        input_path = Path(input_dir)
        if not input_path.exists():
            logger.error(f"Input directory does not exist: {input_dir}")
            return markdown_files

        # Recursively find all .md files
        for file_path in input_path.glob('**/*.md'):
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = f.read()
                    file_name = file_path.name
                    markdown_files.append((file_path, file_name, content))
                    logger.info(f"Loaded file: {file_path}")
            except Exception as e:
                logger.error(f"Error reading file {file_path}: {str(e)}")

        logger.info(f"Loaded {len(markdown_files)} markdown files")
        return markdown_files

    except Exception as e:
        logger.error(f"Error loading markdown files: {str(e)}")
        return markdown_files

def save_processed_article(processed_article, output_dir, filename):
    """
    Save processed article to JSON file.
    Args:
        processed_article (dict): Processed article data
        output_dir (str): Directory to save the output
        filename (str): Name of the output file
    """
    try:
        # Create output directory if it doesn't exist
        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)

        # Create output file path, preserving original name but changing extension to .json
        output_file = output_path / (Path(filename).stem + '.json')

        # Write JSON with proper indentation and UTF-8 encoding
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(processed_article, f, ensure_ascii=False, indent=2)

        logger.info(f"Saved processed article to {output_file}")

    except Exception as e:
        logger.error(f"Error saving processed article {filename}: {str(e)}")

def batch_process(input_files, output_dir, max_workers=4):
    """
    Process multiple markdown files in parallel.
    Args:
        input_files (list): List of tuples (file_path, file_name, file_content)
        output_dir (str): Directory to save the output
        max_workers (int): Maximum number of parallel workers
    """
    from concurrent.futures import ThreadPoolExecutor

    logger.info(f"Starting batch processing with {max_workers} workers")
    processed_count = 0
    error_count = 0

    def process_file(file_info):
        file_path, file_name, content = file_info
        try:
            # Process the article
            processed_article = process_article(content, file_name, file_path)

            # Save the processed article
            save_processed_article(processed_article, output_dir, file_name)

            return True
        except Exception as e:
            logger.error(f"Error processing {file_name}: {str(e)}")
            return False

    # Process files in parallel using ThreadPoolExecutor
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(tqdm(
            executor.map(process_file, input_files),
            total=len(input_files),
            desc="Processing articles"
        ))

    # Count successes and failures
    processed_count = sum(results)
    error_count = len(results) - processed_count

    logger.info(f"Batch processing completed. Processed: {processed_count}, Errors: {error_count}")

    return processed_count, error_count

def create_search_index(processed_dir, index_file):
    """
    Create a simple search index from processed articles.
    Args:
        processed_dir (str): Directory containing processed JSON files
        index_file (str): Output index file path
    """
    try:
        processed_path = Path(processed_dir)

        if not processed_path.exists():
            logger.error(f"Processed directory does not exist: {processed_dir}")
            return False

        # Initialize search index
        search_index = {
            "articles": [],
            "keywords": {},
            "topics": {},
            "entities": {},
            "metadata": {
                "total_articles": 0,
                "languages": {},
                "document_types": {}
            }
        }

        # Process all JSON files
        json_files = list(processed_path.glob('**/*.json'))

        for json_file in tqdm(json_files, desc="Building search index"):
            try:
                with open(json_file, 'r', encoding='utf-8') as f:
                    article = json.load(f)

                # Add to articles list with essential info
                article_info = {
                    "id": json_file.stem,
                    "title": article.get("title", ""),
                    "language": article.get("language", "unknown"),
                    "date": article.get("date", ""),
                    "summary": article.get("summary", ""),
                    "path": str(json_file.relative_to(processed_path))
                }

                search_index["articles"].append(article_info)

                # Update metadata counters
                search_index["metadata"]["total_articles"] += 1

                # Count languages
                lang = article.get("language", "unknown")
                search_index["metadata"]["languages"][lang] = search_index["metadata"]["languages"].get(lang, 0) + 1

                # Count document types
                doc_type = article.get("rich_metadata", {}).get("document_type", "Unknown")
                search_index["metadata"]["document_types"][doc_type] = search_index["metadata"]["document_types"].get(doc_type, 0) + 1

                # Index keywords
                for keyword in article.get("keywords", []):
                    if keyword not in search_index["keywords"]:
                        search_index["keywords"][keyword] = []
                    search_index["keywords"][keyword].append(article_info["id"])

                # Index topics
                for topic in article.get("topics", []):
                    if topic not in search_index["topics"]:
                        search_index["topics"][topic] = []
                    search_index["topics"][topic].append(article_info["id"])

                # Index named entities
                for entity in article.get("named_entities", []):
                    if entity not in search_index["entities"]:
                        search_index["entities"][entity] = []
                    search_index["entities"][entity].append(article_info["id"])

            except Exception as e:
                logger.error(f"Error indexing file {json_file}: {str(e)}")

        # Save the index
        with open(index_file, 'w', encoding='utf-8') as f:
            json.dump(search_index, f, ensure_ascii=False, indent=2)

        logger.info(f"Search index created successfully with {search_index['metadata']['total_articles']} articles")
        return True

    except Exception as e:
        logger.error(f"Error creating search index: {str(e)}")
        return False

# Main article processing function
def process_article(markdown_text, filename, filepath):
    """
    Process a single article through all cleaning steps.
    Args:
        markdown_text (str): The content of the .md file
        filename (str): The name of the file (e.g., "article.md")
        filepath (Path or str): Full path to the .md file, used to extract date if needed
    Returns:
        dict: Structured metadata and article info
    """
    try:
        # Basic cleaning
        cleaned_text = basic_text_cleaning(markdown_text)

        # Extract article structure
        article_structure = extract_article_structure(cleaned_text)

        # Extract main content (keep in original language)
        main_content = extract_main_content(article_structure)

        # Detect language
        language = detect_language(main_content or cleaned_text)

        # Extract title (keep in original language)
        title = extract_title(article_structure, filename)

        # Extract date (standardized format, fallback to folder structure)
        date = extract_date(main_content or cleaned_text, filepath=filepath)

        # Extract source (keep in original language)
        source = extract_source(article_structure, main_content or cleaned_text, language)

        # Extract named entities (keep in original language)
        named_entities = extract_named_entities(main_content, language)

        # Extract topics (standardized to English)
        topics = extract_topics(main_content, language)

        # Extract keywords (standardized to English)
        keywords = extract_keywords(main_content, language)

        # Generate summary (keep in original language)
        summary = generate_summary(main_content)

        # Generate rich metadata (standardized to English)
        rich_metadata = generate_rich_metadata(main_content, language)

        # Analyze content structure
        content_structure = analyze_content_structure(main_content)

        # Final structured document
        processed_article = {
            "language": language,
            "title": title,
            "date": date,
            "source": source,
            "main_content": main_content,
            "named_entities": named_entities,
            "topics": topics,
            "keywords": keywords,
            "summary": summary,
            "content_structure": content_structure,
            "rich_metadata": rich_metadata
        }

        return processed_article

    except Exception as e:
        logger.error(f"Error processing {filename}: {str(e)}")

        # Return a minimal valid structure in case of failure
        return {
            "language": "unknown",
            "title": filename.replace('.md', '').replace('-', ' ').title(),
            "date": "",
            "source": "ETH Zurich",
            "main_content": "",
            "named_entities": [],
            "topics": ["University News"],
            "keywords": ["ETH Zurich"],
            "summary": "",
            "content_structure": {
                "paragraph_count": 0,
                "bullet_points": 0,
                "numbered_items": 0,
                "headings": 0,
                "links": 0
                }
            ,
            "rich_metadata": {
                "document_length_words": 0,
                "readability_score": "unknown",
                "sentiment": "neutral",
                "document_type": "News Article",
                "audience_type": ["Students", "Staff"],
                "context_tags": ["ETH Internal"]
            }
        }

def main():
    """
    Main function to process ETH news articles.
    """
    import argparse

    # Parse command line arguments
    parser = argparse.ArgumentParser(description='Process ETH news articles from markdown files')
    parser.add_argument('--input', '-i', required=True, help='Input directory containing markdown files')
    parser.add_argument('--output', '-o', required=True, help='Output directory for processed JSON files')
    parser.add_argument('--index', '-idx', help='Create search index file')
    parser.add_argument('--workers', '-w', type=int, default=4, help='Number of parallel workers')

    args = parser.parse_args()

    logger.info(f"Starting ETH news processing")
    logger.info(f"Input directory: {args.input}")
    logger.info(f"Output directory: {args.output}")

    # Load markdown files
    markdown_files = load_markdown_files(args.input)

    if not markdown_files:
        logger.error("No markdown files found. Exiting.")
        return

    # Process files in parallel
    processed_count, error_count = batch_process(markdown_files, args.output, args.workers)

    logger.info(f"Processing completed. Processed: {processed_count}, Errors: {error_count}")

    # Create search index if requested
    if args.index:
        logger.info(f"Creating search index: {args.index}")
        create_search_index(args.output, args.index)

    logger.info("ETH news processing completed successfully")



In [15]:
# def run_processing(input_dir, output_dir, index_file=None, max_workers=4):
#     """
#     Run ETH news processing directly from Colab notebook.
#     Args:
#         input_dir (str): Path to the directory containing .md files
#         output_dir (str): Output directory for processed JSON files
#         index_file (str, optional): Path to create search index file
#         max_workers (int, optional): Number of parallel workers
#     """
#     logger.info(f"Starting ETH news processing")
#     logger.info(f"Input directory: {input_dir}")
#     logger.info(f"Output directory: {output_dir}")

#     # Load markdown files
#     markdown_files = load_markdown_files(input_dir)

#     if not markdown_files:
#         logger.error("No markdown files found. Exiting.")
#         return

#     # Process files in parallel
#     processed_count, error_count = batch_process(markdown_files, output_dir, max_workers)

#     logger.info(f"Processing completed. Processed: {processed_count}, Errors: {error_count}")

#     # Create search index if requested
#     if index_file:
#         logger.info(f"Creating search index: {index_file}")
#         create_search_index(output_dir, index_file)

#     logger.info("ETH news processing completed successfully")

#     return processed_count, error_count

# # Example usage in Colab
# input_directory = '/content/news-qa-ethz1/HKNews'  # Path to your markdown files
# output_directory = '/content/news-qa-ethz1/HKNews'  # Where to save processed files
# index_path = '/content/eth_news_index.json'  # Optional search index

# # Run the processing
# run_processing(input_directory, output_directory, index_path, max_workers=4)

Building search index: 100%|██████████| 8329/8329 [00:01<00:00, 5480.00it/s]


(4390, 0)

In [18]:
def save_processed_article(processed_article, input_dir, output_dir, filepath):
    """
    Save processed article to JSON file in the same directory structure as the input.
    Args:
        processed_article (dict): Processed article data
        input_dir (str): Base input directory
        output_dir (str): Base output directory
        filepath (Path): Original file path
    """
    try:
        # Get the relative path from the input directory
        input_path = Path(input_dir)
        relative_path = filepath.relative_to(input_path)

        # Create the same directory structure in the output directory
        output_file_dir = Path(output_dir) / relative_path.parent
        output_file_dir.mkdir(parents=True, exist_ok=True)

        # Create output file path with .json extension
        output_file = output_file_dir / (filepath.stem + '.json')

        # Write JSON with proper indentation and UTF-8 encoding
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(processed_article, f, ensure_ascii=False, indent=2)

        logger.info(f"Saved processed article to {output_file}")

    except Exception as e:
        logger.error(f"Error saving processed article {filepath}: {str(e)}")

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

def batch_process(input_dir, output_dir, max_workers=4):
    """
    Process multiple markdown files in parallel.
    Args:
        input_dir (str): Base input directory
        output_dir (str): Base output directory
        max_workers (int): Maximum number of parallel workers
    """
    logger.info(f"Starting batch processing with {max_workers} workers")

    # Load markdown files with full paths
    input_files = []
    input_path = Path(input_dir)

    # Find all .md files in news-qa-ethz1/HKNews directory and subdirectories
    for file_path in input_path.glob('**/HKNews/**/*.md'):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()
                input_files.append((file_path, content))
                logger.info(f"Loaded file: {file_path}")
        except Exception as e:
            logger.error(f"Error reading file {file_path}: {str(e)}")

    logger.info(f"Loaded {len(input_files)} markdown files")

    # Define the processing function inside batch_process so it can access variables
    def process_file(file_info):
        file_path, content = file_info
        try:
            # Process the article
            processed_article = process_article(content, file_path.name, file_path)

            # Save the processed article
            save_processed_article(processed_article, input_dir, output_dir, file_path)

            return True
        except Exception as e:
            logger.error(f"Error processing {file_path}: {str(e)}")
            return False

    # Process files in parallel
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(tqdm(
            executor.map(process_file, input_files),
            total=len(input_files),
            desc="Processing articles"
        ))

    # Count successes and failures
    processed_count = sum(results)
    error_count = len(input_files) - processed_count

    logger.info(f"Batch processing completed. Processed: {processed_count}, Errors: {error_count}")

    return processed_count, error_count


def run_processing_for_github(base_input_dir, base_output_dir, max_workers=4):
    """
    Run ETH news processing for GitHub repository structure.

    Args:
        base_input_dir (str): Path to the directory containing news-qa-ethz1
        base_output_dir (str): Path where processed files should be saved
        max_workers (int, optional): Number of parallel workers
    """
    # Look for HKNews directory within the repository
    input_dir = Path(base_input_dir)
    logger.info(f"Looking for HKNews in: {input_dir}")

    # Process files in parallel
    processed_count, error_count = batch_process(base_input_dir, base_output_dir, max_workers)

    logger.info(f"Processing completed. Processed: {processed_count}, Errors: {error_count}")
    logger.info("ETH news processing completed successfully")

    return processed_count, error_count


base_dir = '/content/news-qa-ethz1'
output_dir = '/content/news-qa-ethz1/'  # Same as input to keep files in place
processed_count, error_count = run_processing_for_github(base_dir, output_dir)

Processing articles: 100%|██████████| 4390/4390 [01:48<00:00, 40.46it/s]


In [21]:
# Navigate to your repo (if not already there)
#%cd /content/news-qa-ethz1

# Track all newly created JSON files
!git add HKNews/**/*.json

# Commit the changes with a clear message
!git commit -m "🔄 Add processed article metadata as JSON files"

# Push the changes to your GitHub fork
!git push origin main

[main a77591af] 🔄 Add processed article metadata as JSON files
 4390 files changed, 203782 insertions(+), 42494 deletions(-)
Enumerating objects: 8394, done.
Counting objects: 100% (8394/8394), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4906/4906), done.
Writing objects: 100% (4907/4907), 4.35 MiB | 3.16 MiB/s, done.
Total 4907 (delta 3568), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3568/3568), completed with 3415 local objects.
remote: This repository moved. Please use the new location:
remote:   https://github.com/Ddannia/news-qa-ethz1.git
To https://github.com/ddannia/news-qa-ethz1.git
   10d5f9d6..a77591af  main -> main
